# AITR Lab 02: Token-level uncertainty vs hallucination (complete Colab notebook)
**Harshita Kushwaha, PDE2026001.** Compares single-pass token scores with semantic entropy for detecting wrong answers on TriviaQA with
Qwen2.5-1.5B-Instruct, and tests a token-gated cascade. This notebook contains **all source code** of the project and runs the whole pipeline.

**Before you start (2 minutes)**
1. `Runtime > Change runtime type > T4 GPU` (then Save).
2. Read the next cell (settings). The defaults are safe: **the test split stays locked** until you freeze the protocol.
3. Run cells top to bottom (`Runtime > Run all` is safe; it pauses at the two questions where *you* label answers by hand).
4. Results are saved to your Google Drive in `MyDrive/AITR_Lab02_run/` and survive disconnects. If Colab disconnects, run again from the top: every step resumes.

**Honest status:** this code was tested on synthetic data in a sandbox without a GPU (60 tests). It has **never been run on the real model or dataset**. The first run may reveal errors; if a cell fails, copy the red error text and the cell output and send it for debugging (do not guess).

In [ ]:
# ====================== SETTINGS (edit only the True/False values) ======================
USE_DRIVE = True              # save results in Google Drive (recommended; survives disconnects)
DEBUG_LIMIT = None            # e.g. 3 = process only 3 questions per stage, to check quickly that everything works. None = normal
POOL_SIZE = None              # None = 1000 questions (protocol). Use 400 ONLY if the pilot projects more than 6 GPU-hours (then log it)

RUN_TESTS = True              # 60 logic tests on synthetic data (CPU, about 1 minute)
RUN_ENV_TEST = True           # checks GPU, libraries, dataset, model, NLI model, token scoring
RUN_SUBSET = True             # builds the deduplicated, seeded 1000-question subset (ids only)
RUN_PILOT = True              # generation + semantic entropy for the 50-question pilot (calibration part only)
DO_HAND_LABELLING = True      # you judge 30 pilot answers by hand (blind: no scores are shown)
RUN_VALIDATION_TOOLS = True   # sampling-truncation check, cluster viewer, reproducibility rerun, normalization check

RUN_CALIBRATION = False       # STEP 9: rest of the calibration part (400 questions). Turn on after the pilot is validated
I_COMPLETED_THE_CHECKLIST = False   # STEP 10: set True only after validation checks pass and you read experiment_protocol.md Section 14
FREEZE_PROTOCOL = False       # STEP 10: freezes code+config (tag protocol-v1). Cannot be silently changed afterwards
RUN_TEST_SPLIT = False        # STEP 11: generates/evaluates the TEST part (600 questions) and runs the final analysis. Needs the freeze
RUN_ERROR_ANALYSIS = False    # STEP 11b: you categorize 60 test errors by hand

## Step 1. Connect Drive, create folders, install libraries
Creates `/content/repo` (code, rebuilt every session) and links `results/`, `data/`, `logs/`, `paper/` to Drive so outputs persist.
`torch` is already installed in Colab and is not reinstalled. Installing takes a minute or two.

In [ ]:
import os, sys, subprocess, time, json
REPO = "/content/repo"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/AITR_Lab02_run"
else:
    BASE = "/content/AITR_Lab02_run"
os.makedirs(REPO, exist_ok=True)
for d in ["results", "data", "logs", "paper"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
    if not os.path.lexists(f"{REPO}/{d}"):
        os.symlink(f"{BASE}/{d}", f"{REPO}/{d}")
for d in ["configs", "src/uncertainty", "scripts", "tests"]:
    os.makedirs(f"{REPO}/{d}", exist_ok=True)
os.chdir(REPO)
print("repo:", REPO, "| outputs saved in:", BASE)

def sh(cmd, log=None, check=True):
    """Run a shell command, stream its output, and append it to logs/<log> (kept on Drive)."""
    print(f"$ {cmd}")
    lf = open(f"{REPO}/logs/{log}", "a") if log else None
    if lf: lf.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} $ {cmd}\n")
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
        if lf: lf.write(line)
    p.wait()
    if lf: lf.close()
    if p.returncode and check:
        raise RuntimeError(f"Command failed (exit code {p.returncode}). Copy the output above and send it for debugging.")
    return p.returncode

Mounted at /content/drive
repo: /content/repo | outputs saved in: /content/drive/MyDrive/AITR_Lab02_run


In [ ]:
!pip -q install -U "transformers>=4.37.0" accelerate datasets scikit-learn scipy pandas pyyaml matplotlib
import importlib.metadata as md
for p in ["torch","transformers","datasets","accelerate","numpy","scipy","scikit-learn","pandas","PyYAML","matplotlib"]:
    print(f"{p:<14}{md.version(p)}")
print()
!nvidia-smi || echo "NO GPU: use Runtime > Change runtime type > T4 GPU, then run again"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
torch         2.14.1+cu130
transformers  5.22.2
datasets      5.2.0
accelerate    1.15.0
numpy         2.4.6
scipy         1.18.1
scikit-learn  1.9.1
pandas        3.1.1
PyYAML        6.0.3
matplotlib    3.11.2

Fri Nov 13 04:29:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------

## Step 2. Create the project files (about 50 small cells, each only writes one file)
Nothing runs here. Each cell is a source file of the project. You can read them to understand the code
(`src/uncertainty/` = logic, `scripts/` = steps you run, `tests/` = checks, `configs/` = frozen settings).
Run them all (select all and run, or `Runtime > Run all`).

In [ ]:
%%writefile configs/protocol_v1.yaml
# protocol_v1.yaml : frozen settings (matches experiment_protocol.md). Status: READY TO FREEZE (not frozen until tagged protocol-v1).
protocol_version: protocol-v1
status: draft_ready_to_freeze
dataset:
  hf_name: mandarjoshi/trivia_qa
  config: rc.nocontext
  split: validation
  dedupe_by: question_id
  pool_size: 1000          # fallback 400 decided from pilot timing only
  pool_size_fallback: 400
  pool_seed: 42
  split_seed: 42
  calibration_fraction: 0.4
  pilot_size: 50           # first 50 of calibration part
  min_per_class_in_test: 80
model:
  name: Qwen/Qwen2.5-1.5B-Instruct
  dtype: float16
prompt:
  template: "Answer the following question with a short answer only (a few words, no explanation).\nQuestion: {question}\nAnswer:"
  few_shot: none
generation:
  greedy: {do_sample: false, max_new_tokens: 24, repetition_penalty: 1.0}
  sampling: {do_sample: true, num_samples: 10, temperature: 1.0, top_k: 0, top_p: 1.0, repetition_penalty: 1.0, max_new_tokens: 24, seed_base: 1000}
scoring:
  method: teacher_forced
  logprob_dtype: float32
  consistency_check: {n_questions: 5, tolerance: 0.05}   # provisional tolerance
  tokens: [all_before_eos, answer_span]
  aggregators: [mean_nll, sum_nll, max_nll, mean_entropy, max_entropy]
  primary_score: all_mean_nll
semantic_entropy:
  nli_model: microsoft/deberta-large-mnli
  nli_input: "{question} {answer}"
  entailment_rule: strict_bidirectional
  merge_identical_normalized_strings: true
  variant_primary: discrete
  variant_secondary: probability_weighted_optional
label:
  rule: normalized_exact_match_against_normalized_aliases   # verify against official TriviaQA normalization
  sensitivity: containment
  hand_check: {n: 30, blind_to_scores: true, min_agreement: 0.90, seed: 42}
metrics:
  primary: auroc          # positive class = wrong answer; higher is better
  secondary: [auprc, ece_10bin_platt, brier_platt]
  bootstrap: {n: 2000, seed: 2026, paired: true, ci: 0.95}
controls: [length_only, random_seed_2026]
cascade:
  gate_score: all_mean_nll
  primary_quantile: 0.5
  risk_score: platt_calibrated_probability
  secondary_risk_score: rank_rule
  curve_quantiles: [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
decision_rules:
  rq1: {predictive_if_ci_lower_above: 0.5, h1_if_paired_ci_of_se_minus_token_excludes_zero: true}
  rq2: {contrasts: [C1_span_vs_all_meanNLL, C2_maxNLL_vs_meanNLL, C3_meanEntropy_vs_meanNLL], ci: 0.983, min_abs_diff: 0.02}
  rq3: {margin: 0.03, upper_ci_limit: 0.06, supported_if: "D<margin and upper95(D)<upper_ci_limit", inconclusive_if: "D<margin and upper95(D)>=upper_ci_limit", not_supported_if: "D>=margin"}
  margin_justification: UNKNOWN_provisional
compute:
  gpu: UNKNOWN_assumed_colab_T4
  cap_gpu_hours_before_fallback: 6
  checkpoint_every_n_questions: 50
extensions:
  E1_length_adjusted_gate_score:
    status: pre_registered_secondary_exploratory   # registered BEFORE any data; one contrast only; not a replacement for the primary analysis
    run_only_if: length_only_auroc_ci_lower_above_0.5
    adjustment: ols_of_primary_score_on_n_tokens_fitted_on_calibration_only
    min_abs_diff: 0.02
    ci: 0.95


Writing configs/protocol_v1.yaml


In [ ]:
%%writefile src/uncertainty/__init__.py
"""Token-level uncertainty vs hallucination: helper package (protocol-v1)."""


Writing src/uncertainty/__init__.py


In [ ]:
%%writefile src/uncertainty/audit.py
"""Pre-evaluation audit (Prompt 13): runs BEFORE any metric is computed.
Checks ID/label alignment, missing values, partitions, metric direction/ranges, leakage and comparator consistency.
Severity: ERROR = stop (evaluation refuses to run) | WARN = report and explain | INFO | SKIPPED = an input was not available,
so the check did NOT run (never silently counted as passed)."""
import math
import time

import numpy as np
import pandas as pd

from . import metrics as M
from .data import ids_hash
from .semantic_entropy import semantic_entropy_from_ids
from .table import TOKEN_VARIANTS
from .textnorm import is_correct, normalize_answer

PRIMARY = "all_meanNLL"
REQUIRED = (["question_id", "pool_index", "split", "is_pilot", "y_wrong_exact", "y_wrong_contain", "n_tokens", "truncated",
             "SE10", "SE5", "n_clusters", "nli_calls", "seconds_greedy", "seconds_samples", "seconds_nli"] + TOKEN_VARIANTS)
SCORE_COLS = [PRIMARY, "SE10", "n_tokens"] + TOKEN_VARIANTS
TOL = 1e-9
MISSING_ERROR_FRACTION = 0.05          # provisional threshold: more than 5% missing in an evaluated split is an error


class _F:
    def __init__(self):
        self.items = []

    def add(self, check, sev, msg, **details):
        self.items.append({"check": check, "severity": sev, "message": msg, "details": details})


def _parse_time(s):
    return time.mktime(time.strptime(s, "%Y-%m-%d %H:%M:%S"))


def run_audit(df, cfg, mode, ids=None, manifest=None, gens=None, sems=None, metas=None, freeze=None, strict_freeze=True):
    """df: per_question table. ids: subset ids DataFrame. manifest: dict. gens/sems/metas: lists of raw records.
    freeze: dict(tag_exists=bool, record=dict|None, code_matches_record=bool|None) (final mode).
    strict_freeze=False downgrades freeze-related errors to warnings (only for --skip-tag-check; the output says so)."""
    F = _F()
    assert mode in ("pilot", "final")
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        F.add("schema", "ERROR", f"missing columns: {missing}")
        return _report(F, mode, len(df))
    df = df.copy()
    F.add("schema", "INFO", f"{len(df)} rows, all {len(REQUIRED)} required columns present")

    # ---- identifiers and partitions
    if df["question_id"].duplicated().any():
        F.add("ids", "ERROR", f"{int(df['question_id'].duplicated().sum())} duplicated question_id")
    if df["pool_index"].duplicated().any() or df["pool_index"].isna().any():
        F.add("ids", "ERROR", "pool_index is missing or duplicated")
    bad_split = sorted(set(df["split"]) - {"calibration", "test"})
    if bad_split:
        F.add("partitions", "ERROR", f"unknown split values: {bad_split}")
    pil = df["is_pilot"].astype(bool)
    if (pil & (df["split"] != "calibration")).any():
        F.add("partitions", "ERROR", "pilot rows outside the calibration part (leakage of test questions into the pilot)")
    cal_ids, test_ids = set(df.loc[df.split == "calibration", "question_id"]), set(df.loc[df.split == "test", "question_id"])
    if cal_ids & test_ids:
        F.add("leakage", "ERROR", f"{len(cal_ids & test_ids)} question_ids appear in BOTH calibration and test")
    else:
        F.add("leakage", "INFO", "no question_id appears in both calibration and test")

    # ---- labels
    for c in ("y_wrong_exact", "y_wrong_contain"):
        if df[c].isna().any() or not set(df[c].dropna().unique()) <= {0, 1}:
            F.add("labels", "ERROR", f"{c} must be 0/1 with no missing values")
    if ((df["y_wrong_contain"] == 1) & (df["y_wrong_exact"] == 0)).any():
        F.add("labels", "ERROR", "exact-match says correct but containment says wrong: impossible, label code inconsistent")

    # ---- scope for completeness and direction (never uses test labels in pilot mode; calibration labels in final mode)
    scope_mask = pil if mode == "pilot" else df["split"].isin(["calibration", "test"])
    scope = df[scope_mask]
    for sp, d in (scope.groupby("split") if len(scope) else []):
        for c in SCORE_COLS:
            frac = float(d[c].isna().mean())
            if frac > MISSING_ERROR_FRACTION:
                F.add("missing", "ERROR", f"{c} missing in {frac:.1%} of {sp} rows", split=sp, column=c)
            elif frac > 0:
                F.add("missing", "WARN", f"{c} missing in {int(d[c].isna().sum())} {sp} rows (dropped listwise at evaluation)", split=sp, column=c)
    if len(scope) and not any(f["check"] == "missing" for f in F.items):
        F.add("missing", "INFO", "no missing scores in the evaluated rows")

    # ---- value ranges and internal consistency (hold by construction if the code is right)
    allc = df
    for pref in ("all", "span"):
        for k in ("meanNLL", "sumNLL", "maxNLL", "meanH", "maxH"):
            col = f"{pref}_{k}"
            if (allc[col].dropna() < -TOL).any():
                F.add("ranges", "ERROR", f"{col} has negative values (NLL and entropy must be >= 0)")
    ok = allc.dropna(subset=["all_meanNLL", "all_maxNLL", "all_sumNLL"])
    if (ok["all_maxNLL"] + TOL < ok["all_meanNLL"]).any():
        F.add("ranges", "ERROR", "max NLL below mean NLL")
    if len(ok) and (np.abs(ok["all_sumNLL"] - ok["all_meanNLL"] * ok["n_tokens"]) > 1e-6 * (1 + ok["all_sumNLL"].abs())).any():
        F.add("ranges", "ERROR", "all_sumNLL != all_meanNLL * n_tokens")
    se = allc.dropna(subset=["SE10"])
    n_s = int(cfg["generation"]["sampling"]["num_samples"])
    if len(se):
        if ((se["SE10"] < -TOL) | (se["SE10"] > math.log(n_s) + 1e-9)).any():
            F.add("ranges", "ERROR", f"SE10 outside [0, ln {n_s}]")
        if ((se["SE5"] < -TOL) | (se["SE5"] > math.log(5) + 1e-9)).any():
            F.add("ranges", "ERROR", "SE5 outside [0, ln 5]")
        if ((se["n_clusters"] < 1) | (se["n_clusters"] > n_s)).any():
            F.add("ranges", "ERROR", f"n_clusters outside [1, {n_s}]")
        if (((se["SE10"].abs() < 1e-12) != (se["n_clusters"] == 1))).any():
            F.add("ranges", "ERROR", "SE10 == 0 must hold exactly when n_clusters == 1")
    if (allc[["seconds_greedy", "seconds_samples"]].dropna() < 0).any().any() or (allc["seconds_nli"].dropna() < 0).any():
        F.add("ranges", "ERROR", "negative timing values")
    if not any(f["check"] == "ranges" and f["severity"] == "ERROR" for f in F.items):
        F.add("ranges", "INFO", "value ranges and internal identities hold")

    # ---- metric direction (higher = more uncertain = predicts WRONG). Uses pilot/calibration labels only.
    dscope = df[pil] if mode == "pilot" else df[df.split == "calibration"]
    dscope = dscope.dropna(subset=SCORE_COLS)
    if len(dscope) >= 20 and dscope["y_wrong_exact"].nunique() == 2:
        y = dscope["y_wrong_exact"].to_numpy()
        flipped, const, vals = [], [], {}
        for c in SCORE_COLS:
            s = dscope[c].to_numpy(float)
            if np.nanstd(s) == 0:
                const.append(c)
                continue
            a = M.auroc(y, s)
            vals[c] = round(a, 4)
            if a < 0.5:
                flipped.append((c, round(a, 4)))
        if [c for c in const if c != "n_tokens"]:
            F.add("direction", "ERROR", f"constant score columns (no information): {[c for c in const if c != 'n_tokens']}")
        if "n_tokens" in const:
            F.add("direction", "WARN", "answer length is constant: the length-only control carries no information")
        if flipped:
            F.add("direction", "WARN", "AUROC below 0.5 on the " + ("pilot" if mode == "pilot" else "calibration") +
                  f" part for {flipped}: orientation may be inverted OR the score is genuinely anti-predictive. Check before evaluating the test part.",
                  columns=[c for c, _ in flipped])
        F.add("direction", "INFO", "orientation convention: every score is oriented so that HIGHER = more uncertain = predicts a WRONG answer; "
              "AUROC (wrong = positive) is computed on that convention", scope_auroc=vals)
    else:
        F.add("direction", "SKIPPED", "not enough labelled rows with both classes in the pilot/calibration scope to check orientation")

    # ---- class balance
    if mode == "pilot":
        if len(dscope) and dscope["y_wrong_exact"].nunique() < 2:
            F.add("balance", "ERROR", "pilot rows contain only one class: AUROC undefined")
    else:
        mincls = int(cfg["dataset"]["min_per_class_in_test"])
        t = df[df.split == "test"]
        nw, nc = int((t["y_wrong_exact"] == 1).sum()), int((t["y_wrong_exact"] == 0).sum())
        if min(nw, nc) < mincls:
            F.add("balance", "ERROR", f"test has {nc} correct / {nw} wrong; need at least {mincls} of each (protocol Section 4)")
        else:
            F.add("balance", "INFO", f"test class counts: {nc} correct / {nw} wrong")
        c = df[df.split == "calibration"]
        if c["y_wrong_exact"].nunique() < 2:
            F.add("balance", "ERROR", "calibration part contains one class only: Platt scaling impossible")

    # ---- alignment with the saved subset ids
    if ids is None:
        F.add("alignment", "SKIPPED", "data/subset_question_ids.csv not available: alignment with the saved subset not checked")
    else:
        if manifest is not None and ids_hash(ids) != manifest.get("ids_sha256"):
            F.add("alignment", "ERROR", "subset ids file does not match the SHA-256 in its manifest")
        extra = set(df["question_id"]) - set(ids["question_id"])
        if extra:
            F.add("alignment", "ERROR", f"{len(extra)} table rows are not in the saved subset")
        m = df.merge(ids, on="question_id", suffixes=("", "_ids"))
        for col in ("split", "pool_index"):
            if (m[col] != m[col + "_ids"]).any():
                F.add("alignment", "ERROR", f"{col} differs between the table and the saved subset for {int((m[col] != m[col + '_ids']).sum())} rows")
        if (m["is_pilot"].astype(bool) != m["is_pilot_ids"].astype(bool)).any():
            F.add("alignment", "ERROR", "is_pilot differs between the table and the saved subset")
        notyet = set(ids["question_id"]) - set(df["question_id"])
        if notyet:
            F.add("alignment", "WARN" if mode == "final" else "INFO",
                  f"{len(notyet)} subset questions have no row in the table (not generated yet)")
        if not any(f["check"] == "alignment" and f["severity"] == "ERROR" for f in F.items):
            F.add("alignment", "INFO", "table rows agree with the saved subset (ids, split, pool_index, pilot flag)")

    # ---- recomputation from raw records (label alignment and score alignment)
    if gens is None:
        F.add("raw", "SKIPPED", "results/raw/generations.jsonl not available: labels and scores were NOT recomputed from raw outputs")
    else:
        G = {g["question_id"]: g for g in gens}
        bad_lab = bad_tok = bad_nt = 0
        T = df.drop_duplicates("question_id", keep="first").set_index("question_id")   # duplicates are reported above; do not crash on them
        for qid, g in G.items():
            if qid not in T.index:
                continue
            r = T.loc[qid]
            y = int(not is_correct(g["greedy_text"], g["normalized_aliases"]))
            bad_lab += int(pd.isna(r["y_wrong_exact"]) or y != int(r["y_wrong_exact"]))
            keep = [i for i, e in enumerate(g["is_eos"]) if not e]
            bad_nt += int(pd.isna(r["n_tokens"]) or len(keep) != int(r["n_tokens"]))
            if keep and not math.isnan(r["all_meanNLL"]):
                mean_nll = -float(np.mean([g["token_logprob"][i] for i in keep]))
                bad_tok += int(abs(mean_nll - float(r["all_meanNLL"])) > 1e-9)
        for label, k in (("labels recomputed from raw answers", bad_lab), ("n_tokens recomputed", bad_nt), ("mean NLL recomputed", bad_tok)):
            F.add("raw", "ERROR" if k else "INFO", f"{label}: {k} mismatches vs the table" if k else f"{label}: all match the table")
        orphan_g = set(G) - set(df["question_id"])
        if orphan_g:
            F.add("raw", "WARN", f"{len(orphan_g)} raw generations are not in the table (rebuild with scripts/04_build_table.py)")
        n_bad_samples = sum(len(g["samples"]) != n_s for g in G.values())
        if n_bad_samples:
            F.add("comparators", "ERROR", f"{n_bad_samples} records do not have exactly {n_s} sampled answers: semantic entropy not comparable across questions")
        mx = int(cfg["generation"]["greedy"]["max_new_tokens"])
        if any(len(g["token_ids"]) > mx for g in G.values()):
            F.add("comparators", "ERROR", f"a greedy answer exceeds max_new_tokens={mx}: generation settings differ from the protocol")
        if len({g.get("logit_vector_size") for g in G.values()}) > 1:
            F.add("comparators", "WARN", "logit vector size differs between records (different model?)")
        # duplicate question text across splits
        seen = {}
        for g in G.values():
            seen.setdefault(normalize_answer(g["question"]), set()).add(g["split"])
        cross = [t for t, s in seen.items() if len(s) > 1]
        if cross:
            F.add("leakage", "ERROR", f"{len(cross)} question texts appear in more than one split (near-duplicate leakage)")
        else:
            F.add("leakage", "INFO", "no question text appears in more than one split (raw records)")
        # semantic records
        if sems is not None:
            S = {s["question_id"]: s for s in sems}
            orph = set(S) - set(G)
            if orph:
                F.add("raw", "ERROR", f"{len(orph)} semantic-entropy records have no generation record")
            bad_se = 0
            for qid, s in S.items():
                if qid in T.index and not math.isnan(T.loc[qid, "SE10"]):
                    bad_se += int(abs(semantic_entropy_from_ids(s["cluster_ids"]) - float(T.loc[qid, "SE10"])) > 1e-9)
            F.add("raw", "ERROR" if bad_se else "INFO", f"SE10 recomputed from cluster ids: {bad_se} mismatches" if bad_se else "SE10 recomputed from cluster ids: all match")
        # argmax consistency (V1)
        am = [g.get("argmax_match_rate") for g in G.values() if g["split"] != "test" and g.get("argmax_match_rate") is not None]
        if am and float(np.nanmean(am)) < 0.99:
            F.add("comparators", "WARN", f"greedy answers are not the argmax of the teacher-forced scores (mean match {np.nanmean(am):.3f}): scoring and generation disagree")

    # ---- comparator settings across runs
    if metas is None:
        F.add("comparators", "SKIPPED", "results/raw/meta_runs.jsonl not available: environment consistency across runs not checked")
    else:
        gm = [m for m in metas if m.get("stage") == "generate"]
        envs = {(m.get("torch"), m.get("transformers"), m.get("gpu")) for m in gm}
        if len(envs) > 1:
            F.add("comparators", "WARN", f"generation was run in {len(envs)} different environments (torch/transformers/GPU): report this; "
                  "GPU non-determinism may differ between parts", environments=[list(e) for e in envs])
        elif gm:
            F.add("comparators", "INFO", "all generation runs used the same torch/transformers/GPU")
        if len({m.get("protocol_status") for m in gm}) > 1:
            F.add("comparators", "WARN", "generation runs were made under different protocol statuses (draft vs frozen)")

    # ---- protocol order (final mode)
    if mode == "final":
        sev = "ERROR" if strict_freeze else "WARN"
        if freeze is None:
            F.add("protocol", sev, "no freeze information available")
        else:
            if not freeze.get("tag_exists"):
                F.add("protocol", sev, "git tag protocol-v1 not found")
            rec = freeze.get("record")
            if rec is None:
                F.add("protocol", sev, "no freeze record (data/protocol_freeze.json): protocol was never frozen")
            else:
                if freeze.get("code_matches_record") is False:
                    F.add("protocol", sev, "code/config differ from the frozen version (undisclosed protocol change)")
                if metas is not None:
                    tt = [m["time"] for m in metas if m.get("stage") == "generate" and m.get("split") in ("test", "all")]
                    if tt and _parse_time(min(tt)) < _parse_time(rec["time"]):
                        F.add("protocol", sev, f"TEST generation started ({min(tt)}) BEFORE the freeze ({rec['time']})")
                    elif not tt:
                        F.add("protocol", "WARN", "no meta record of test generation: cannot verify that it came after the freeze")
                    else:
                        F.add("protocol", "INFO", "test generation started after the freeze")
            if not any(f["check"] == "protocol" and f["severity"] in ("ERROR", "WARN") for f in F.items):
                F.add("protocol", "INFO", "tag present, code identical to the frozen version")
    else:
        F.add("protocol", "INFO", "pilot mode: freeze checks not applicable (no test rows are read)")
    return _report(F, mode, len(df))


def _report(F, mode, n):
    n_err = sum(f["severity"] == "ERROR" for f in F.items)
    return {"mode": mode, "n_rows": n, "generated": time.strftime("%Y-%m-%d %H:%M:%S"), "findings": F.items,
            "n_error": n_err, "n_warn": sum(f["severity"] == "WARN" for f in F.items),
            "n_skipped": sum(f["severity"] == "SKIPPED" for f in F.items), "passed": n_err == 0}


def format_report(rep):
    lines = [f"AUDIT ({rep['mode']}): {rep['n_rows']} rows | {rep['n_error']} ERROR, {rep['n_warn']} WARN, {rep['n_skipped']} SKIPPED"]
    for f in rep["findings"]:
        lines.append(f"  [{f['severity']:<7}] {f['check']:<12} {f['message']}")
    lines.append("AUDIT " + ("PASSED (no ERROR). Read every WARN and SKIPPED line." if rep["passed"] else "FAILED: fix the ERRORs; evaluation will not run."))
    return "\n".join(lines)


def load_audit_inputs(root):
    """Load whatever audit inputs exist on disk (None when a file is missing; SKIPPED checks say so)."""
    import json
    from pathlib import Path

    from .io_utils import read_jsonl
    root = Path(root)

    def jl(p):
        return read_jsonl(p) if p.exists() else None

    ids = pd.read_csv(root / "data" / "subset_question_ids.csv", dtype={"question_id": str}) if (root / "data" / "subset_question_ids.csv").exists() else None
    man = json.load(open(root / "data" / "subset_manifest.json")) if (root / "data" / "subset_manifest.json").exists() else None
    return {"ids": ids, "manifest": man, "gens": jl(root / "results" / "raw" / "generations.jsonl"),
            "sems": jl(root / "results" / "raw" / "semantic.jsonl"), "metas": jl(root / "results" / "raw" / "meta_runs.jsonl")}


Writing src/uncertainty/audit.py


In [ ]:
%%writefile src/uncertainty/cascade.py
"""RQ3: token-gated cascade, evaluated OFFLINE from stored scores (protocol Section 10)."""
import numpy as np


def threshold(s_cal, q):
    """q-quantile of the calibration gate scores. q<=0 escalates everything, q>=1 escalates nothing."""
    if q <= 0:
        return -np.inf
    if q >= 1:
        return np.inf
    return float(np.quantile(np.asarray(s_cal, dtype=float), q))


def cascade_risk(q, s_cal, s_test, se_test, platt_tok, platt_se):
    """Primary cascade: risk = Platt(SE) for escalated questions, Platt(token score) otherwise.
    Returns (risk, escalated_mask)."""
    s_test = np.asarray(s_test, dtype=float)
    esc = s_test >= threshold(s_cal, q)
    risk = np.where(esc, platt_se.predict(se_test), platt_tok.predict(s_test))
    return risk, esc


def rank_rule_risk(esc, s_test, se_test):
    """Secondary cascade: all escalated questions rank above all accepted ones; escalated ordered by SE (ties by token
    score), accepted by token score. Returned values are comparable ranks in (0, 2)."""
    esc = np.asarray(esc, dtype=bool)
    s = np.asarray(s_test, dtype=float)
    se = np.asarray(se_test, dtype=float)
    out = np.zeros(len(s))

    def ranks(primary, secondary):
        order = np.lexsort((secondary, primary))
        r = np.empty(len(primary))
        r[order] = np.arange(1, len(primary) + 1)
        return r / (len(primary) + 1)

    if (~esc).any():
        out[~esc] = 0.5 * ranks(s[~esc], s[~esc])
    if esc.any():
        out[esc] = 1.0 + 0.5 * ranks(se[esc], s[esc])
    return out


def cost_summary(esc, nli_calls, seconds_greedy, seconds_samples, seconds_nli, n_samples):
    """Cost of the cascade on the evaluated questions: the token stage runs for all, the SE stage only for escalated.
    Counts are exact; seconds are the measured per-stage times summed accordingly."""
    esc = np.asarray(esc, dtype=bool)
    n = len(esc)
    nli = np.asarray(nli_calls, dtype=float)
    sg = np.asarray(seconds_greedy, dtype=float)
    ss = np.asarray(seconds_samples, dtype=float)
    sn = np.asarray(seconds_nli, dtype=float)
    return {
        "escalated_fraction": float(esc.mean()),
        "extra_generations_per_question": float(n_samples * esc.sum() / n),
        "nli_calls_per_question": float(nli[esc].sum() / n),
        "seconds_per_question": float((sg.sum() + (ss + sn)[esc].sum()) / n),
    }


Writing src/uncertainty/cascade.py


In [ ]:
%%writefile src/uncertainty/data.py
"""Cycle 1: data loading, validation, deduplication, sampling and splitting (protocol Section 4).

Pipeline:  records -> DataFrame -> dedupe -> validate/drop unlabellable -> sample pool -> split -> save ids only.
Everything is deterministic given the seeds, independent of the order in which rows were loaded.
Only question IDs are saved (not question text), because TriviaQA terms are unspecified (data/README.md).
"""
import hashlib
import json

import numpy as np
import pandas as pd

COLUMNS = ["question_id", "question", "gold_value", "aliases", "normalized_aliases", "question_source"]
IDS_COLUMNS = ["pool_index", "question_id", "split", "is_pilot"]


def rows_from_records(records) -> pd.DataFrame:
    """records: iterable of dicts shaped like a TriviaQA rc.nocontext example
    (keys: question, question_id, question_source, answer{value, aliases, normalized_aliases})."""
    rows = []
    for r in records:
        a = r["answer"]
        rows.append({
            "question_id": str(r["question_id"]),
            "question": str(r["question"]).strip(),
            "gold_value": str(a.get("value", "")).strip(),
            "aliases": list(a.get("aliases", [])),
            "normalized_aliases": list(a.get("normalized_aliases", [])),
            "question_source": str(r.get("question_source", "")),
        })
    return pd.DataFrame(rows, columns=COLUMNS)


def dedupe(df: pd.DataFrame):
    """Drop repeated question_ids, then repeated question texts. Deterministic: rows are first sorted by
    (question_id, question_source) so the kept row does not depend on file order.
    Returns (df_clean, stats)."""
    n0 = len(df)
    d = df.sort_values(["question_id", "question_source"], kind="stable").reset_index(drop=True)
    alias_key = d["normalized_aliases"].map(lambda x: tuple(sorted(x)))
    n_conflict = int((d.assign(_k=alias_key).groupby("question_id")["_k"].nunique() > 1).sum())
    n_dup_id_rows = int(d.duplicated("question_id", keep="first").sum())
    d = d.drop_duplicates("question_id", keep="first")
    n_dup_text_rows = int(d.duplicated("question", keep="first").sum())
    d = d.drop_duplicates("question", keep="first").reset_index(drop=True)
    stats = {"rows_before": n0, "dropped_repeated_question_id": n_dup_id_rows,
             "dropped_repeated_question_text": n_dup_text_rows,
             "question_ids_with_conflicting_aliases": n_conflict, "rows_after": len(d)}
    return d, stats


def validate(df: pd.DataFrame) -> dict:
    """Return counts of problems (does not modify df)."""
    return {
        "repeated_question_id": int(df["question_id"].duplicated().sum()),
        "empty_question": int((df["question"].str.len() == 0).sum()),
        "no_normalized_aliases": int((df["normalized_aliases"].map(len) == 0).sum()),
        "empty_gold_value": int((df["gold_value"] == "").sum()),
    }


def drop_unlabellable(df: pd.DataFrame):
    """Rows without any alias cannot be labelled, so they are removed BEFORE sampling (depends on no model output)."""
    keep = df["normalized_aliases"].map(len) > 0
    return df[keep].reset_index(drop=True), int((~keep).sum())


def sample_pool(df: pd.DataFrame, pool_size: int, seed: int) -> pd.DataFrame:
    """Seeded sample of `pool_size` questions. Sorting by question_id first makes the result independent of load order.
    `pool_index` (0..pool_size-1) is the question's position in the pool and defines its sampling seed (1000 + pool_index)."""
    d = df.sort_values("question_id", kind="stable").reset_index(drop=True)
    if pool_size > len(d):
        raise ValueError(f"pool_size {pool_size} > available questions {len(d)}")
    idx = np.random.RandomState(seed).permutation(len(d))[:pool_size]
    pool = d.iloc[idx].reset_index(drop=True)
    pool.insert(0, "pool_index", np.arange(len(pool)))
    return pool


def split_pool(pool: pd.DataFrame, calibration_fraction: float, seed: int, pilot_size: int) -> pd.DataFrame:
    """Split BY QUESTION into calibration/test. The pilot is the first `pilot_size` questions of the calibration part
    (in the permuted order); it never contains test questions."""
    n = len(pool)
    n_cal = int(round(calibration_fraction * n))
    perm = np.random.RandomState(seed).permutation(n)
    cal_idx, test_idx = perm[:n_cal], perm[n_cal:]
    split = np.array(["test"] * n, dtype=object)
    split[cal_idx] = "calibration"
    pilot = np.zeros(n, dtype=bool)
    pilot[cal_idx[:pilot_size]] = True
    out = pool.copy()
    out["split"] = split
    out["is_pilot"] = pilot
    return out


def ids_table(df: pd.DataFrame) -> pd.DataFrame:
    """The only table that is saved: ids and assignments, no question text."""
    return df[IDS_COLUMNS].copy()


def ids_hash(ids_df: pd.DataFrame) -> str:
    s = "\n".join(f"{r.pool_index},{r.question_id},{r.split},{int(r.is_pilot)}" for r in ids_df.itertuples())
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def build_subset(df_raw: pd.DataFrame, ds_cfg: dict, pool_size=None):
    """Full cycle-1 pipeline on an already loaded DataFrame. Returns (subset_df, report dict)."""
    pool_size = int(pool_size or ds_cfg["pool_size"])
    d, dstats = dedupe(df_raw)
    problems = validate(d)
    d, n_unlabellable = drop_unlabellable(d)
    pool = sample_pool(d, pool_size, ds_cfg["pool_seed"])
    sub = split_pool(pool, ds_cfg["calibration_fraction"], ds_cfg["split_seed"], ds_cfg["pilot_size"])
    ids = ids_table(sub)
    report = {
        "dedupe": dstats, "validation_after_dedupe": problems, "dropped_unlabellable": n_unlabellable,
        "available_after_cleaning": len(d), "pool_size": pool_size,
        "n_calibration": int((sub["split"] == "calibration").sum()), "n_test": int((sub["split"] == "test").sum()),
        "n_pilot": int(sub["is_pilot"].sum()), "ids_sha256": ids_hash(ids),
        "pool_seed": ds_cfg["pool_seed"], "split_seed": ds_cfg["split_seed"],
    }
    return sub, report


def save_subset(sub: pd.DataFrame, report: dict, ids_path, manifest_path, extra: dict = None):
    ids = ids_table(sub)
    ids.to_csv(ids_path, index=False)
    with open(manifest_path, "w") as f:
        json.dump({**report, **(extra or {})}, f, indent=2, default=str)
    return ids


def attach_text(ids: pd.DataFrame, full_clean_df: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the full subset (with question text and aliases) from the saved ids and a freshly loaded, cleaned dataset."""
    merged = ids.merge(full_clean_df, on="question_id", how="left", validate="one_to_one")
    missing = int(merged["question"].isna().sum())
    if missing:
        raise ValueError(f"{missing} saved question_ids not found in the loaded dataset")
    return merged


Writing src/uncertainty/data.py


In [ ]:
%%writefile src/uncertainty/diagnostics.py
"""Safe diagnostics collection for debugging (Prompt 11). Never prints environment variables; redacts token-like strings."""
import re

_PATTERNS = [
    (re.compile(r"hf_[A-Za-z0-9]{16,}"), "hf_[REDACTED]"),
    (re.compile(r"sk-[A-Za-z0-9_\-]{16,}"), "sk-[REDACTED]"),
    (re.compile(r"(?i)(api[_-]?key|token|secret|password)\s*[=:]\s*\S+"), r"\1=[REDACTED]"),
    (re.compile(r"(?i)authorization:\s*bearer\s+\S+"), "Authorization: Bearer [REDACTED]"),
]


def redact(text: str) -> str:
    for pat, repl in _PATTERNS:
        text = pat.sub(repl, text)
    return text


def tail(path, n=60):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return f"(file not found: {path})\n"


Writing src/uncertainty/diagnostics.py


In [ ]:
%%writefile src/uncertainty/evaluation.py
"""Cycle 6/7: evaluation exactly as written in experiment_protocol.md (Sections 9 to 12) + figures.

mode="pilot": uses ONLY the pilot rows (calibration part); no test row is read; no calibration/cascade analysis.
mode="final": calibration part fits Platt models and escalation thresholds; the test part is evaluated once.
"""
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

from . import cascade as C
from . import metrics as M
from .table import TOKEN_VARIANTS

PRIMARY = "all_meanNLL"


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)


def _git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=10).stdout.strip() or None
    except Exception:  # noqa: BLE001
        return None


def run_analysis(df, cfg, mode, out_dir, n_boot=None, make_figs=True, ignore_balance=False, provenance=None):
    assert mode in ("pilot", "final")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    bcfg = cfg["metrics"]["bootstrap"]
    B, seed = int(n_boot or bcfg["n"]), int(bcfg["seed"])
    ci_level = float(bcfg["ci"])
    rules = cfg["decision_rules"]
    flags = []
    need = [PRIMARY, "SE10", "n_tokens", "y_wrong_exact"] + TOKEN_VARIANTS

    if mode == "pilot":
        ev = df[df["is_pilot"].astype(bool)].copy()
        cal = None
    else:
        ev = df[df["split"] == "test"].copy()
        cal = df[df["split"] == "calibration"].copy()
    n0 = len(ev)
    ev = ev.dropna(subset=need).reset_index(drop=True)
    if n0 != len(ev):
        flags.append(f"{n0 - len(ev)} of {n0} evaluation rows dropped for missing scores (listwise, same rows for all methods)")
    if cal is not None:
        c0 = len(cal)
        cal = cal.dropna(subset=need).reset_index(drop=True)
        if c0 != len(cal):
            flags.append(f"{c0 - len(cal)} of {c0} calibration rows dropped for missing scores")
    if len(ev) < 10:
        raise RuntimeError(f"only {len(ev)} usable evaluation rows")

    y = ev["y_wrong_exact"].to_numpy().astype(int)
    n, n_wrong = len(y), int(y.sum())
    n_correct = n - n_wrong
    if min(n_wrong, n_correct) == 0:
        raise RuntimeError("evaluation rows contain only one class; AUROC undefined")
    min_cls = int(cfg["dataset"]["min_per_class_in_test"])
    balance_ok = min(n_wrong, n_correct) >= min_cls
    if mode == "final" and not balance_ok:
        msg = f"class balance violated: {n_correct} correct / {n_wrong} wrong in test (need >= {min_cls} each)"
        if not ignore_balance:
            raise RuntimeError(msg + ". Protocol Section 4 step 7: enlarge the pool (class counts only; do not look at scores).")
        flags.append(msg + " (ignored by --ignore-balance)")

    idx = M.bootstrap_indices(n, B, seed)
    rand = M.random_scores(n, seed)
    S = {"token": ev[PRIMARY].to_numpy(float), "se": ev["SE10"].to_numpy(float),
         "length": ev["n_tokens"].to_numpy(float), "random": rand}
    LABEL = {"token": "Mean NLL, all tokens (primary)", "se": "Semantic entropy (N=10)",
             "length": "Answer length only (control)", "random": "Random score (control)"}

    # ---------------- Table 2: main comparison
    platt, p_ev = {}, {}
    if cal is not None:
        cal_scores = {"token": cal[PRIMARY].to_numpy(float), "se": cal["SE10"].to_numpy(float),
                      "length": cal["n_tokens"].to_numpy(float), "random": M.random_scores(len(cal), seed + 1)}
        yc = cal["y_wrong_exact"].to_numpy().astype(int)
        for k in S:
            platt[k] = M.PlattScaler().fit(cal_scores[k], yc)
            p_ev[k] = platt[k].predict(S[k])
    boots = {k: M.boot_stat(M.auroc, y, S[k], idx) for k in S}
    rows2 = []
    for k in S:
        lo, hi = M.percentile_ci(boots[k], ci_level)
        ap = M.boot_stat(M.auprc, y, S[k], idx)
        alo, ahi = M.percentile_ci(ap, ci_level)
        rows2.append({"score": LABEL[k], "key": k, "auroc": M.auroc(y, S[k]), "auroc_lo": lo, "auroc_hi": hi,
                      "auprc": M.auprc(y, S[k]), "auprc_lo": alo, "auprc_hi": ahi,
                      "ece": M.ece(y, p_ev[k]) if k in p_ev else np.nan,
                      "brier": M.brier(y, p_ev[k]) if k in p_ev else np.nan,
                      "platt_slope": platt[k].slope if k in platt else np.nan,
                      "n": n, "n_wrong": n_wrong, "prevalence_wrong": n_wrong / n})
    t2 = pd.DataFrame(rows2)
    t2.to_csv(out_dir / "table2.csv", index=False)

    # ---------------- RQ1
    d_boot = boots["se"] - boots["token"]
    d_lo, d_hi = M.percentile_ci(d_boot, ci_level)
    tok_lo = float(t2.loc[t2.key == "token", "auroc_lo"].iloc[0])
    rq1 = {"token_auroc": M.auroc(y, S["token"]), "token_auroc_ci": [tok_lo, float(t2.loc[t2.key == "token", "auroc_hi"].iloc[0])],
           "token_is_predictive": bool(tok_lo > rules["rq1"]["predictive_if_ci_lower_above"]),
           "se_auroc": M.auroc(y, S["se"]), "diff_se_minus_token": M.auroc(y, S["se"]) - M.auroc(y, S["token"]),
           "diff_ci": [d_lo, d_hi],
           "H1": "supported" if d_lo > 0 else "not supported (CI of the difference includes 0 or favours the token score)"}

    # ---------------- RQ2: aggregation ablation, contrasts, length
    vboots = {v: M.boot_stat(M.auroc, y, ev[v].to_numpy(float), idx) for v in TOKEN_VARIANTS}
    rows3 = []
    for v in TOKEN_VARIANTS:
        lo, hi = M.percentile_ci(vboots[v], ci_level)
        rows3.append({"kind": "variant", "name": v, "auroc": M.auroc(y, ev[v].to_numpy(float)), "lo": lo, "hi": hi,
                      "matters": np.nan})
    c_level = float(rules["rq2"]["ci"])
    min_diff = float(rules["rq2"]["min_abs_diff"])
    contrasts = [("C1_span_vs_all_meanNLL", "span_meanNLL", "all_meanNLL"), ("C2_maxNLL_vs_meanNLL", "all_maxNLL", "all_meanNLL"),
                 ("C3_meanH_vs_meanNLL", "all_meanH", "all_meanNLL")]
    rq2 = {"contrasts": {}}
    for name, a, b in contrasts:
        pt = M.auroc(y, ev[a].to_numpy(float)) - M.auroc(y, ev[b].to_numpy(float))
        lo, hi = M.percentile_ci(vboots[a] - vboots[b], c_level)
        matters = bool((lo > 0 or hi < 0) and abs(pt) >= min_diff)
        rows3.append({"kind": "contrast", "name": name, "auroc": pt, "lo": lo, "hi": hi, "matters": matters})
        rq2["contrasts"][name] = {"difference": pt, "ci": [lo, hi], "ci_level": c_level, "matters": matters}
    t3 = pd.DataFrame(rows3)
    t3.to_csv(out_dir / "table3.csv", index=False)
    len_lo = float(t2.loc[t2.key == "length", "auroc_lo"].iloc[0])
    rho, pval = M.spearman(S["token"], S["length"])
    nt = S["length"]
    edges = np.unique(np.quantile(nt, [1 / 3, 2 / 3]))
    if len(edges) < 2:
        edges = np.unique(np.quantile(nt, [0.5]))
    strata = []
    if len(edges) >= 1 and len(np.unique(nt)) > 1:
        g = np.digitize(nt, edges)
        for gi in np.unique(g):
            m = g == gi
            strata.append({"stratum": int(gi), "n": int(m.sum()), "n_wrong": int(y[m].sum()),
                           "auroc_token": M.auroc(y[m], S["token"][m]), "auroc_se": M.auroc(y[m], S["se"][m]),
                           "token_count_range": [float(nt[m].min()), float(nt[m].max())]})
    rq2.update({"length_only_auroc": float(t2.loc[t2.key == "length", "auroc"].iloc[0]), "length_auroc_ci_lower": len_lo,
                "length_confound_present": bool(len_lo > 0.5), "spearman_token_score_vs_length": rho, "spearman_p": pval,
                "length_strata": strata})

    # ---------------- RQ3: cascade (final only)
    rq3, t4 = None, None
    if cal is not None:
        q_primary = float(cfg["cascade"]["primary_quantile"])
        quants = sorted(set(float(q) for q in cfg["cascade"]["curve_quantiles"]) | {q_primary})
        s_cal, se_cal = cal[PRIMARY].to_numpy(float), cal["SE10"].to_numpy(float)
        nli = ev["nli_calls"].fillna(0).to_numpy(float)
        rows4, cboots = [], {}
        for q in quants:
            risk, esc = C.cascade_risk(q, s_cal, S["token"], S["se"], platt["token"], platt["se"])
            bs = M.boot_stat(M.auroc, y, risk, idx)
            cboots[q] = bs
            lo, hi = M.percentile_ci(bs, ci_level)
            rr = C.rank_rule_risk(esc, S["token"], S["se"])
            cost = C.cost_summary(esc, nli, ev["seconds_greedy"].to_numpy(float), ev["seconds_samples"].to_numpy(float),
                                  ev["seconds_nli"].fillna(0).to_numpy(float), cfg["generation"]["sampling"]["num_samples"])
            rows4.append({"quantile_q": q, "role": "token-only" if q >= 1 else ("SE-for-all" if q <= 0 else ("primary" if q == q_primary else "curve")),
                          "auroc": M.auroc(y, risk), "auroc_lo": lo, "auroc_hi": hi, "auroc_rank_rule": M.auroc(y, rr), **cost})
        t4 = pd.DataFrame(rows4)
        t4.to_csv(out_dir / "table4.csv", index=False)
        prim = t4.loc[t4.quantile_q == q_primary].iloc[0]
        D = M.auroc(y, S["se"]) - float(prim["auroc"])
        D_boot = boots["se"] - cboots[q_primary]
        D_lo, D_hi = M.percentile_ci(D_boot, ci_level)
        margin, upper = float(rules["rq3"]["margin"]), float(rules["rq3"]["upper_ci_limit"])
        outcome = "not supported" if D >= margin else ("supported" if D_hi < upper else "inconclusive")
        rq3 = {"primary_quantile": q_primary, "realized_escalated_fraction": float(prim["escalated_fraction"]),
               "auroc_cascade": float(prim["auroc"]), "auroc_se_raw": M.auroc(y, S["se"]), "D": D, "D_ci": [D_lo, D_hi],
               "margin": margin, "upper_ci_limit": upper, "H3": outcome,
               "extra_generations_per_question_cascade": float(prim["extra_generations_per_question"]),
               "extra_generations_per_question_se_for_all": float(cfg["generation"]["sampling"]["num_samples"]),
               "platt_slope_token": platt["token"].slope, "platt_slope_se": platt["se"].slope}
        for k in ("token", "se"):
            if platt[k].slope <= 0:
                flags.append(f"Platt slope for {k} is {platt[k].slope:.3f} <= 0: the calibrated ranking is inverted relative to the raw score")
        if rq1["token_auroc"] < 0.5 + 1e-9 or not rq1["token_is_predictive"]:
            flags.append("RQ1: token score is not predictive by the pre-specified rule; RQ3 is expected to fail and must be reported as such")

    # ---------------- Sensitivity (Table 6)
    sens = []
    se5 = ev["SE5"].to_numpy(float)
    if not np.isnan(se5).any():
        bs = M.boot_stat(M.auroc, y, se5, idx)
        lo, hi = M.percentile_ci(bs, ci_level)
        sens.append({"analysis": "SE from first 5 of 10 samples", "auroc": M.auroc(y, se5), "lo": lo, "hi": hi})
    if "y_wrong_contain" in ev and not ev["y_wrong_contain"].isna().any():
        yc2 = ev["y_wrong_contain"].to_numpy().astype(int)
        for k, lab in (("token", "Mean NLL"), ("se", "Semantic entropy")):
            bs = M.boot_stat(M.auroc, yc2, S[k], idx)
            lo, hi = M.percentile_ci(bs, ci_level)
            sens.append({"analysis": f"{lab}, containment label", "auroc": M.auroc(yc2, S[k]), "lo": lo, "hi": hi})
    pd.DataFrame(sens).to_csv(out_dir / "table6.csv", index=False)

    # ---------------- results.csv (long format) and summary.json
    long_rows = []
    for r in rows2:
        for m in ("auroc", "auprc", "ece", "brier"):
            long_rows.append({"mode": mode, "analysis": "main", "name": r["score"], "metric": m, "value": r[m],
                              "ci_lo": r.get(m + "_lo", np.nan), "ci_hi": r.get(m + "_hi", np.nan), "n": n})
    for r in rows3:
        long_rows.append({"mode": mode, "analysis": "ablation_" + r["kind"], "name": r["name"], "metric": "auroc_or_difference",
                          "value": r["auroc"], "ci_lo": r["lo"], "ci_hi": r["hi"], "n": n})
    if t4 is not None:
        for _, r in t4.iterrows():
            long_rows.append({"mode": mode, "analysis": "cascade", "name": f"q={r.quantile_q}", "metric": "auroc",
                              "value": r.auroc, "ci_lo": r.auroc_lo, "ci_hi": r.auroc_hi, "n": n})
    for r in sens:
        long_rows.append({"mode": mode, "analysis": "sensitivity", "name": r["analysis"], "metric": "auroc",
                          "value": r["auroc"], "ci_lo": r["lo"], "ci_hi": r["hi"], "n": n})
    res_df = pd.DataFrame(long_rows)
    for k, v in (provenance or {}).items():          # traceability: every row carries where it came from
        res_df[k] = v
    res_df.to_csv(out_dir / "results.csv", index=False)

    summary = {"mode": mode, "generated": time.strftime("%Y-%m-%d %H:%M:%S"), "git_commit": _git_commit(),
               "n_eval": n, "n_wrong": n_wrong, "n_correct": n_correct, "class_balance_ok": balance_ok,
               "n_calibration": None if cal is None else len(cal), "bootstrap": {"n": B, "seed": seed, "ci_level": ci_level},
               "rq1": rq1, "rq2": rq2, "rq3": rq3, "flags": flags, "provenance": provenance,
               "note": "pilot mode is exploratory and uses calibration-part questions only" if mode == "pilot" else "final mode"}
    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=_json_default)

    if make_figs:
        _figures(out_dir, y, S, ev, t3, t4, p_ev, idx, seed)
    return summary


def _figures(out_dir, y, S, ev, t3, t4, p_ev, idx, seed, n_fig_boot=300):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve

    matplotlib.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 9, "axes.labelsize": 9, "legend.fontsize": 7.5,
                                "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                                "figure.dpi": 100, "savefig.dpi": 300})
    fd = Path(out_dir) / "figs"
    fd.mkdir(exist_ok=True)

    def save(fig, name):
        for ext in ("png", "pdf"):
            fig.savefig(fd / f"{name}.{ext}", dpi=300, bbox_inches="tight")
        plt.close(fig)

    labels = {"token": "Mean NLL (primary)", "se": "Semantic entropy", "length": "Length only", "random": "Random"}
    # Fig 2: ROC curves with bootstrap band (n_fig_boot resamples)
    grid = np.linspace(0, 1, 101)
    fig, ax = plt.subplots(figsize=(4.2, 4))
    for k, sc in S.items():
        fpr, tpr, _ = roc_curve(y, sc)
        ax.plot(fpr, tpr, label=f"{labels[k]} (AUROC {M.auroc(y, sc):.3f})")
        tprs = []
        for i in idx[:n_fig_boot]:
            if y[i].min() == y[i].max():
                continue
            f_, t_, _ = roc_curve(y[i], sc[i])
            tprs.append(np.interp(grid, f_, t_))
        if tprs:
            ax.fill_between(grid, np.percentile(tprs, 2.5, axis=0), np.percentile(tprs, 97.5, axis=0), alpha=0.12)
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (wrong answers)")
    ax.legend(fontsize=7, loc="lower right"); save(fig, "fig2_roc")
    # Fig 3: AUROC per aggregator
    v = t3[t3.kind == "variant"].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(4.6, 3.6))
    ypos = np.arange(len(v))
    ax.errorbar(v["auroc"], ypos, xerr=[v["auroc"] - v["lo"], v["hi"] - v["auroc"]], fmt="o", capsize=3)
    ax.axvline(0.5, color="k", ls=":", lw=0.8)
    pretty = {"meanNLL": "mean NLL", "sumNLL": "sum NLL", "maxNLL": "max NLL", "meanH": "mean entropy", "maxH": "max entropy"}

    def pn(name):
        ts, k = name.split("_", 1)
        return f"{pretty[k]} ({'all tokens' if ts == 'all' else 'answer span'})"
    ax.set_yticks(ypos); ax.set_yticklabels([pn(x) for x in v["name"]], fontsize=7); ax.invert_yaxis()
    ax.set_xlabel("AUROC (95% CI)"); save(fig, "fig3_aggregators")
    # Fig 4: cascade curve (final only)
    if t4 is not None:
        d = t4.sort_values("escalated_fraction")
        fig, ax = plt.subplots(figsize=(4.6, 3.6))
        l1, = ax.plot(d["escalated_fraction"], d["auroc"], "o-", color="tab:blue", label="Cascade AUROC (95% CI band)")
        ax.fill_between(d["escalated_fraction"], d["auroc_lo"], d["auroc_hi"], alpha=0.15)
        pr = d[d.role == "primary"]
        if len(pr):
            ax.axvline(float(pr["escalated_fraction"].iloc[0]), color="grey", ls="--", lw=0.8)
        ax.set_xlabel("Fraction of questions escalated to semantic entropy"); ax.set_ylabel("AUROC")
        ax2 = ax.twinx()
        l2, = ax2.plot(d["escalated_fraction"], d["seconds_per_question"], "s--", color="tab:red", label="Measured seconds per question")
        ax2.set_ylabel("Measured seconds per question", color="tab:red"); ax2.grid(False)
        ax.set_ylabel("AUROC", color="tab:blue")
        ax.legend([l1, l2], [l1.get_label(), l2.get_label()], loc="upper left", fontsize=7)
        save(fig, "fig4_cascade")
    # Fig 5: reliability diagram (final only)
    if p_ev:
        fig, ax = plt.subplots(figsize=(4.2, 4))
        for k in ("token", "se"):
            rt = [r for r in M.reliability_table(y, p_ev[k]) if r["n"] > 0]
            ax.plot([r["mean_pred"] for r in rt], [r["frac_wrong"] for r in rt], "o-", label=labels[k])
        ax.plot([0, 1], [0, 1], "k:", lw=0.8)
        ax.set_xlabel("Predicted probability of a wrong answer"); ax.set_ylabel("Observed fraction wrong")
        ax.legend(fontsize=7); save(fig, "fig5_reliability")


Writing src/uncertainty/evaluation.py


In [ ]:
%%writefile src/uncertainty/extensions.py
"""Pre-registered secondary extension E1: length-adjusted gate score (ONE isolated change; baseline columns are never modified).

Motivation (from my review, limitation L5): token scores can be confounded with answer length. E1 removes the linear
dependence of the primary score on the number of generated tokens, with the line fitted on the CALIBRATION part only.
This is a routine adjustment, not a claimed novelty. It runs only if the length confound is present by the frozen rule."""
import json
from pathlib import Path

import numpy as np

from . import metrics as M
from .evaluation import PRIMARY
from .table import TOKEN_VARIANTS


class LengthAdjuster:
    """score_adj = score - (a + b * n_tokens), with (a, b) from ordinary least squares on calibration data."""

    def fit(self, score, n_tokens):
        x = np.asarray(n_tokens, dtype=float)
        y = np.asarray(score, dtype=float)
        if np.var(x) == 0:
            raise ValueError("n_tokens is constant in the calibration part: no length adjustment is possible")
        self.b = float(np.cov(x, y, bias=True)[0, 1] / np.var(x))
        self.a = float(y.mean() - self.b * x.mean())
        return self

    def transform(self, score, n_tokens):
        return np.asarray(score, dtype=float) - (self.a + self.b * np.asarray(n_tokens, dtype=float))


def run_length_adjusted_extension(df, cfg, out_dir, n_boot=None):
    """Compare AUROC of the length-adjusted primary score with the unchanged primary score on the TEST part (paired bootstrap).
    Does not modify df. Writes extension_E1.json. Returns the result dict."""
    ec = cfg["extensions"]["E1_length_adjusted_gate_score"]
    bcfg = cfg["metrics"]["bootstrap"]
    B, seed = int(n_boot or bcfg["n"]), int(bcfg["seed"])
    need = [PRIMARY, "SE10", "n_tokens", "y_wrong_exact"] + TOKEN_VARIANTS      # same rows as the main analysis -> same bootstrap pairing
    cal = df[df["split"] == "calibration"].dropna(subset=need)
    ev = df[df["split"] == "test"].dropna(subset=need).reset_index(drop=True)
    y = ev["y_wrong_exact"].to_numpy().astype(int)
    idx = M.bootstrap_indices(len(y), B, seed)
    out = {"experiment": "E1_length_adjusted_gate_score", "status": ec["status"], "n_eval": int(len(y))}

    len_boot = M.boot_stat(M.auroc, y, ev["n_tokens"].to_numpy(float), idx)
    len_lo, len_hi = M.percentile_ci(len_boot, float(ec["ci"]))
    out.update({"length_only_auroc_ci": [len_lo, len_hi], "trigger_length_confound_present": bool(len_lo > 0.5)})
    if not out["trigger_length_confound_present"]:
        out.update({"ran": False, "reason": "trigger not met: length-only AUROC CI does not exclude 0.5"})
    else:
        try:
            adj = LengthAdjuster().fit(cal[PRIMARY].to_numpy(float), cal["n_tokens"].to_numpy(float))
        except ValueError as e:
            out.update({"ran": False, "reason": str(e)})
        else:
            s0 = ev[PRIMARY].to_numpy(float)
            s1 = adj.transform(s0, ev["n_tokens"].to_numpy(float))
            b0, b1 = M.boot_stat(M.auroc, y, s0, idx), M.boot_stat(M.auroc, y, s1, idx)
            diff = M.auroc(y, s1) - M.auroc(y, s0)
            lo, hi = M.percentile_ci(b1 - b0, float(ec["ci"]))
            verdict = ("improves" if (lo > 0 and diff >= ec["min_abs_diff"]) else
                       "worse" if hi < 0 else "no evidence of improvement")
            out.update({"ran": True, "fit_intercept": adj.a, "fit_slope_per_token": adj.b,
                        "auroc_primary": M.auroc(y, s0), "auroc_adjusted": M.auroc(y, s1), "difference": diff,
                        "difference_ci": [lo, hi], "verdict": verdict,
                        "note": "exploratory secondary analysis; one pre-registered contrast; the primary analysis is unchanged"})
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(out_dir) / "extension_E1.json", "w") as f:
        json.dump(out, f, indent=2, default=float)
    return out


Writing src/uncertainty/extensions.py


In [ ]:
%%writefile src/uncertainty/generation.py
"""Cycle 2: greedy answer + teacher-forced token scores + sampled answers for ONE question.

Needs torch/transformers (imported lazily). The helper functions at the top are pure Python and are unit-tested
without a GPU; `generate_record` itself runs the model and is first exercised by scripts/02_generate.py [TEST REQUIRED].
"""
import time

from .textnorm import extract_short_answer


def eos_id_set(tok, model):
    ids = set()
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    e = model.generation_config.eos_token_id
    if isinstance(e, int):
        ids.add(e)
    elif isinstance(e, (list, tuple)):
        ids.update(int(x) for x in e)
    return ids


def token_char_spans(decode_fn, ids):
    """Character span (start, end) of each token in the decoded text, from decoding growing prefixes.
    decode_fn(list_of_ids) -> str. Special tokens decode to '' (zero-length span). Approximate for
    multi-byte characters split across tokens (documented limitation)."""
    spans, prev = [], 0
    for i in range(len(ids)):
        cur = len(decode_fn(list(ids[: i + 1])))
        spans.append((prev, cur))
        prev = cur
    return spans


def answer_span_mask(spans, text, short_answer):
    """True for tokens overlapping the extracted short answer inside `text`. All False if the answer is not found."""
    if not short_answer:
        return [False] * len(spans)
    start = text.find(short_answer)
    if start < 0:
        return [False] * len(spans)
    end = start + len(short_answer)
    return [bool(e > s and s < end and e > start) for (s, e) in spans]


def load_model_and_tokenizer(model_name, device):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
    except Exception:  # noqa: BLE001  (newer transformers may prefer `dtype`)
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)
    model.to(device).eval()
    return tok, model


def _sync():
    import torch
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def generate_record(model, tok, question, cfg, pool_index):
    """Run the frozen generation protocol for one question. Returns a JSON-serializable dict."""
    import torch

    from .prompting import prompt_input_ids
    from .scoring import teacher_forced_token_scores
    from .seed import seed_for_question, set_seed

    g, s = cfg["generation"]["greedy"], cfg["generation"]["sampling"]
    pids = prompt_input_ids(tok, question).to(model.device)
    attn = torch.ones_like(pids).unsqueeze(0)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    eos = eos_id_set(tok, model)
    n_prompt = int(pids.shape[0])

    # --- greedy answer (deterministic) and its teacher-forced scores
    _sync(); t0 = time.time()
    with torch.no_grad():
        out = model.generate(input_ids=pids.unsqueeze(0), attention_mask=attn, do_sample=False,
                             temperature=None, top_p=None, top_k=None, repetition_penalty=g["repetition_penalty"],
                             max_new_tokens=g["max_new_tokens"], pad_token_id=pad)
    gen = out[0, n_prompt:].cpu()
    text = tok.decode(gen, skip_special_tokens=True)
    tf = teacher_forced_token_scores(model, pids.cpu(), gen)
    _sync(); t_greedy = time.time() - t0

    ids = [int(x) for x in gen]
    is_eos = [i in eos for i in ids]
    short = extract_short_answer(text)
    decode_fn = lambda x: tok.decode(x, skip_special_tokens=True)  # noqa: E731
    spans = token_char_spans(decode_fn, ids)
    mask = answer_span_mask(spans, text, short)
    truncated = (len(ids) >= g["max_new_tokens"]) and not (len(ids) > 0 and is_eos[-1])

    # --- sampled answers (seeded per question; pure sampling, no truncation)
    set_seed(seed_for_question(s["seed_base"], pool_index))
    _sync(); t1 = time.time()
    with torch.no_grad():
        outs = model.generate(input_ids=pids.unsqueeze(0), attention_mask=attn, do_sample=True,
                              temperature=s["temperature"], top_k=s["top_k"], top_p=s["top_p"],
                              repetition_penalty=s["repetition_penalty"], max_new_tokens=s["max_new_tokens"],
                              num_return_sequences=s["num_samples"], pad_token_id=pad)
    _sync(); t_samples = time.time() - t1
    samples = [tok.decode(o[n_prompt:], skip_special_tokens=True) for o in outs]

    return {
        "greedy_text": text, "short_answer": short, "token_ids": ids,
        "tokens": tok.convert_ids_to_tokens(ids),
        "token_logprob": [float(x) for x in tf["logprob"]], "token_entropy": [float(x) for x in tf["entropy"]],
        "is_eos": is_eos, "span_mask": mask, "truncated": bool(truncated),
        "argmax_match_rate": float((tf["argmax"] == gen).float().mean()) if len(ids) else float("nan"),
        "logit_vector_size": tf["vocab"], "samples": samples,
        "seconds_greedy": round(t_greedy, 4), "seconds_samples": round(t_samples, 4),
    }


Writing src/uncertainty/generation.py


In [ ]:
%%writefile src/uncertainty/handcheck.py
"""Hand-check helpers: blind label check (protocol Section 6, step 5) and error-analysis sampling."""
import random

YES = {"y", "yes", "1", "true", "correct", "c"}
NO = {"n", "no", "0", "false", "wrong", "w"}
ERROR_CATEGORIES = ["genuine_model_error", "missing_alias_or_label_noise", "nli_over_fragmentation", "truncation", "other"]


def parse_human_label(x):
    """Return True (answer is correct), False (answer is wrong) or None (blank/unclear)."""
    if x is None:
        return None
    t = str(x).strip().lower()
    if t in YES:
        return True
    if t in NO:
        return False
    return None


def select_for_labelcheck(records, n=30, seed=42):
    """Deterministic sample of n pilot records (sorted by pool_index first). Returns the list sorted by pool_index."""
    recs = sorted(records, key=lambda r: r["pool_index"])
    if len(recs) <= n:
        return recs
    return sorted(random.Random(seed).sample(recs, n), key=lambda r: r["pool_index"])


def agreement(auto_correct, human_correct):
    """auto_correct, human_correct: parallel lists of bool/None. Unclear human labels are excluded and counted."""
    pairs = [(a, h) for a, h in zip(auto_correct, human_correct) if h is not None]
    n = len(pairs)
    agree = sum(1 for a, h in pairs if a == h)
    return {"n_compared": n, "n_unclear_excluded": len(auto_correct) - n, "agreement": (agree / n) if n else float("nan"),
            "auto_correct_human_wrong": sum(1 for a, h in pairs if a and not h),
            "auto_wrong_human_correct": sum(1 for a, h in pairs if (not a) and h)}


def select_error_analysis(table_rows, n_each=30):
    """table_rows: list of dicts with question_id, SE10, y_wrong_exact (test rows). Returns (confident_errors, uncertain_correct):
    the n wrong answers with the LOWEST SE10 and the n correct answers with the HIGHEST SE10 (ties broken by question_id)."""
    wrong = sorted([r for r in table_rows if r["y_wrong_exact"] == 1], key=lambda r: (r["SE10"], r["question_id"]))[:n_each]
    right = sorted([r for r in table_rows if r["y_wrong_exact"] == 0], key=lambda r: (-r["SE10"], r["question_id"]))[:n_each]
    return wrong, right


# ---------------------------------------------------------------- interactive helpers (used from the Colab notebook)
def ask_choice(prompt, choices, input_fn=input, show=print):
    """Ask until the answer (lower-cased, stripped) is one of `choices`."""
    while True:
        ans = str(input_fn(prompt)).strip().lower()
        if ans in choices:
            return ans
        show(f"  please answer one of: {sorted(choices)}")


def interactive_labelcheck(rows, input_fn=input, show=print):
    """Blind label check: shows ONLY question, model answer, gold answer and aliases (never an uncertainty score).
    rows: list of dicts with question, model_answer, gold_value, gold_aliases, human_correct. Rows already filled are skipped.
    Answers: y / n / ? (unclear, stored as 'unclear') / q (stop and keep progress). Returns the rows (modified in place)."""
    for i, r in enumerate(rows, 1):
        if str(r.get("human_correct", "")).strip():
            continue
        show(f"\n[{i}/{len(rows)}] QUESTION: {r['question']}\n    model answer : {r['model_answer']}\n"
             f"    gold answer  : {r['gold_value']}   (aliases: {r['gold_aliases']})")
        a = ask_choice("Is the model answer CORRECT? y / n / ? (unclear) / q (stop and save): ", {"y", "n", "?", "q"}, input_fn, show)
        if a == "q":
            break
        r["human_correct"] = {"y": "yes", "n": "no", "?": "unclear"}[a]
    return rows


def interactive_error_categories(rows, input_fn=input, show=print):
    """Error analysis: assign one category per row (column `category`). Rows already categorized are skipped. q stops and keeps progress."""
    cats = {str(i + 1): c for i, c in enumerate(ERROR_CATEGORIES)}
    legend = "  ".join(f"{k}={v}" for k, v in cats.items())
    for i, r in enumerate(rows, 1):
        if str(r.get("category", "")).strip():
            continue
        show(f"\n[{i}/{len(rows)}] ({r['group']}) QUESTION: {r['question']}\n    model answer : {r['model_answer']}\n"
             f"    gold answer  : {r['gold_value']} | clusters among 10 samples: {r.get('n_clusters', '?')}")
        a = ask_choice(f"Category? {legend} | q=stop and save: ", set(cats) | {"q"}, input_fn, show)
        if a == "q":
            break
        r["category"] = cats[a]
    return rows


Writing src/uncertainty/handcheck.py


In [ ]:
%%writefile src/uncertainty/io_utils.py
"""Resumable JSON-lines storage. A run may be interrupted (Colab disconnects); we resume by question_id."""
import json
from pathlib import Path


def read_jsonl(path):
    """Read all VALID lines. A truncated last line (interrupted write) is ignored."""
    p = Path(path)
    if not p.exists():
        return []
    out = []
    with open(p) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                break
    return out


def repair_jsonl(path):
    """Rewrite the file with only its valid lines (call once at the start of a run). Returns number of valid lines."""
    p = Path(path)
    if not p.exists():
        return 0
    recs = read_jsonl(p)
    with open(p, "w") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
    return len(recs)


def append_jsonl(path, obj):
    with open(path, "a") as f:
        f.write(json.dumps(obj) + "\n")
        f.flush()


def done_ids(path, key="question_id"):
    return {r[key] for r in read_jsonl(path)}


Writing src/uncertainty/io_utils.py


In [ ]:
%%writefile src/uncertainty/latex_tables.py
"""Turn results/final/*.csv into LaTeX tables (so no number in the paper is typed by hand)."""
from pathlib import Path

import pandas as pd


def _ci(v, lo, hi, d=3):
    return f"{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"


def make_tables(table_dir, out_dir):
    table_dir, out_dir = Path(table_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    t2 = pd.read_csv(table_dir / "table2.csv")
    L = ["\\begin{table}[t]", "\\centering\\small\\setlength{\\tabcolsep}{4pt}",
         "\\caption{Detection of wrong answers on the test split: AUROC and AUPRC with 95\\% bootstrap intervals, and ECE after Platt scaling.}",
         "\\label{tab:main}", "\\begin{tabular}{@{}lccc@{}}", "\\toprule", "Score & AUROC & AUPRC & ECE \\\\ \\midrule"]
    for _, r in t2.iterrows():
        ece = "--" if pd.isna(r["ece"]) else f"{r['ece']:.3f}"
        name = str(r["score"]).replace("&", "\\&")
        L.append(f"{name} & {_ci(r['auroc'], r['auroc_lo'], r['auroc_hi'])} & "
                 f"{_ci(r['auprc'], r['auprc_lo'], r['auprc_hi'])} & {ece} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    (out_dir / "table_main.tex").write_text("\n".join(L) + "\n")

    p4 = table_dir / "table4.csv"
    if p4.exists():
        t4 = pd.read_csv(p4)
        name = {"token-only": "Token score only", "primary": "Cascade ($q{=}0.5$)", "SE-for-all": "SE for all questions"}
        L = ["\\begin{table}[t]", "\\centering\\small\\setlength{\\tabcolsep}{4pt}",
             "\\caption{Cascade at the primary operating point versus the two references (counted extra generations per question).}",
             "\\label{tab:cascade}", "\\begin{tabular}{@{}lccc@{}}", "\\toprule",
             "System & AUROC & Escal. & Extra gen. \\\\ \\midrule"]
        for role in ("token-only", "primary", "SE-for-all"):
            r = t4[t4.role == role].iloc[0]
            L.append(f"{name[role]} & {_ci(r['auroc'], r['auroc_lo'], r['auroc_hi'])} & {100 * r['escalated_fraction']:.0f}\\% & "
                     f"{r['extra_generations_per_question']:.1f} \\\\")
        L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
        (out_dir / "table_cascade.tex").write_text("\n".join(L) + "\n")

    t3 = pd.read_csv(table_dir / "table3.csv")
    L = ["\\begin{table}[t]", "\\centering\\small\\setlength{\\tabcolsep}{3pt}",
         "\\caption{Aggregation ablation (AUROC, 95\\% CI) and the three pre-specified contrasts (98.3\\% CI).}",
         "\\label{tab:ablation}", "\\begin{tabular}{@{}lcc@{}}", "\\toprule", "Variant / contrast & Value [CI] & Matters \\\\ \\midrule"]
    for _, r in t3.iterrows():
        m = "" if pd.isna(r["matters"]) else ("yes" if bool(r["matters"]) else "no")
        L.append(f"{str(r['name']).replace('_', ' ')} & {_ci(r['auroc'], r['lo'], r['hi'])} & {m} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    (out_dir / "table_ablation.tex").write_text("\n".join(L) + "\n")

    p6 = table_dir / "table6.csv"
    if p6.exists() and p6.stat().st_size > 5:
        t6 = pd.read_csv(p6)
        L = ["\\begin{table}[t]", "\\centering\\small", "\\caption{Sensitivity analyses (AUROC, 95\\% CI).}", "\\label{tab:sens}",
             "\\begin{tabular}{@{}lc@{}}", "\\toprule", "Analysis & AUROC [CI] \\\\ \\midrule"]
        for _, r in t6.iterrows():
            L.append(f"{r['analysis']} & {_ci(r['auroc'], r['lo'], r['hi'])} \\\\")
        L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
        (out_dir / "table_sensitivity.tex").write_text("\n".join(L) + "\n")
    return sorted(p.name for p in out_dir.glob("*.tex"))


Writing src/uncertainty/latex_tables.py


In [ ]:
%%writefile src/uncertainty/metrics.py
"""Metrics and statistics (protocol Section 9). Pure numpy/scipy/sklearn; unit-tested."""
import numpy as np
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score


def auroc(y, s):
    """AUROC with positive class y==1 (wrong answer); higher score = predicted positive. Ties get average ranks."""
    y = np.asarray(y).astype(bool)
    s = np.asarray(s, dtype=float)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    r = rankdata(s)
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def auprc(y, s):
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan")
    return float(average_precision_score(y, np.asarray(s, dtype=float)))


def bootstrap_indices(n, n_boot, seed):
    """Index matrix [n_boot, n]; reuse the SAME matrix for every score so differences are paired."""
    return np.random.RandomState(seed).randint(0, n, size=(n_boot, n))


def boot_stat(fn, y, s, idx):
    y = np.asarray(y)
    s = np.asarray(s, dtype=float)
    return np.array([fn(y[i], s[i]) for i in idx])


def percentile_ci(values, level=0.95):
    lo = (1 - level) / 2 * 100
    return float(np.nanpercentile(values, lo)), float(np.nanpercentile(values, 100 - lo))


class PlattScaler:
    """One-feature logistic calibration (fit on the CALIBRATION part only)."""

    def fit(self, x, y):
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        y = np.asarray(y).astype(int)
        if len(np.unique(y)) < 2:
            raise ValueError("calibration labels contain only one class")
        self.mu = float(x.mean())
        sd = float(x.std())
        self.sd = sd if sd > 0 else 1.0
        self.lr = LogisticRegression(C=1e6, max_iter=1000).fit((x - self.mu) / self.sd, y)
        return self

    def predict(self, x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        return self.lr.predict_proba((x - self.mu) / self.sd)[:, 1]

    @property
    def slope(self):
        return float(self.lr.coef_[0, 0])


def reliability_table(y, p, n_bins=10):
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 0.0, 1.0)
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    rows = []
    for k in range(n_bins):
        m = b == k
        rows.append({"bin": k, "n": int(m.sum()), "mean_pred": float(p[m].mean()) if m.any() else float("nan"),
                     "frac_wrong": float(y[m].mean()) if m.any() else float("nan")})
    return rows


def ece(y, p, n_bins=10):
    """Expected calibration error with equal-width bins (lower is better)."""
    rows = reliability_table(y, p, n_bins)
    n = sum(r["n"] for r in rows)
    return float(sum(r["n"] / n * abs(r["frac_wrong"] - r["mean_pred"]) for r in rows if r["n"] > 0))


def brier(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))


def random_scores(n, seed):
    return np.random.RandomState(seed).rand(n)


def spearman(x, y):
    rho, pval = spearmanr(x, y)
    return float(rho), float(pval)


Writing src/uncertainty/metrics.py


In [ ]:
%%writefile src/uncertainty/prompting.py
"""Prompt construction (protocol Section 2)."""

PROMPT_TEMPLATE = (
    "Answer the following question with a short answer only (a few words, no explanation).\n"
    "Question: {question}\nAnswer:"
)


def render_chat_prompt(tokenizer, question: str) -> str:
    """Render the model's chat template as text (version-robust; no tensors returned here).

    NOTE: some chat templates insert a default system message when none is given.
    Print the rendered text once and record it in the log (it is part of the frozen prompt).
    """
    messages = [{"role": "user", "content": PROMPT_TEMPLATE.format(question=question.strip())}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def prompt_input_ids(tokenizer, question: str):
    """1-D LongTensor of prompt token ids. add_special_tokens=False because the template already adds them."""
    text = render_chat_prompt(tokenizer, question)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0]


Writing src/uncertainty/prompting.py


In [ ]:
%%writefile src/uncertainty/report.py
"""Prompt 14: builds the Results-and-Discussion OUTLINE from real output files, with rule-based, non-exaggerating wording.
It computes NO new statistics: every number is copied from results/final/*.csv|json. Mechanisms and alternative explanations are
listed as things to CHECK, never as findings; [AUTHOR] marks what only the author can decide after looking at the evidence."""
import math

import pandas as pd


def f3(x):
    return "n/a" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.3f}"


def ci(v, lo, hi):
    return f"{f3(v)} (95% CI {f3(lo)} to {f3(hi)})"


def _row(t, col, val):
    r = t[t[col] == val]
    return r.iloc[0] if len(r) else None


def build_outline(summary, t2, t3, t4, t6=None, e1=None, t5=None, handcheck=None):
    S, L = summary, []
    p = S.get("provenance") or {}
    rq1, rq2, rq3 = S["rq1"], S["rq2"], S["rq3"]
    L.append("# Results and Discussion outline (generated from real output files; no new analysis)\n")
    L.append(f"Source: `results/per_question.csv` sha256 {str(p.get('input_sha256'))[:16]}... | commit {p.get('git_commit')} | tag {p.get('protocol_tag')} | "
             f"audit passed: {p.get('audit_passed')} | audit overridden: {p.get('audit_overridden')}\n")
    if not p.get("audit_passed") or p.get("audit_overridden"):
        L.append("> **WARNING: the pre-evaluation audit did not pass or was overridden. Do not interpret these numbers until it passes.**\n")
    for fl in S.get("flags", []):
        L.append(f"> FLAG: {fl}\n")
    n, nw = S["n_eval"], S["n_wrong"]
    L.append(f"## 0. What was evaluated\nTest part: {n} questions, {nw} wrong ({nw / n:.1%}) and {S['n_correct']} correct. Paired bootstrap, "
             f"{S['bootstrap']['n']} resamples, seed {S['bootstrap']['seed']}. Label = normalized exact match with a TriviaQA alias "
             "(not the same thing as 'hallucination'). Metric = AUROC, positive class = wrong answer, higher score = more uncertain.\n")

    # ------------------------------------------------------------------ RQ1
    tok, se = _row(t2, "key", "token"), _row(t2, "key", "se")
    L.append("## 1. RQ1: token score versus semantic entropy\n### Direct observations (copied from the tables)")
    L.append(f"- Mean NLL (primary token score): AUROC {ci(tok.auroc, tok.auroc_lo, tok.auroc_hi)}")
    L.append(f"- Semantic entropy (N=10): AUROC {ci(se.auroc, se.auroc_lo, se.auroc_hi)}")
    L.append(f"- Difference (SE minus token): {ci(rq1['diff_se_minus_token'], rq1['diff_ci'][0], rq1['diff_ci'][1])}")
    for k, nm in (("length", "Length-only control"), ("random", "Random control")):
        r = _row(t2, "key", k)
        L.append(f"- {nm}: AUROC {ci(r.auroc, r.auroc_lo, r.auroc_hi)}")
    lo_d, hi_d = rq1["diff_ci"]
    L.append("\n### Verdict by the pre-registered rule")
    L.append(f"- Token score predictive (lower CI bound above 0.5): **{'yes' if rq1['token_is_predictive'] else 'no'}**")
    L.append(f"- H1 (semantic entropy has the higher AUROC): **{rq1['H1']}**")
    se_lo = float(se.auroc_lo)
    L.append("\n### Wording the evidence allows")
    if se_lo <= 0.5:
        L.append("- The semantic-entropy interval includes 0.5, so semantic entropy is **not shown to be predictive here**. Check the pipeline "
                 "(label audit, cluster viewer, number of one-cluster questions) **before** drawing any scientific conclusion.")
    elif not rq1["token_is_predictive"]:
        L.append(f"- In this setting there is no evidence that the mean-NLL score predicts wrong answers (its interval includes 0.5). "
                 f"Semantic entropy reached AUROC {f3(se.auroc)}.")
        if lo_d > 0:
            L.append("- H1 holds by the pre-registered rule only in this narrow sense: semantic entropy exceeded a score that was **not shown to be above chance**. "
                     "That is not evidence that semantic entropy beats a token score that works.")
    elif lo_d > 0:
        L.append(f"- In this setting semantic entropy had a higher AUROC than mean NLL by {f3(rq1['diff_se_minus_token'])} "
                 f"(95% interval {f3(lo_d)} to {f3(hi_d)}). H1 is supported **for this model, dataset, prompt and label rule**.")
    elif hi_d < 0:
        L.append(f"- Mean NLL had a higher AUROC than semantic entropy by {f3(-rq1['diff_se_minus_token'])}; this **contradicts H1** and the "
                 "review's framing in this setting. Report it as such; do not reinterpret it as support.")
    else:
        L.append("- The interval of the difference includes 0: there is **no evidence of a difference** at this sample size. H1 is not supported.")
    L.append("\n### Not supported by this evidence (do not write)")
    L.append("- 'Semantic entropy is better than token scores' in general, or for other models/datasets/long-form text.")
    if lo_d <= 0 <= hi_d:
        L.append("- 'The two are equivalent/comparable': an interval that includes 0 is **not** evidence of equivalence (no equivalence margin was pre-registered).")
    if not rq1["token_is_predictive"]:
        L.append("- 'Token-level scores do not work': only that this score on this setup was not shown to predict errors.")
    L.append("- Any causal explanation (why one signal is better) as an observed fact.")
    L.append("\n### Alternative explanations to check before writing the discussion")
    L.append("- Label noise (hand-check agreement; error-analysis category 'missing alias / label noise').")
    L.append("- Low diversity of sampled answers (share of questions with one cluster; semantic entropy is then 0 even for confident wrong answers).")
    L.append("- NLI over-fragmentation of short answers (cluster viewer; error-analysis category).")
    L.append("- Length confound (Section 2) and prevalence of wrong answers (AUPRC must be read against it).")
    L.append("- Possible TriviaQA contamination in the model's pre-training (cannot be checked).")
    L.append("\n### Mechanism hypotheses (NOT findings; test or label as conjecture)")
    L.append("- [AUTHOR: state at most one or two conjectures and the evidence that would test each.]\n")

    # ------------------------------------------------------------------ RQ2
    L.append("## 2. RQ2: aggregation and length (exploratory; no hypothesis)\n### Direct observations")
    for _, r in t3[t3.kind == "variant"].iterrows():
        L.append(f"- {r['name']}: AUROC {ci(r.auroc, r.lo, r.hi)}")
    L.append("\n### Pre-specified contrasts (98.3% intervals; only these three are confirmatory)")
    for name, c in rq2["contrasts"].items():
        verdict = ("**matters** (interval excludes 0 and difference at least 0.02)" if c["matters"]
                   else "no evidence that it matters (interval includes 0 or difference below 0.02); this is **not** evidence of equality")
        L.append(f"- {name}: difference {ci(c['difference'], c['ci'][0], c['ci'][1])} -> {verdict}")
    L.append(f"\n- Length-only AUROC {f3(rq2['length_only_auroc'])} (lower CI bound {f3(rq2['length_auroc_ci_lower'])}): length confound "
             f"**{'present' if rq2['length_confound_present'] else 'not shown'}** by the pre-registered rule.")
    L.append(f"- Spearman correlation between the primary score and token count: {f3(rq2['spearman_token_score_vs_length'])} (descriptive; p-value assumes independent questions and is approximate with ties).")
    for st in rq2.get("length_strata", []):
        L.append(f"  - length stratum {st['stratum']} (tokens {st['token_count_range'][0]:.0f} to {st['token_count_range'][1]:.0f}): n={st['n']}, wrong={st['n_wrong']}, "
                 f"AUROC token {f3(st['auroc_token'])}, SE {f3(st['auroc_se'])} (no intervals: small strata)")
    L.append("\n### Not supported\n- Ranking the ten aggregators: all comparisons other than the three contrasts are descriptive.\n"
             "- 'Answer-span scoring helps/does not help' beyond contrast C1 as stated above.")
    L.append("\n### Alternative explanations to check\n- Most answers are only a few tokens long, so answer-span and all-token scores may be nearly identical (share of `span_fallback` rows).\n"
             "- Ties in few-token answers make maximum and minimum aggregators coarse.\n")

    # ------------------------------------------------------------------ RQ3
    prim = _row(t4, "role", "primary")
    tonly, seall = _row(t4, "role", "token-only"), _row(t4, "role", "SE-for-all")
    L.append("## 3. RQ3: token-gated cascade (offline simulation from stored scores)\n### Direct observations")
    L.append(f"- Token only: AUROC {ci(tonly.auroc, tonly.auroc_lo, tonly.auroc_hi)}; extra generations per question 0")
    L.append(f"- Cascade at the primary operating point: AUROC {ci(prim.auroc, prim.auroc_lo, prim.auroc_hi)}; escalated fraction {prim.escalated_fraction:.1%}; "
             f"extra generations per question {prim.extra_generations_per_question:.2f}; NLI calls per question {prim.nli_calls_per_question:.2f} (counted)")
    L.append(f"- Semantic entropy for all questions: AUROC {ci(seall.auroc, seall.auroc_lo, seall.auroc_hi)}; extra generations per question "
             f"{seall.extra_generations_per_question:.2f}")
    L.append(f"- Measured seconds per question: token only {f3(tonly.seconds_per_question)}, cascade {f3(prim.seconds_per_question)}, SE for all {f3(seall.seconds_per_question)} "
             "(one GPU/session; not portable to other hardware)")
    L.append(f"- D = AUROC(SE) minus AUROC(cascade): {ci(rq3['D'], rq3['D_ci'][0], rq3['D_ci'][1])}; margin {rq3['margin']}, upper limit {rq3['upper_ci_limit']}")
    L.append(f"\n### Verdict by the pre-registered rule\n- H3: **{rq3['H3']}**")
    L.append("\n### Wording the evidence allows")
    if rq3["H3"] == "supported":
        L.append(f"- At the primary operating point the cascade kept AUROC within {rq3['margin']} of full semantic entropy (D = {f3(rq3['D'])}; upper bound {f3(rq3['D_ci'][1])}) "
                 f"while escalating {prim.escalated_fraction:.0%} of questions. This holds for the offline simulation in this setting only.")
    elif rq3["H3"] == "inconclusive":
        L.append("- The point estimate of the loss is below the margin but the interval cannot exclude a larger loss: the data are **inconclusive**. "
                 "Do not write that the cascade 'preserves' detection quality.")
    else:
        L.append(f"- The cascade lost at least {rq3['margin']} AUROC relative to full semantic entropy at the primary operating point: H3 is **not supported**. "
                 "Report the compute saved and the quality lost together; do not turn this into a positive claim about efficiency alone.")
    if not rq1["token_is_predictive"]:
        L.append("- RQ1 found no evidence that the gate score predicts errors; a cascade that depends on it was expected to fail (pre-stated).")
    L.append("\n### Not supported\n- 'The cascade reduces latency' as a deployment claim: costs are counted/measured offline on one GPU and do not include batching, caching or engineering overhead.\n"
             "- Any claim about operating points other than the primary one beyond describing the curve (they were not tested as hypotheses).")
    L.append("\n### Alternative explanations to check\n- Platt slope signs (flags above): a negative slope inverts the calibrated ranking of one stage.\n"
             "- Ties in semantic entropy (few distinct values) and in token scores at the threshold.\n- The gate selecting questions by length rather than uncertainty.\n")

    # ------------------------------------------------------------------ E1
    L.append("## 4. E1: length-adjusted gate score (pre-registered, exploratory)")
    if e1 is None:
        L.append("- `extension_E1.json` not found: E1 was not run.\n")
    elif not e1.get("ran"):
        L.append(f"- Not run: {e1.get('reason')}. This is a result of the pre-registered trigger, not a failure.\n")
    else:
        L.append(f"- AUROC adjusted {f3(e1['auroc_adjusted'])} versus primary {f3(e1['auroc_primary'])}; difference {ci(e1['difference'], e1['difference_ci'][0], e1['difference_ci'][1])}; verdict: **{e1['verdict']}** "
                 "(exploratory secondary analysis, one contrast). Do not describe it as an improvement unless the verdict says so.\n")

    # ------------------------------------------------------------------ cost, variation, errors
    L.append("## 5. Variation across runs\nThere is **one stored generation run**. The intervals describe question-sampling variability only; they do not include "
             "the random seed of the sampling, the model, the prompt or the dataset. [AUTHOR: if a rerun comparison exists (scripts/09), report its result here.]\n")
    L.append("## 6. Error and failure modes")
    if t5 is not None and len(t5):
        for _, r in t5.iterrows():
            L.append(f"- {r['group']} / {r['category']}: {int(r['count'])}")
        L.append("Interpretation: [AUTHOR: say which categories dominate and what that implies for the label rule and for the claims above.]\n")
    else:
        L.append("- Error analysis not yet done (`scripts/05_handcheck.py --make-erroranalysis --frozen`). Required before the discussion.\n")
    if handcheck:
        L.append(f"- Label hand-check: agreement {f3(handcheck.get('agreement'))} on {handcheck.get('n_compared')} answers (target {handcheck.get('target')}).")
    if t6 is not None and len(t6):
        L.append("\n### Sensitivity (descriptive)")
        for _, r in t6.iterrows():
            L.append(f"- {r['analysis']}: AUROC {ci(r.auroc, r.lo, r.hi)}")
        L.append("")

    # ------------------------------------------------------------------ limitations
    L.append("## 7. Limitations (state in the paper)")
    for t in ["One small instruction-tuned model, one short-form closed-book dataset, one prompt: no claim beyond this setting (not long-form, multimodal or long-context).",
              "The label is alias matching: a correct answer worded differently counts as wrong; 'wrong answer' is not 'hallucination' or 'confabulation'.",
              "AUROC measures ranking of wrong above correct answers; it does not measure calibration, accuracy-coverage trade-offs or the usefulness of abstention. ECE depends on the binning and on a Platt fit on a small calibration part.",
              "Semantic entropy measures consistency of meaning across samples, not truth: consistently wrong answers get low entropy.",
              "Decision margins (0.02, 0.03, 50%) are my own choices, not derived from the literature.",
              "Single stored run; no estimate of seed-to-seed variance; GPU non-determinism possible.",
              "The cascade is an offline simulation; costs are counted and measured on one GPU.",
              "TriviaQA terms are unspecified by the authors; possible contamination of the model's pre-training cannot be checked; the dataset revision is not pinned.",
              "Hand-check covers 30 pilot answers only."]:
        L.append(f"- {t}")
    L.append("")

    # ------------------------------------------------------------------ follow-ups (max 3)
    wide = float((t2["auroc_hi"] - t2["auroc_lo"]).max()) > 0.10
    if se_lo <= 0.5 or not rq1["token_is_predictive"]:
        first = "A"
    elif rq3["H3"] == "inconclusive" or wide:
        first = "B"
    else:
        first = "C"
    L.append("## 8. Follow-up experiments (at most three; ranked by cost, with likely information gained)")
    L.append("| Rank by cost | Follow-up | Cost | Likely information gained |\n|---|---|---|---|")
    L.append("| 1 (cheapest) | A. Re-analysis of the stored data (cluster-count distribution, label-noise audit, alternative gate scores) **labelled post hoc/exploratory** | no GPU | Separates 'real negative result' from 'pipeline artefact'; weak confirmatory value |")
    L.append("| 2 | B. Second sampling seed and a larger pool of questions | about the cost of the main run (UNKNOWN until timed) | Estimates seed variance and narrows the intervals |")
    L.append("| 3 (most expensive) | C. A second model size or a second closed-book dataset | highest | Tests whether any pattern generalizes |")
    names = {"A": "A (re-analysis of stored data)", "B": "B (second seed, larger pool)", "C": "C (second model or dataset)"}
    why = {"A": "a key result is a negative or null one, so first rule out a pipeline artefact",
           "B": "the cascade verdict is inconclusive or intervals are wide, so more data and a variance estimate carry the most information",
           "C": "the within-setting result is clear, so generalization is the main open question"}
    L.append(f"\nSuggested first, given these results: **{names[first]}**, because {why[first]}.\n")
    return "\n".join(L)


Writing src/uncertainty/report.py


In [ ]:
%%writefile src/uncertainty/repro.py
"""Reproducibility comparison of two generation runs on the same questions."""


def compare_generation_runs(a, b, lp_tol=0.05):
    A = {r["question_id"]: r for r in a}
    B = {r["question_id"]: r for r in b}
    common = sorted(set(A) & set(B))
    same_greedy = sum(A[q]["greedy_text"] == B[q]["greedy_text"] for q in common)
    same_samples = sum(A[q]["samples"] == B[q]["samples"] for q in common)
    diffs = [abs(x - y) for q in common if len(A[q]["token_logprob"]) == len(B[q]["token_logprob"])
             for x, y in zip(A[q]["token_logprob"], B[q]["token_logprob"])]
    max_diff = max(diffs) if diffs else float("nan")
    n = len(common)
    return {"n_common": n, "greedy_identical": same_greedy, "samples_identical": same_samples,
            "max_abs_token_logprob_diff": max_diff,
            "greedy_ok": n > 0 and same_greedy == n, "logprob_ok": bool(diffs) and max_diff <= lp_tol,
            "samples_ok": n > 0 and same_samples == n}


Writing src/uncertainty/repro.py


In [ ]:
%%writefile src/uncertainty/scoring.py
"""Token-level scores (protocol Section 5): teacher-forced log-probabilities and entropies."""
import math

import numpy as np


def teacher_forced_token_scores(model, prompt_ids, answer_ids):
    """Score `answer_ids` given `prompt_ids` with ONE forward pass on raw logits (no logit processors).

    prompt_ids, answer_ids: 1-D LongTensors. answer_ids may end with the end-of-turn token.
    Returns dict of CPU float32/long tensors, one entry per answer token:
      logprob  : log p(token | prefix)
      entropy  : Shannon entropy (nats) of the full next-token distribution at that step
      argmax   : most likely token id at that step (for greedy decoding it should equal the token)
      vocab    : size of the logit vector (model vocabulary dimension)
    """
    import torch

    with torch.no_grad():
        device = model.device
        n_prompt = int(prompt_ids.shape[0])
        n_ans = int(answer_ids.shape[0])
        input_ids = torch.cat([prompt_ids, answer_ids]).unsqueeze(0).to(device)
        logits = model(input_ids=input_ids).logits[0]          # [L, V]
        pred = logits[n_prompt - 1 : n_prompt + n_ans - 1].float()  # logits that predict answer tokens
        logp = torch.log_softmax(pred, dim=-1)                  # float32
        ans = answer_ids.to(device).unsqueeze(1)
        token_lp = logp.gather(1, ans).squeeze(1)
        entropy = -(logp.exp() * logp).sum(dim=-1)
        return {
            "logprob": token_lp.cpu(),
            "entropy": entropy.cpu(),
            "argmax": pred.argmax(dim=-1).cpu(),
            "vocab": int(pred.shape[-1]),
        }


def aggregate_scores(token_logprobs, token_entropies) -> dict:
    """Aggregators from the protocol. Inputs: sequences of per-token log-probs and entropies
    for the tokens being scored. Higher = more uncertain for every returned value."""
    lp = np.asarray(token_logprobs, dtype=np.float64)
    ent = np.asarray(token_entropies, dtype=np.float64)
    if lp.size == 0:
        nan = float("nan")
        return {"mean_nll": nan, "sum_nll": nan, "max_nll": nan, "mean_entropy": nan, "max_entropy": nan, "n_tokens": 0}
    nll = -lp
    return {
        "mean_nll": float(nll.mean()),
        "sum_nll": float(nll.sum()),
        "max_nll": float(nll.max()),      # = -log(min token probability)
        "mean_entropy": float(ent.mean()),
        "max_entropy": float(ent.max()),
        "n_tokens": int(lp.size),
    }


def worked_example_from_review():
    """The review's illustrative numbers: token probabilities 0.80, 0.60, 0.90."""
    probs = [0.80, 0.60, 0.90]
    lps = [math.log(p) for p in probs]
    return {
        "joint_probability": float(np.prod(probs)),
        "mean_nll": float(-np.mean(lps)),
    }


Writing src/uncertainty/scoring.py


In [ ]:
%%writefile src/uncertainty/seed.py
"""Random-seed control. Seeds python, numpy and (if installed) torch/CUDA."""
import os
import random

import numpy as np


def set_seed(seed: int, deterministic: bool = False) -> dict:
    """Seed all random number generators we use.

    deterministic=True additionally asks PyTorch for deterministic algorithms
    (can be slower and may raise for some operations; we use warn_only=True).
    Returns a small dict describing what was seeded, for the log.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    info = {"seed": seed, "torch": False, "cuda": False, "deterministic": deterministic}
    try:
        import torch

        torch.manual_seed(seed)
        info["torch"] = True
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            info["cuda"] = True
        if deterministic:
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
            torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass
    return info


def seed_for_question(base: int, index: int) -> int:
    """Per-question sampling seed from the protocol: base (1000) + question index in the pool."""
    return int(base) + int(index)


Writing src/uncertainty/seed.py


In [ ]:
%%writefile src/uncertainty/semantic_entropy.py
"""Cycle 5: semantic entropy (discrete variant) by greedy NLI clustering of sampled answers (protocol Section 7).

`cluster_samples` is pure Python: it takes an `entails(question, a, b) -> bool` function, so it is unit-tested with a fake.
`NLIEntailment` wraps DeBERTa-large-MNLI [TEST REQUIRED on GPU].
"""
import math
from collections import Counter

from .textnorm import extract_short_answer, normalize_answer


def cluster_samples(question, samples, entails):
    """Greedy sequential clustering.
    A sample joins the FIRST existing cluster whose representative (its first member) it matches:
      - identical normalized strings match without an NLI call (documented shortcut), otherwise
      - strict bidirectional entailment: entails(rep, sample) AND entails(sample, rep).
    Empty answers never go through NLI (they only match other empty answers).
    Returns (cluster_ids, n_nli_calls) where n_nli_calls counts directional entailment evaluations."""
    reps, ids, calls = [], [], 0
    for s in samples:
        short = extract_short_answer(s)
        norm = normalize_answer(short)
        assigned = None
        for ci, (rep_short, rep_norm) in enumerate(reps):
            if norm == rep_norm:
                assigned = ci
                break
            if norm == "" or rep_norm == "":
                continue
            calls += 1
            if not entails(question, rep_short, short):
                continue
            calls += 1
            if entails(question, short, rep_short):
                assigned = ci
                break
        if assigned is None:
            reps.append((short, norm))
            assigned = len(reps) - 1
        ids.append(assigned)
    return ids, calls


def semantic_entropy_from_ids(cluster_ids, k=None):
    """Discrete semantic entropy (nats) from cluster ids of the first k samples (all if k is None).
    Because clustering is greedy and sequential, the first k ids equal the clustering of the first k samples alone."""
    ids = list(cluster_ids if k is None else cluster_ids[:k])
    n = len(ids)
    if n == 0:
        return float("nan")
    counts = Counter(ids)
    return float(-sum((c / n) * math.log(c / n) for c in counts.values()))


class NLIEntailment:
    """entails(question, a, b): True if the NLI model's argmax label for (premise='q a', hypothesis='q b') is entailment."""

    def __init__(self, model_name, device, max_length=256):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
        self.device, self.max_length = device, max_length
        id2label = {int(k): v for k, v in self.model.config.id2label.items()}
        hits = [i for i, lab in id2label.items() if "entail" in lab.lower()]
        if len(hits) != 1:
            raise RuntimeError(f"cannot identify the entailment label in {id2label}")
        self.entail_idx, self.id2label = hits[0], id2label
        self.cache = {}
        self.n_model_calls = 0

    def __call__(self, question, a, b):
        key = (question, a, b)
        if key in self.cache:
            return self.cache[key]
        enc = self.tok(f"{question} {a}", f"{question} {b}", return_tensors="pt", truncation=True,
                       max_length=self.max_length).to(self.device)
        with self.torch.no_grad():
            pred = int(self.model(**enc).logits[0].argmax())
        self.n_model_calls += 1
        self.cache[key] = (pred == self.entail_idx)
        return self.cache[key]


Writing src/uncertainty/semantic_entropy.py


In [ ]:
%%writefile src/uncertainty/table.py
"""Cycle 4: merge raw generations + semantic-entropy results into the per-question table.
The public table contains NO question text, NO gold answers and NO model answer strings (a correct answer equals the gold
answer, and TriviaQA's terms are unspecified); those stay in the local, git-ignored results/raw/ files."""
import numpy as np
import pandas as pd

from .scoring import aggregate_scores
from .semantic_entropy import semantic_entropy_from_ids
from .textnorm import is_correct, is_correct_containment

NAME = {"mean_nll": "meanNLL", "sum_nll": "sumNLL", "max_nll": "maxNLL", "mean_entropy": "meanH", "max_entropy": "maxH"}
TOKEN_VARIANTS = [f"{ts}_{a}" for ts in ("all", "span") for a in NAME.values()]


def row_from_records(gen, sem=None):
    """gen: one raw generation record (see generation.generate_record plus question_id/pool_index/split/is_pilot and
    normalized_aliases). sem: optional semantic-entropy record."""
    lp, ent, eos, mask = gen["token_logprob"], gen["token_entropy"], gen["is_eos"], gen["span_mask"]
    keep_all = [i for i, e in enumerate(eos) if not e]
    keep_span = [i for i in keep_all if mask[i]]
    fallback = len(keep_span) == 0 and len(keep_all) > 0
    if fallback:
        keep_span = keep_all           # answer text not located: use all tokens and flag it
    row = {"question_id": gen["question_id"], "pool_index": gen["pool_index"], "split": gen["split"],
           "is_pilot": bool(gen["is_pilot"]), "n_tokens": len(keep_all),
           "truncated": bool(gen["truncated"]), "span_fallback": bool(fallback),
           "argmax_match_rate": gen.get("argmax_match_rate", np.nan)}
    for ts, keep in (("all", keep_all), ("span", keep_span)):
        agg = aggregate_scores([lp[i] for i in keep], [ent[i] for i in keep])
        for k, nm in NAME.items():
            row[f"{ts}_{nm}"] = agg[k]
    aliases = gen["normalized_aliases"]
    row["y_wrong_exact"] = int(not is_correct(gen["greedy_text"], aliases))
    row["y_wrong_contain"] = int(not is_correct_containment(gen["greedy_text"], aliases))
    row["seconds_greedy"] = gen["seconds_greedy"]
    row["seconds_samples"] = gen["seconds_samples"]
    if sem is not None:
        ids = sem["cluster_ids"]
        row.update({"n_clusters": len(set(ids)), "SE10": semantic_entropy_from_ids(ids),
                    "SE5": semantic_entropy_from_ids(ids, k=5), "nli_calls": sem["nli_calls"],
                    "seconds_nli": sem["seconds_nli"]})
    else:
        row.update({"n_clusters": np.nan, "SE10": np.nan, "SE5": np.nan, "nli_calls": np.nan, "seconds_nli": np.nan})
    return row


def build_table(gens, sems):
    """gens: list of raw generation records; sems: list of semantic records. Returns DataFrame sorted by pool_index."""
    by_id = {s["question_id"]: s for s in sems}
    rows = [row_from_records(g, by_id.get(g["question_id"])) for g in gens]
    return pd.DataFrame(rows).sort_values("pool_index").reset_index(drop=True)


Writing src/uncertainty/table.py


In [ ]:
%%writefile src/uncertainty/textnorm.py
"""Answer extraction, normalization and labelling.

Normalization follows the OFFICIAL TriviaQA evaluation normalization (as reproduced in the T5 authors' qa_utils.py, which states
that it matches the official script): lowercase; replace every punctuation character, including the characters
(left/right single quote, acute accent, backtick, underscore), by a SPACE; remove the articles a/an/the; collapse whitespace.
NOTE: this differs from the SQuAD normalization, which DELETES punctuation ("U.S.A." -> "usa"). The dataset's
`normalized_aliases` follow the TriviaQA style, so using the SQuAD style would mislabel correct answers as wrong.
Still to do (checklist): compare once with the official script file in the TriviaQA repository itself.
"""
import re
import string

_PUNCT = set(string.punctuation + "\u2018\u2019\u00b4`_")
_ARTICLES = re.compile(r"\b(a|an|the)\b")


def normalize_answer(text: str) -> str:
    text = text.lower()
    text = "".join(" " if ch in _PUNCT else ch for ch in text)
    text = _ARTICLES.sub(" ", text)
    return " ".join(text.split())


def normalize_squad_style(text: str) -> str:
    """Kept ONLY to demonstrate the difference in tests/sensitivity checks. Not used for labels."""
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def extract_short_answer(generated_text: str) -> str:
    """Strip, keep the first line, remove trailing periods (protocol Section 6, step 1)."""
    text = generated_text.strip()
    if not text:
        return ""
    first_line = text.split("\n")[0].strip()
    return first_line.rstrip(".").strip()


def is_correct(generated_text: str, normalized_aliases) -> bool:
    """Exact-match label: True if the normalized short answer equals any normalized alias."""
    answer = normalize_answer(extract_short_answer(generated_text))
    aliases = {normalize_answer(a) for a in normalized_aliases}
    return answer in aliases and answer != ""


def is_correct_containment(generated_text: str, normalized_aliases) -> bool:
    """Sensitivity label: a normalized alias occurs inside the normalized answer (word-boundary match)."""
    answer = normalize_answer(extract_short_answer(generated_text))
    if not answer:
        return False
    padded = f" {answer} "
    return any(f" {normalize_answer(a)} " in padded for a in normalized_aliases if normalize_answer(a))


Writing src/uncertainty/textnorm.py


In [ ]:
%%writefile scripts/_common.py
"""Shared helpers for the pipeline scripts."""
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "results" / "raw"
GEN = RAW / "generations.jsonl"
SEM = RAW / "semantic.jsonl"


def load_cfg():
    return yaml.safe_load(open(ROOT / "configs" / "protocol_v1.yaml"))


def git_commit():
    try:
        return subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True, timeout=10).stdout.strip() or None
    except Exception:  # noqa: BLE001
        return None


def git_tag_exists(tag="protocol-v1"):
    try:
        r = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", f"refs/tags/{tag}"], capture_output=True, text=True, timeout=10)
        return r.returncode == 0
    except Exception:  # noqa: BLE001
        return False


def select_rows(df, split):
    """split in pilot|calibration|test|all. Pilot rows first, then by pool_index."""
    if split == "pilot":
        sel = df[df["is_pilot"].astype(bool)]
    elif split == "calibration":
        sel = df[df["split"] == "calibration"]
    elif split == "test":
        sel = df[df["split"] == "test"]
    else:
        sel = df
    return sel.assign(_p=~sel["is_pilot"].astype(bool)).sort_values(["_p", "pool_index"]).drop(columns="_p")


def require_test_permission(split, allowed):
    if split in ("test", "all") and not allowed:
        raise SystemExit("Refusing to touch the TEST part without --allow-test.\n"
                         "Protocol: run the 50-question pilot first (it decides pool size 1000 vs 400 from timing only), "
                         "complete experiment_protocol.md Section 14 and tag protocol-v1.")


def load_subset_with_text(cfg):
    """Rebuild the subset (with question text and aliases) from the saved ids and a fresh download; verify the manifest hash."""
    from uncertainty import data as D
    ids = pd.read_csv(ROOT / "data" / "subset_question_ids.csv", dtype={"question_id": str})
    manifest = json.load(open(ROOT / "data" / "subset_manifest.json"))
    if D.ids_hash(ids) != manifest["ids_sha256"]:
        raise SystemExit("data/subset_question_ids.csv does not match its manifest hash: do not continue.")
    from datasets import load_dataset
    ds = cfg["dataset"]
    raw = load_dataset(ds["hf_name"], ds["config"], split=ds["split"])
    clean, _ = D.dedupe(D.rows_from_records(raw))
    clean, _ = D.drop_unlabellable(clean)
    return D.attach_text(ids, clean).sort_values("pool_index").reset_index(drop=True)


def code_files():
    return sorted([*ROOT.glob("configs/*.yaml"), *ROOT.glob("src/uncertainty/*.py"), *ROOT.glob("scripts/*.py")])


def code_digest():
    h = hashlib.sha256()
    for p in code_files():
        h.update(str(p.relative_to(ROOT)).encode())
        h.update(p.read_bytes())
    return h.hexdigest()


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def freeze_record():
    p = ROOT / "data" / "protocol_freeze.json"
    return json.load(open(p)) if p.exists() else None


def freeze_info():
    rec = freeze_record()
    return {"tag_exists": git_tag_exists(), "record": rec, "code_matches_record": (rec["sha256"] == code_digest()) if rec else None}


def audit_from_disk(mode, cfg, strict_freeze=True):
    """Run the pre-evaluation audit on whatever is on disk (inputs that are missing make their checks SKIPPED, never passed)."""
    from uncertainty.audit import load_audit_inputs, run_audit
    df = pd.read_csv(ROOT / "results" / "per_question.csv")
    return run_audit(df, cfg, mode, freeze=freeze_info() if mode == "final" else None, strict_freeze=strict_freeze,
                     **load_audit_inputs(ROOT))


Writing scripts/_common.py


In [ ]:
%%writefile scripts/00_env_test.py
#!/usr/bin/env python
"""Environment test for the token-uncertainty project (protocol-v1).

Run from the repository root:   python scripts/00_env_test.py
Useful flags: --skip-dataset  --skip-nli  --allow-cpu  --n-questions 5

What it does (each stage is independent and reports PASS / WARN / FAIL):
  1 system        Python, OS, RAM, disk, nvidia-smi
  2 libraries     versions of the packages we use; transformers >= 4.37 for Qwen2.5
  3 gpu           CUDA visible, device name, memory, small fp16 matrix multiply
  4 seed          seeding gives identical random numbers twice
  5 config        configs/protocol_v1.yaml loads and has the key settings
  6 dataset       TriviaQA rc.nocontext validation: fields, size, duplicate question_ids
  7 model         loads Qwen2.5-1.5B-Instruct in fp16; prints generation_config and rendered prompt
  8 greedy        greedy answers + teacher-forced token scores + consistency check vs generate-time scores
  9 sampling      10 samples at T=1 (no truncation); same seed twice gives same samples?
 10 nli           loads deberta-large-mnli; prints label order; 3 hand-made pairs
 11 report        writes results/env_report.json, results/requirements_candidate.txt, results/pip_freeze.txt

Nothing here prints tokens or environment variables. Output is also saved to logs/env_test_output.txt.
"""
import argparse
import importlib
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from importlib import metadata
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

RESULTS = []
CTX = {}


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()

    def flush(self):
        for s in self.streams:
            s.flush()


def run_stage(name, fn, *args, **kwargs):
    t0 = time.time()
    print(f"\n=== [{name}] ===")
    try:
        status, detail = fn(*args, **kwargs)
    except Exception as exc:  # noqa: BLE001  (we want to keep going and report)
        status, detail = "FAIL", {"error": f"{type(exc).__name__}: {exc}"}
        print("FAIL:", detail["error"])
        traceback.print_exc(limit=4, file=sys.stdout)
    dt = time.time() - t0
    RESULTS.append({"stage": name, "status": status, "seconds": round(dt, 2), "detail": detail})
    print(f"--> {name}: {status} ({dt:.1f}s)")
    return status


def dist_version(dist_name):
    try:
        return metadata.version(dist_name)
    except metadata.PackageNotFoundError:
        return None


# ---------------------------------------------------------------- stages
def stage_system():
    d = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_count": os.cpu_count()}
    print("Python  :", d["python"])
    print("Platform:", d["platform"])
    print("CPUs    :", d["cpu_count"])
    try:
        with open("/proc/meminfo") as f:
            mem_kb = int(f.readline().split()[1])
        d["ram_gb"] = round(mem_kb / 1024 / 1024, 1)
        print("RAM (GB):", d["ram_gb"])
    except Exception:  # noqa: BLE001
        print("RAM     : not readable on this OS")
    total, used, free = shutil.disk_usage(str(ROOT))
    d["disk_free_gb"] = round(free / 1e9, 1)
    print("Disk free (GB):", d["disk_free_gb"])
    smi = shutil.which("nvidia-smi")
    if not smi:
        print("nvidia-smi not found: no NVIDIA GPU visible to this runtime.")
        d["nvidia_smi"] = None
        return "WARN", d
    out = subprocess.run([smi], capture_output=True, text=True, timeout=30).stdout
    print(out)
    d["nvidia_smi"] = out.splitlines()[:12]
    return "PASS", d


def stage_libraries():
    pkgs = [("torch", "torch"), ("transformers", "transformers"), ("datasets", "datasets"),
            ("accelerate", "accelerate"), ("numpy", "numpy"), ("scipy", "scipy"),
            ("scikit-learn", "scikit-learn"), ("pandas", "pandas"), ("PyYAML", "PyYAML"),
            ("matplotlib", "matplotlib")]
    versions, missing = {}, []
    for label, dist in pkgs:
        v = dist_version(dist)
        versions[label] = v
        print(f"{label:<14}{v if v else 'MISSING'}")
        if v is None:
            missing.append(label)
    CTX["versions"] = versions
    status = "PASS"
    notes = []
    tv = versions.get("transformers")
    if tv:
        try:
            from packaging.version import Version
            if Version(tv) < Version("4.37.0"):
                notes.append("transformers < 4.37.0: Qwen2.5 loading fails with KeyError: 'qwen2' (model card). Upgrade.")
                status = "FAIL"
        except Exception:  # noqa: BLE001
            notes.append("could not compare transformers version")
    if missing:
        notes.append(f"missing packages: {missing}")
        status = "FAIL"
    for n in notes:
        print("NOTE:", n)
    return status, {"versions": versions, "notes": notes}


def stage_gpu(allow_cpu):
    import torch
    d = {"torch_cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available()}
    print("torch CUDA build :", d["torch_cuda_build"])
    print("CUDA available   :", d["cuda_available"])
    if not torch.cuda.is_available():
        CTX["device"] = "cpu"
        print("No CUDA device. In Colab: Runtime > Change runtime type > T4 GPU, then restart and rerun.")
        return ("WARN" if allow_cpu else "FAIL"), d
    CTX["device"] = "cuda"
    props = torch.cuda.get_device_properties(0)
    d.update({"name": props.name, "total_memory_gb": round(props.total_memory / 1e9, 1),
              "capability": f"{props.major}.{props.minor}", "bf16_supported": bool(torch.cuda.is_bf16_supported())})
    print("GPU              :", d["name"])
    print("GPU memory (GB)  :", d["total_memory_gb"])
    print("Compute capab.   :", d["capability"])
    print("bf16 supported   :", d["bf16_supported"], "(protocol uses fp16)")
    a = torch.randn(2048, 2048, device="cuda", dtype=torch.float16)
    b = torch.randn(2048, 2048, device="cuda", dtype=torch.float16)
    torch.cuda.synchronize()
    t0 = time.time()
    c = a @ b
    torch.cuda.synchronize()
    d["fp16_matmul_seconds"] = round(time.time() - t0, 4)
    ok = bool(torch.isfinite(c).all().item())
    print("fp16 matmul 2048x2048 finite:", ok, f"({d['fp16_matmul_seconds']}s, includes warm-up)")
    return ("PASS" if ok else "FAIL"), d


def stage_seed():
    import random
    import numpy as np
    import torch
    from uncertainty.seed import set_seed

    def draw():
        vals = [random.random(), float(np.random.rand()), float(torch.rand(1).item())]
        if torch.cuda.is_available():
            vals.append(float(torch.rand(1, device="cuda").item()))
        return vals

    set_seed(42)
    first = draw()
    set_seed(42)
    second = draw()
    print("first :", first)
    print("second:", second)
    same = first == second
    print("identical after reseeding:", same)
    return ("PASS" if same else "FAIL"), {"identical": same}


def stage_config():
    import yaml
    cfg_path = ROOT / "configs" / "protocol_v1.yaml"
    cfg = yaml.safe_load(open(cfg_path))
    CTX["cfg"] = cfg
    print("protocol_version:", cfg["protocol_version"], "| status:", cfg["status"])
    print("model:", cfg["model"]["name"], "| dataset:", cfg["dataset"]["hf_name"], cfg["dataset"]["config"], cfg["dataset"]["split"])
    print("primary score:", cfg["scoring"]["primary_score"], "| samples:", cfg["generation"]["sampling"]["num_samples"])
    return "PASS", {"protocol_version": cfg["protocol_version"], "status": cfg["status"]}


def stage_dataset(check_test):
    from datasets import load_dataset
    cfg = CTX["cfg"]["dataset"]
    ds = load_dataset(cfg["hf_name"], cfg["config"], split=cfg["split"])
    CTX["ds"] = ds
    print("rows:", len(ds))
    print("columns:", ds.column_names)
    ex = ds[0]
    print("example question   :", ex["question"])
    print("example question_id:", ex["question_id"])
    print("answer.value       :", ex["answer"]["value"])
    print("n aliases          :", len(ex["answer"]["aliases"]), "| first normalized:", ex["answer"]["normalized_aliases"][:5])
    ids = ds["question_id"]
    n_unique = len(set(ids))
    n_dup = len(ids) - n_unique
    print(f"unique question_id: {n_unique} | duplicate rows: {n_dup}")
    qtexts = ds["question"]
    print("unique question strings:", len(set(qtexts)))
    empty = sum(1 for a in ds["answer"] if not a["normalized_aliases"])
    print("rows with no aliases:", empty)
    detail = {"rows": len(ds), "unique_question_id": n_unique, "duplicate_rows": n_dup,
              "unique_question_text": len(set(qtexts)), "rows_without_aliases": empty}
    if check_test:
        te = load_dataset(cfg["hf_name"], cfg["config"], split="test")
        no_ans = sum(1 for a in te.select(range(min(100, len(te))))["answer"] if not a["normalized_aliases"])
        print(f"test split: first 100 rows without aliases = {no_ans}")
        detail["test_first100_without_aliases"] = no_ans
    status = "WARN" if (n_dup > 0 or empty > 0) else "PASS"
    if n_dup > 0:
        print("NOTE: duplicates exist; deduplicate by question_id before sampling (protocol Section 4, step 2).")
    return status, detail


def load_model_and_tokenizer():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = CTX["cfg"]["model"]["name"]
    tok = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16)
    except Exception as exc:  # newer transformers may prefer `dtype`
        print("torch_dtype load failed (", type(exc).__name__, "), retrying with dtype=")
        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16)
    model.to(CTX["device"]).eval()
    return tok, model


def stage_model():
    import torch
    from uncertainty.prompting import render_chat_prompt
    tok, model = load_model_and_tokenizer()
    CTX["tok"], CTX["model"] = tok, model
    n_params = sum(p.numel() for p in model.parameters())
    print("parameters (M):", round(n_params / 1e6, 1))
    print("dtype:", next(model.parameters()).dtype, "| device:", next(model.parameters()).device)
    print("tokenizer length:", len(tok), "| model.config.vocab_size:", model.config.vocab_size)
    print("eos_token:", tok.eos_token, tok.eos_token_id, "| generation_config.eos_token_id:", model.generation_config.eos_token_id)
    gc = model.generation_config.to_dict()
    keys = ["do_sample", "temperature", "top_k", "top_p", "repetition_penalty", "max_new_tokens", "eos_token_id", "pad_token_id"]
    print("generation_config (selected):", {k: gc.get(k) for k in keys})
    rp = gc.get("repetition_penalty")
    notes = []
    if rp not in (None, 1.0):
        notes.append(f"default repetition_penalty={rp}: would distort generate-time scores; protocol sets 1.0 explicitly")
    rendered = render_chat_prompt(tok, "Who wrote the novel 'Pride and Prejudice'?")
    print("--- rendered prompt (this exact text, incl. any default system message, is part of the protocol) ---")
    print(rendered)
    print("--- end rendered prompt ---")
    if torch.cuda.is_available():
        print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
    for n in notes:
        print("NOTE:", n)
    detail = {"params_million": round(n_params / 1e6, 1), "generation_config": {k: gc.get(k) for k in keys}, "notes": notes}
    return ("WARN" if notes else "PASS"), detail


def eos_id_set(tok, model):
    ids = set()
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    e = model.generation_config.eos_token_id
    if isinstance(e, int):
        ids.add(e)
    elif isinstance(e, (list, tuple)):
        ids.update(int(x) for x in e)
    return ids


def stage_greedy(n_questions):
    import torch
    from uncertainty.prompting import prompt_input_ids
    from uncertainty.scoring import aggregate_scores, teacher_forced_token_scores
    from uncertainty.textnorm import is_correct

    tok, model, cfg = CTX["tok"], CTX["model"], CTX["cfg"]
    tol = cfg["scoring"]["consistency_check"]["tolerance"]
    max_new = cfg["generation"]["greedy"]["max_new_tokens"]
    eos = eos_id_set(tok, model)

    if "ds" in CTX:
        qs = [(CTX["ds"][i]["question"], CTX["ds"][i]["answer"]["normalized_aliases"], CTX["ds"][i]["answer"]["value"])
              for i in range(n_questions)]
    else:
        qs = [("Who wrote the novel 'Pride and Prejudice'?", ["jane austen", "austen"], "Jane Austen")]
        print("(dataset not loaded: using one fixed question)")

    max_diffs, rows, flags = [], [], []
    for qi, (q, aliases, gold) in enumerate(qs):
        pids = prompt_input_ids(tok, q).to(model.device)
        out = model.generate(input_ids=pids.unsqueeze(0), attention_mask=torch.ones_like(pids).unsqueeze(0),
                             do_sample=False, temperature=None, top_p=None, top_k=None,
                             repetition_penalty=1.0, max_new_tokens=max_new,
                             output_scores=True, return_dict_in_generate=True,
                             pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        gen = out.sequences[0, pids.shape[0]:].cpu()
        text = tok.decode(gen, skip_special_tokens=True)
        tf = teacher_forced_token_scores(model, pids.cpu(), gen)
        # generate-time log-probs from the returned scores
        if len(out.scores) != gen.shape[0]:
            flags.append(f"q{qi}: len(scores)={len(out.scores)} != n generated tokens={gen.shape[0]}")
            continue
        gen_lp = torch.stack([torch.log_softmax(s[0].float(), -1)[gen[t].to(s.device)].cpu() for t, s in enumerate(out.scores)])
        diff = (gen_lp - tf["logprob"]).abs()
        max_diffs.append(float(diff.max()))
        keep = [i for i in range(gen.shape[0]) if int(gen[i]) not in eos]
        agg = aggregate_scores(tf["logprob"][keep].tolist(), tf["entropy"][keep].tolist())
        argmax_rate = float((tf["argmax"] == gen).float().mean())
        correct = is_correct(text, aliases)
        rows.append((qi, text.strip()[:40], gold[:25], correct, gen.shape[0], round(agg["mean_nll"], 3), round(argmax_rate, 2), round(max_diffs[-1], 4)))
        if qi == 0:
            print("\nFirst question, per-token table (EOS tokens are scored here but excluded from aggregates):")
            print(f"{'token':<16}{'id':>8}{'logprob':>10}{'prob':>8}{'entropy':>9}  eos?")
            for t in range(gen.shape[0]):
                lp, h = float(tf["logprob"][t]), float(tf["entropy"][t])
                print(f"{tok.decode(gen[t])!r:<16}{int(gen[t]):>8}{lp:>10.3f}{math.exp(lp):>8.3f}{h:>9.3f}  {'EOS' if int(gen[t]) in eos else ''}")
                if lp > 1e-4 or h < -1e-4 or h > math.log(tf["vocab"]) + 1e-3:
                    flags.append(f"q0 token {t}: value outside valid range (logprob {lp}, entropy {h})")
            print("logit vector size (V):", tf["vocab"])
    print(f"\n{'q':<3}{'answer':<42}{'gold':<27}{'ok?':<6}{'nTok':<6}{'meanNLL':<9}{'argmax=tok':<12}{'max|diff|'}")
    for r in rows:
        print(f"{r[0]:<3}{r[1]!r:<42}{r[2]!r:<27}{str(r[3]):<6}{r[4]:<6}{r[5]:<9}{r[6]:<12}{r[7]}")
    worst = max(max_diffs) if max_diffs else float("nan")
    print(f"\nconsistency check: max |generate-time logprob - teacher-forced logprob| = {worst:.4f} (tolerance {tol}, provisional)")
    for f in flags:
        print("FLAG:", f)
    status = "PASS"
    if flags or not max_diffs or worst > tol:
        status = "WARN"
    return status, {"max_abs_logprob_diff": worst, "tolerance": tol, "flags": flags, "rows": [list(map(str, r)) for r in rows]}


def stage_sampling():
    import torch
    from uncertainty.prompting import prompt_input_ids
    from uncertainty.seed import seed_for_question, set_seed
    from uncertainty.textnorm import extract_short_answer, normalize_answer

    tok, model, cfg = CTX["tok"], CTX["model"], CTX["cfg"]
    sc = cfg["generation"]["sampling"]
    q = CTX["ds"][0]["question"] if "ds" in CTX else "Who wrote the novel 'Pride and Prejudice'?"
    pids = prompt_input_ids(tok, q).to(model.device)

    def sample():
        set_seed(seed_for_question(sc["seed_base"], 0))
        t0 = time.time()
        out = model.generate(input_ids=pids.unsqueeze(0), attention_mask=torch.ones_like(pids).unsqueeze(0),
                             do_sample=True, temperature=sc["temperature"], top_k=sc["top_k"], top_p=sc["top_p"],
                             repetition_penalty=sc["repetition_penalty"], max_new_tokens=sc["max_new_tokens"],
                             num_return_sequences=sc["num_samples"],
                             pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = time.time() - t0
        texts = [tok.decode(o[pids.shape[0]:], skip_special_tokens=True) for o in out]
        return texts, dt

    a, t_a = sample()
    b, t_b = sample()
    print("question:", q)
    for i, t in enumerate(a):
        print(f"  sample {i + 1:>2}: {extract_short_answer(t)!r}")
    uniq = len({normalize_answer(extract_short_answer(t)) for t in a})
    print("distinct normalized answers among 10:", uniq)
    same = a == b
    print("same seed -> identical samples:", same)
    print(f"seconds for 10 samples (one question, rough): {t_a:.2f}, {t_b:.2f}")
    status = "PASS" if same else "WARN"
    if not same:
        print("NOTE: GPU sampling may be non-deterministic across calls; log this and rely on stored samples.")
    return status, {"distinct_answers": uniq, "reproducible": same, "seconds_10_samples": round(t_a, 2)}


def stage_nli():
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    name = CTX["cfg"]["semantic_entropy"]["nli_model"]
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name).to(CTX["device"]).eval()
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    print("id2label (read from the model, do not assume):", id2label)
    print("tokenizer.model_max_length:", tok.model_max_length, "(we set max_length explicitly in calls)")
    tests = [
        ("The cat sat on the mat.", "A cat is on the mat.", "entailment"),
        ("The cat sat on the mat.", "The cat is not on the mat.", "contradiction"),
        ("The cat sat on the mat.", "The cat is hungry.", "neutral"),
    ]
    ok = 0
    for prem, hyp, expected in tests:
        enc = tok(prem, hyp, return_tensors="pt", truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits[0].float(), -1).cpu()
        pred = id2label[int(probs.argmax())].lower()
        ok += int(pred == expected)
        print(f"premise={prem!r} hypothesis={hyp!r}\n  expected={expected} predicted={pred} probs={[round(float(p), 3) for p in probs]}")
    CTX["nli_ok"] = ok
    return ("PASS" if ok == len(tests) else "WARN"), {"id2label": id2label, "hand_pairs_as_expected": f"{ok}/{len(tests)}"}


def stage_report():
    out_dir = ROOT / "results"
    out_dir.mkdir(exist_ok=True)
    versions = CTX.get("versions", {})
    with open(out_dir / "env_report.json", "w") as f:
        json.dump({"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "stages": RESULTS}, f, indent=2, default=str)
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    (out_dir / "pip_freeze.txt").write_text(freeze)
    lines = [f"{k}=={v}" for k, v in versions.items() if v]
    (out_dir / "requirements_candidate.txt").write_text(
        "# Versions seen in THIS run. Commit as requirements.txt ONLY if all stages passed.\n" + "\n".join(lines) + "\n")
    print("wrote results/env_report.json, results/pip_freeze.txt, results/requirements_candidate.txt")
    return "PASS", {}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--skip-dataset", action="store_true")
    ap.add_argument("--skip-nli", action="store_true")
    ap.add_argument("--allow-cpu", action="store_true", help="continue without a GPU (slow; for debugging only)")
    ap.add_argument("--check-test", action="store_true", help="also confirm the TriviaQA test split has no answers")
    ap.add_argument("--n-questions", type=int, default=5)
    args = ap.parse_args()

    (ROOT / "logs").mkdir(exist_ok=True)
    log_f = open(ROOT / "logs" / "env_test_output.txt", "w")
    sys.stdout = Tee(sys.__stdout__, log_f)
    print("ENVIRONMENT TEST  | started", time.strftime("%Y-%m-%d %H:%M:%S"), "| repo:", ROOT)

    run_stage("1 system", stage_system)
    s = run_stage("2 libraries", stage_libraries)
    if s == "FAIL" and any(v is None for k, v in CTX.get("versions", {}).items() if k in ("torch", "transformers")):
        print("\nCore libraries missing: cannot continue. Install them (see README) and rerun.")
    else:
        g = run_stage("3 gpu", stage_gpu, args.allow_cpu)
        run_stage("4 seed", stage_seed)
        c = run_stage("5 config", stage_config)
        if c == "PASS" and not args.skip_dataset:
            run_stage("6 dataset", stage_dataset, args.check_test)
        if c == "PASS" and (g != "FAIL"):
            m = run_stage("7 model", stage_model)
            if m != "FAIL":
                run_stage("8 greedy+scoring+consistency", stage_greedy, args.n_questions)
                run_stage("9 sampling", stage_sampling)
            if not args.skip_nli:
                run_stage("10 nli", stage_nli)
    run_stage("11 report", stage_report)

    print("\n================ SUMMARY ================")
    for r in RESULTS:
        print(f"{r['stage']:<32}{r['status']:<6}{r['seconds']:>8.1f}s")
    n = {k: sum(1 for r in RESULTS if r["status"] == k) for k in ("PASS", "WARN", "FAIL")}
    print(f"\nENV TEST SUMMARY: {n['PASS']} PASS, {n['WARN']} WARN, {n['FAIL']} FAIL")
    print("Next step: paste this whole output into environment_verification_log.md. Do not edit it.")
    sys.stdout = sys.__stdout__
    log_f.close()
    sys.exit(1 if n["FAIL"] else 0)


if __name__ == "__main__":
    main()


Writing scripts/00_env_test.py


In [ ]:
%%writefile scripts/01_build_subset.py
#!/usr/bin/env python
"""Cycle 1: load TriviaQA validation, deduplicate, validate, sample the pool, split, save ids only.

Run from the repo root:  python scripts/01_build_subset.py
Options: --pool-size N   (ONLY for the pre-declared 400 fallback; the reason must be logged in protocol_changes.md)
         --no-write      (print everything but do not write files)
Outputs: data/subset_question_ids.csv, data/subset_manifest.json, logs/01_build_subset_output.txt
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from uncertainty import data as D  # noqa: E402


class Tee:
    def __init__(self, *s):
        self.s = s

    def write(self, x):
        for f in self.s:
            f.write(x)
            f.flush()

    def flush(self):
        for f in self.s:
            f.flush()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pool-size", type=int, default=None)
    ap.add_argument("--no-write", action="store_true")
    args = ap.parse_args()

    (ROOT / "logs").mkdir(exist_ok=True)
    (ROOT / "data").mkdir(exist_ok=True)
    log = open(ROOT / "logs" / "01_build_subset_output.txt", "w")
    sys.stdout = Tee(sys.__stdout__, log)

    cfg = yaml.safe_load(open(ROOT / "configs" / "protocol_v1.yaml"))
    ds_cfg = cfg["dataset"]
    print("CYCLE 1 | build subset |", time.strftime("%Y-%m-%d %H:%M:%S"))
    print("protocol:", cfg["protocol_version"], "| status:", cfg["status"])
    if args.pool_size:
        print(f"WARNING: pool size overridden to {args.pool_size} (allowed only for the declared fallback; log it).")

    from datasets import load_dataset
    ds = load_dataset(ds_cfg["hf_name"], ds_cfg["config"], split=ds_cfg["split"])
    print(f"loaded {ds_cfg['hf_name']} / {ds_cfg['config']} / {ds_cfg['split']}: {len(ds)} rows")
    df = D.rows_from_records(ds)
    print("columns:", list(df.columns))
    print("question_source values:", df["question_source"].value_counts().head(5).to_dict())
    print("problems BEFORE cleaning:", D.validate(df))

    sub, rep = D.build_subset(df, ds_cfg, args.pool_size)
    print("\n--- cleaning ---")
    for k, v in rep["dedupe"].items():
        print(f"{k:<42}{v}")
    print("problems AFTER dedupe:", rep["validation_after_dedupe"])
    print("rows dropped for having no alias:", rep["dropped_unlabellable"])
    print("questions available after cleaning:", rep["available_after_cleaning"])

    print("\n--- subset ---")
    print(f"pool size {rep['pool_size']} | calibration {rep['n_calibration']} | test {rep['n_test']} | pilot {rep['n_pilot']}")
    assert rep["n_calibration"] + rep["n_test"] == rep["pool_size"]
    pilot_splits = set(sub.loc[sub["is_pilot"], "split"])
    print("pilot rows belong to split(s):", pilot_splits)
    assert pilot_splits <= {"calibration"}, "pilot must come from calibration only"
    ids = D.ids_table(sub)
    assert ids["question_id"].is_unique
    print("first 3 pool rows:")
    for r in sub.head(3).itertuples():
        print(f"  #{r.pool_index} id={r.question_id} split={r.split} pilot={r.is_pilot} | {r.question[:70]!r} -> {r.gold_value!r}")
    print("alias count per question (min/median/max):",
          int(sub['normalized_aliases'].map(len).min()), int(sub['normalized_aliases'].map(len).median()), int(sub['normalized_aliases'].map(len).max()))

    # determinism self-check: shuffle the input rows and rebuild; the id table must be identical
    shuffled = df.sample(frac=1.0, random_state=123).reset_index(drop=True)
    sub2, rep2 = D.build_subset(shuffled, ds_cfg, args.pool_size)
    same = rep2["ids_sha256"] == rep["ids_sha256"]
    print("\ndeterminism self-check (shuffled input gives the same ids):", same)
    print("ids sha256:", rep["ids_sha256"])
    assert same, "subset depends on row order: bug"

    if not args.no_write:
        extra = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "protocol_version": cfg["protocol_version"],
                 "protocol_status": cfg["status"], "numpy": np.__version__, "pandas": pd.__version__,
                 "rows_loaded": len(df)}
        D.save_subset(sub, rep, ROOT / "data" / "subset_question_ids.csv", ROOT / "data" / "subset_manifest.json", extra)
        print("wrote data/subset_question_ids.csv (ids only) and data/subset_manifest.json")
    print("\nCYCLE 1 DONE. Paste this whole output into code_understanding.md (Cycle 1 section).")
    sys.stdout = sys.__stdout__
    log.close()


if __name__ == "__main__":
    main()


Writing scripts/01_build_subset.py


In [ ]:
%%writefile scripts/02_generate.py
#!/usr/bin/env python
"""Cycle 2: greedy answer + teacher-forced token scores + 10 samples, per question. Resumable. [TEST REQUIRED]

  python scripts/02_generate.py --split pilot            # first: the 50-question pilot (calibration part only)
  python scripts/02_generate.py --split calibration      # rest of the calibration part
  python scripts/02_generate.py --split test --allow-test  # only after the protocol is frozen
Options: --limit N (debug), --allow-cpu (very slow, debugging only)
Output: results/raw/generations.jsonl (local only: contains question text and gold answers; git-ignored)."""
import argparse
import sys
import time

from pathlib import Path

from _common import GEN, RAW, ROOT, git_commit, load_cfg, load_subset_with_text, require_test_permission, select_rows

from uncertainty.generation import generate_record, load_model_and_tokenizer
from uncertainty.io_utils import append_jsonl, done_ids, read_jsonl, repair_jsonl


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", choices=["pilot", "calibration", "test", "all"], required=True)
    ap.add_argument("--allow-test", action="store_true")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--allow-cpu", action="store_true")
    ap.add_argument("--out", default=None, help="alternative output file (e.g. for the reproducibility rerun)")
    args = ap.parse_args()
    out = Path(args.out) if args.out else GEN
    require_test_permission(args.split, args.allow_test)

    import torch
    cfg = load_cfg()
    device = "cuda" if torch.cuda.is_available() else None
    if device is None:
        if not args.allow_cpu:
            raise SystemExit("No GPU. In Colab: Runtime > Change runtime type > T4 GPU (or use --allow-cpu for debugging).")
        device = "cpu"
    RAW.mkdir(parents=True, exist_ok=True)
    n_valid = repair_jsonl(out)
    sub = select_rows(load_subset_with_text(cfg), args.split)
    done = done_ids(out)
    todo = [r for r in sub.itertuples() if r.question_id not in done]
    if args.limit:
        todo = todo[: args.limit]
    print(f"split={args.split} | selected {len(sub)} | already done {len(sub) - len([r for r in sub.itertuples() if r.question_id not in done])}"
          f" | to run now {len(todo)} | protocol {cfg['protocol_version']} ({cfg['status']})")
    tok, model = load_model_and_tokenizer(cfg["model"]["name"], device)
    import transformers
    append_jsonl(RAW / "meta_runs.jsonl", {"stage": "generate", "time": time.strftime("%Y-%m-%d %H:%M:%S"), "split": args.split,
                                           "n_todo": len(todo), "git_commit": git_commit(), "torch": torch.__version__,
                                           "transformers": transformers.__version__,
                                           "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
                                           "protocol_status": cfg["status"]})
    t0 = time.time()
    for k, r in enumerate(todo, 1):
        rec = generate_record(model, tok, r.question, cfg, int(r.pool_index))
        rec.update({"question_id": r.question_id, "pool_index": int(r.pool_index), "split": r.split, "is_pilot": bool(r.is_pilot),
                    "question": r.question, "gold_value": r.gold_value, "normalized_aliases": list(r.normalized_aliases)})
        append_jsonl(out, rec)
        if k % 10 == 0 or k == len(todo):
            el = time.time() - t0
            print(f"  {k}/{len(todo)} | {el / k:.2f} s/question | ETA {(len(todo) - k) * el / k / 60:.1f} min")
    recs = read_jsonl(out)
    if recs:
        per_q = sum(r["seconds_greedy"] + r["seconds_samples"] for r in recs) / len(recs)
        print(f"\nmean generation seconds per question over {len(recs)} stored questions: {per_q:.2f}")
        print(f"projected generation time for the pool of {cfg['dataset']['pool_size']}: {per_q * cfg['dataset']['pool_size'] / 3600:.2f} h "
              f"(NLI time comes on top; cap before using the 400 fallback: {cfg['compute']['cap_gpu_hours_before_fallback']} GPU-hours)")
    print("GENERATION DONE (valid lines before this run:", n_valid, ")")


if __name__ == "__main__":
    main()


Writing scripts/02_generate.py


In [ ]:
%%writefile scripts/03_semantic_entropy.py
#!/usr/bin/env python
"""Cycle 5: cluster the 10 samples with DeBERTa-large-MNLI and store the cluster ids. Resumable. [TEST REQUIRED]

  python scripts/03_semantic_entropy.py --split pilot
Output: results/raw/semantic.jsonl (cluster ids, NLI call counts, seconds)."""
import argparse
import time

from _common import GEN, RAW, SEM, load_cfg, require_test_permission

from uncertainty.io_utils import append_jsonl, done_ids, read_jsonl, repair_jsonl
from uncertainty.semantic_entropy import NLIEntailment, cluster_samples, semantic_entropy_from_ids


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", choices=["pilot", "calibration", "test", "all"], required=True)
    ap.add_argument("--allow-test", action="store_true")
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()
    require_test_permission(args.split, args.allow_test)
    import torch
    cfg = load_cfg()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    repair_jsonl(SEM)
    gens = read_jsonl(GEN)
    if args.split == "pilot":
        gens = [g for g in gens if g["is_pilot"]]
    elif args.split in ("calibration", "test"):
        gens = [g for g in gens if g["split"] == args.split]
    done = done_ids(SEM)
    todo = sorted([g for g in gens if g["question_id"] not in done], key=lambda g: g["pool_index"])
    if args.limit:
        todo = todo[: args.limit]
    print(f"split={args.split} | generations available {len(gens)} | to cluster now {len(todo)} | device {device}")
    nli = NLIEntailment(cfg["semantic_entropy"]["nli_model"], device)
    print("NLI label map:", nli.id2label, "| entailment index:", nli.entail_idx)
    t0 = time.time()
    for k, g in enumerate(todo, 1):
        t1 = time.time()
        ids, calls = cluster_samples(g["question"], g["samples"], nli)
        if device == "cuda":
            torch.cuda.synchronize()
        append_jsonl(SEM, {"question_id": g["question_id"], "cluster_ids": ids, "nli_calls": calls,
                           "seconds_nli": round(time.time() - t1, 4)})
        if k % 10 == 0 or k == len(todo):
            el = time.time() - t0
            print(f"  {k}/{len(todo)} | {el / k:.2f} s/question | ETA {(len(todo) - k) * el / k / 60:.1f} min")
    recs = [r for r in read_jsonl(SEM)]
    if recs and args.split != "test":
        import numpy as np
        nc = np.array([len(set(r["cluster_ids"])) for r in recs])
        print("\nclusters per question (distribution): ", {int(k): int((nc == k).sum()) for k in np.unique(nc)})
        print("questions where all 10 samples form ONE cluster:", int((nc == 1).sum()), "of", len(nc),
              "(instruct models may show low diversity; report this)")
        print("mean NLI directional calls per question:", round(float(np.mean([r['nli_calls'] for r in recs])), 2))
    print("SEMANTIC ENTROPY DONE")


if __name__ == "__main__":
    main()


Writing scripts/03_semantic_entropy.py


In [ ]:
%%writefile scripts/04_build_table.py
#!/usr/bin/env python
"""Cycle 4: merge generations + semantic entropy into results/per_question.csv (no question text, no gold answers).
Prints COUNTS only (class counts are allowed before the protocol tag; scores of test rows are never printed)."""
import numpy as np

from _common import GEN, ROOT, SEM, load_cfg

from uncertainty.io_utils import read_jsonl
from uncertainty.table import build_table


def main():
    gens, sems = read_jsonl(GEN), read_jsonl(SEM)
    if not gens:
        raise SystemExit("results/raw/generations.jsonl is empty: run scripts/02_generate.py first")
    df = build_table(gens, sems)
    out = ROOT / "results" / "per_question.csv"
    df.to_csv(out, index=False)
    print(f"wrote {out} | rows {len(df)} | with semantic entropy {int(df['SE10'].notna().sum())}")
    print("rows per split:", df["split"].value_counts().to_dict(), "| pilot rows:", int(df["is_pilot"].sum()))
    for sp in ("calibration", "test"):
        d = df[df["split"] == sp]
        if len(d):
            print(f"{sp}: correct {int((d['y_wrong_exact'] == 0).sum())} | wrong {int((d['y_wrong_exact'] == 1).sum())} (exact-match label)")
    nt = df[df["split"] != "test"]            # diagnostics on non-test rows only
    print("non-test diagnostics -> truncated:", int(nt["truncated"].sum()), "| span fallback:", int(nt["span_fallback"].sum()),
          "| zero-token answers:", int((nt["n_tokens"] == 0).sum()),
          "| mean argmax=token rate (greedy, expect about 1.0):", round(float(np.nanmean(nt["argmax_match_rate"])), 4) if len(nt) else "n/a")


if __name__ == "__main__":
    main()


Writing scripts/04_build_table.py


In [ ]:
%%writefile scripts/05_handcheck.py
#!/usr/bin/env python
"""Hand-check sheets. [TEST REQUIRED]
  python scripts/05_handcheck.py --make-labelcheck           -> results/raw/handcheck_sheet.csv (30 pilot answers; NO scores shown)
  (fill the column human_correct with yes/no by hand, WITHOUT looking at any uncertainty score)
  python scripts/05_handcheck.py --score-labelcheck          -> agreement with the automatic label (target >= 0.90)
  python scripts/05_handcheck.py --make-erroranalysis --frozen        (after the protocol tag; uses TEST rows)
  python scripts/05_handcheck.py --summarize-erroranalysis --frozen   (after filling the `category` column)"""
import argparse
import csv
import json

import pandas as pd

from _common import GEN, RAW, ROOT, SEM, git_tag_exists, load_cfg

from uncertainty.handcheck import (ERROR_CATEGORIES, agreement, parse_human_label, select_error_analysis,
                                   select_for_labelcheck)
from uncertainty.io_utils import read_jsonl
from uncertainty.textnorm import is_correct


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--make-labelcheck", action="store_true")
    ap.add_argument("--score-labelcheck", action="store_true")
    ap.add_argument("--make-erroranalysis", action="store_true")
    ap.add_argument("--summarize-erroranalysis", action="store_true")
    ap.add_argument("--show-clusters", type=int, default=0, help="print the samples and cluster ids of N pilot questions for manual validation")
    ap.add_argument("--frozen", action="store_true")
    ap.add_argument("--skip-tag-check", action="store_true")
    a = ap.parse_args()
    cfg = load_cfg()
    hc = cfg["label"]["hand_check"]
    sheet = RAW / "handcheck_sheet.csv"
    gens = {g["question_id"]: g for g in read_jsonl(GEN)}

    if a.show_clusters:
        sems = {r["question_id"]: r for r in read_jsonl(SEM)}
        shown = 0
        for g in sorted([g for g in gens.values() if g["is_pilot"]], key=lambda g: g["pool_index"]):
            if g["question_id"] not in sems or shown >= a.show_clusters:
                continue
            ids = sems[g["question_id"]]["cluster_ids"]
            print(f"\nQ: {g['question']}\n   gold: {g['gold_value']} | greedy: {g['short_answer']!r} | clusters: {len(set(ids))}")
            for c in sorted(set(ids)):
                print(f"   cluster {c}: " + " ; ".join(repr(s.strip()[:40]) for s, i in zip(g["samples"], ids) if i == c))
            shown += 1
        print("\nFor each question note: wrong MERGE (different meanings in one cluster) or wrong SPLIT (same meaning in two clusters).")

    if a.make_labelcheck:
        pilot = [g for g in gens.values() if g["is_pilot"]]
        chosen = select_for_labelcheck(pilot, hc["n"], hc["seed"])
        if len(chosen) < hc["n"]:
            print(f"WARNING: only {len(chosen)} pilot answers available")
        with open(sheet, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["question_id", "question", "model_answer", "gold_value", "gold_aliases", "human_correct"])
            for g in chosen:
                w.writerow([g["question_id"], g["question"], g["short_answer"], g["gold_value"],
                            " | ".join(g["normalized_aliases"][:8]), ""])
        print(f"wrote {sheet} ({len(chosen)} rows). Fill human_correct with yes/no. Do NOT look at uncertainty scores while doing it.")

    if a.score_labelcheck:
        df = pd.read_csv(sheet, dtype=str).fillna("")
        auto = [is_correct(gens[q]["greedy_text"], gens[q]["normalized_aliases"]) for q in df["question_id"]]
        hum = [parse_human_label(x) for x in df["human_correct"]]
        res = agreement(auto, hum)
        res["target"] = hc["min_agreement"]
        res["meets_target"] = bool(res["agreement"] >= hc["min_agreement"]) if res["n_compared"] else False
        print(json.dumps(res, indent=2))
        for q, au, hu in zip(df["question_id"], auto, hum):
            if hu is not None and au != hu:
                print("DISAGREEMENT:", q, "| answer:", gens[q]["short_answer"], "| gold:", gens[q]["gold_value"], "| auto_correct:", au, "| human_correct:", hu)
        out = ROOT / "results" / "pilot"
        out.mkdir(parents=True, exist_ok=True)
        json.dump(res, open(out / "handcheck_agreement.json", "w"), indent=2)

    if a.make_erroranalysis or a.summarize_erroranalysis:
        if not a.frozen or not (a.skip_tag_check or git_tag_exists()):
            raise SystemExit("Error analysis uses TEST rows: needs --frozen and the git tag protocol-v1 (or --skip-tag-check).")
        path = RAW / "error_analysis_sheet.csv"
        if a.make_erroranalysis:
            t = pd.read_csv(ROOT / "results" / "per_question.csv")
            t = t[(t["split"] == "test") & t["SE10"].notna()]
            wrong, right = select_error_analysis(t.to_dict("records"), 30)
            with open(path, "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["group", "question_id", "question", "model_answer", "gold_value", "n_clusters", "category"])
                for grp, rows in (("confident_error", wrong), ("uncertain_correct", right)):
                    for r in rows:
                        g = gens[r["question_id"]]
                        w.writerow([grp, r["question_id"], g["question"], g["short_answer"], g["gold_value"], r["n_clusters"], ""])
            print(f"wrote {path}. Fill `category` with one of: {ERROR_CATEGORIES}")
        else:
            df = pd.read_csv(path, dtype=str).fillna("")
            bad = df[~df["category"].isin(ERROR_CATEGORIES)]
            if len(bad):
                raise SystemExit(f"{len(bad)} rows have a missing/invalid category")
            out = ROOT / "results" / "final"
            out.mkdir(parents=True, exist_ok=True)
            df.groupby(["group", "category"]).size().reset_index(name="count").to_csv(out / "table5.csv", index=False)
            print(df.groupby(["group", "category"]).size())


if __name__ == "__main__":
    main()


Writing scripts/05_handcheck.py


In [ ]:
%%writefile scripts/06_evaluate.py
#!/usr/bin/env python
"""Cycle 6/7: audit, then evaluation per experiment_protocol.md. [TEST REQUIRED on real data]
  python scripts/06_evaluate.py --mode pilot                       # pilot rows only; writes results/pilot/
  python scripts/06_evaluate.py --mode final --frozen              # test rows; needs git tag protocol-v1; writes results/final/
The pre-evaluation audit runs FIRST and the evaluation refuses to continue on any ERROR (--skip-audit overrides and is recorded).
Options: --n-boot N, --ignore-balance, --skip-tag-check (logged), --skip-audit (logged), --no-figs"""
import argparse
import json

import pandas as pd

from _common import ROOT, audit_from_disk, code_digest, freeze_record, git_commit, git_tag_exists, load_cfg, sha256_file

from uncertainty.audit import format_report
from uncertainty.evaluation import run_analysis


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["pilot", "final"], required=True)
    ap.add_argument("--frozen", action="store_true")
    ap.add_argument("--skip-tag-check", action="store_true")
    ap.add_argument("--skip-audit", action="store_true")
    ap.add_argument("--ignore-balance", action="store_true")
    ap.add_argument("--n-boot", type=int, default=None)
    ap.add_argument("--no-figs", action="store_true")
    a = ap.parse_args()
    cfg = load_cfg()
    notes = []
    if a.mode == "final":
        if not a.frozen:
            raise SystemExit("final mode reads the TEST rows: pass --frozen only after experiment_protocol.md Section 14 is complete.")
        if not git_tag_exists("protocol-v1"):
            if not a.skip_tag_check:
                raise SystemExit("git tag protocol-v1 not found. Freeze first: python scripts/11_freeze_protocol.py --freeze")
            notes.append("WARNING: evaluated WITHOUT the protocol-v1 git tag (--skip-tag-check)")
    out = ROOT / "results" / ("pilot" if a.mode == "pilot" else "final")
    out.mkdir(parents=True, exist_ok=True)

    rep = audit_from_disk(a.mode, cfg, strict_freeze=not a.skip_tag_check)
    print(format_report(rep))
    json.dump(rep, open(out / "audit_report.json", "w"), indent=2, default=str)
    if not rep["passed"]:
        if not a.skip_audit:
            raise SystemExit("\nAudit FAILED: no metric was computed. Fix the ERRORs above (or send the audit output for debugging).")
        notes.append("WARNING: evaluated although the audit FAILED (--skip-audit)")

    pq = ROOT / "results" / "per_question.csv"
    rec = freeze_record()
    prov = {"input_file": "results/per_question.csv", "input_sha256": sha256_file(pq), "git_commit": git_commit(),
            "protocol_tag": "protocol-v1" if git_tag_exists("protocol-v1") else None, "freeze_sha256": rec["sha256"] if rec else None,
            "code_sha256": code_digest(), "audit_passed": rep["passed"], "audit_overridden": bool(a.skip_audit and not rep["passed"])}
    s = run_analysis(pd.read_csv(pq), cfg, a.mode, out, n_boot=a.n_boot, make_figs=not a.no_figs, ignore_balance=a.ignore_balance, provenance=prov)
    s["flags"] = notes + s["flags"]
    json.dump(s, open(out / "summary.json", "w"), indent=2, default=str)
    print(f"\nmode={a.mode} | n_eval={s['n_eval']} (wrong {s['n_wrong']}, correct {s['n_correct']}) | bootstrap {s['bootstrap']}")
    print(pd.read_csv(out / "table2.csv")[["score", "auroc", "auroc_lo", "auroc_hi", "auprc", "ece"]].round(3).to_string(index=False))
    print("RQ1:", json.dumps(s["rq1"], default=str))
    if s["rq3"]:
        print("RQ3:", json.dumps(s["rq3"], default=str))
    for f in s["flags"]:
        print("FLAG:", f)
    print(f"outputs in {out} (results.csv carries the input hash, commit and freeze hash on every row)")
    if a.mode == "pilot":
        print("PILOT = exploratory sanity check on calibration-part questions. Do not report pilot numbers as results.")


if __name__ == "__main__":
    main()


Writing scripts/06_evaluate.py


In [ ]:
%%writefile scripts/07_make_latex_tables.py
#!/usr/bin/env python
"""Write LaTeX tables from results/final/*.csv into paper/tables/ (no number is typed by hand)."""
from _common import ROOT

from uncertainty.latex_tables import make_tables

if __name__ == "__main__":
    if not (ROOT / "results" / "final" / "table2.csv").exists():
        raise SystemExit("results/final/table2.csv not found: run scripts/06_evaluate.py --mode final --frozen first.")
    made = make_tables(ROOT / "results" / "final", ROOT / "paper" / "tables")
    print("wrote:", made, "-> in main.tex replace the placeholder tables by \\input{tables/<name>.tex}; figures are in results/final/figs/")


Writing scripts/07_make_latex_tables.py


In [ ]:
%%writefile scripts/08_sampling_check.py
#!/usr/bin/env python
"""Statistical check that sampling is NOT silently truncated (top-k/top-p from the model's default generation config).
For a few questions: draw many first-token samples with the protocol settings and compare the observed frequency of the most
likely token with its model probability (z-score), and the observed mass outside the top-20 tokens with the model's tail mass.
Needs a GPU. [TEST REQUIRED]   Usage: python scripts/08_sampling_check.py [--n 2000]"""
import argparse
import math

import torch
from _common import load_cfg

from uncertainty.generation import load_model_and_tokenizer
from uncertainty.prompting import prompt_input_ids
from uncertainty.seed import set_seed

QUESTIONS = ["Who wrote the novel 'Pride and Prejudice'?", "What is the capital city of Australia?",
             "Which element has the chemical symbol Fe?", "In which year did the Titanic sink?"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=2000)
    n = ap.parse_args().n
    cfg = load_cfg()
    s = cfg["generation"]["sampling"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load_model_and_tokenizer(cfg["model"]["name"], device)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    print(f"{'question':<46}{'p(top1)':>9}{'freq':>8}{'z':>7}{'tail mass':>11}{'observed':>10}")
    worst = 0.0
    for q in QUESTIONS:
        pids = prompt_input_ids(tok, q).to(model.device)
        with torch.no_grad():
            p = torch.softmax(model(pids.unsqueeze(0)).logits[0, -1].float(), -1)
        top1 = int(p.argmax())
        p1 = float(p[top1])
        tail = float(torch.sort(p, descending=True).values[20:].sum())          # mass outside the 20 most likely tokens
        top20 = set(torch.topk(p, 20).indices.tolist())
        set_seed(12345)
        with torch.no_grad():
            out = model.generate(input_ids=pids.unsqueeze(0).repeat(n, 1), attention_mask=torch.ones(n, pids.shape[0], device=model.device, dtype=torch.long),
                                 do_sample=True, temperature=s["temperature"], top_k=s["top_k"], top_p=s["top_p"],
                                 repetition_penalty=s["repetition_penalty"], max_new_tokens=1, pad_token_id=pad)
        first = out[:, pids.shape[0]].tolist()
        freq = sum(t == top1 for t in first) / n
        obs_tail = sum(t not in top20 for t in first) / n
        z = (freq - p1) / math.sqrt(max(p1 * (1 - p1), 1e-12) / n)
        worst = max(worst, abs(z))
        print(f"{q[:44]:<46}{p1:>9.3f}{freq:>8.3f}{z:>7.2f}{tail:>11.4f}{obs_tail:>10.4f}")
    print(f"\nlargest |z| = {worst:.2f}  -> {'PASS (consistent with untruncated sampling)' if worst < 4 else 'FAIL: sampling looks truncated or temperature differs; do NOT continue'}")
    print("If tail mass is clearly > 0 but observed is 0.0000, top-k/top-p from the default generation config is still active.")


if __name__ == "__main__":
    main()


Writing scripts/08_sampling_check.py


In [ ]:
%%writefile scripts/09_repro_check.py
#!/usr/bin/env python
"""Compare two generation runs on the same questions (greedy answers, token log-probs, samples).
Usage: python scripts/02_generate.py --split pilot --limit 5 --out results/raw/generations_rerun.jsonl
       python scripts/09_repro_check.py results/raw/generations.jsonl results/raw/generations_rerun.jsonl"""
import json
import sys

from _common import ROOT

from uncertainty.io_utils import read_jsonl
from uncertainty.repro import compare_generation_runs

if __name__ == "__main__":
    if len(sys.argv) != 3:
        raise SystemExit(__doc__)
    res = compare_generation_runs(read_jsonl(sys.argv[1]), read_jsonl(sys.argv[2]))
    print(json.dumps(res, indent=2))
    print("Greedy answers must be identical; log-prob difference within tolerance; identical samples are expected with the same seed "
          "but GPU non-determinism may break this (report it; stored samples are never regenerated).")


Writing scripts/09_repro_check.py


In [ ]:
%%writefile scripts/10_extension_length_adjusted.py
#!/usr/bin/env python
"""Pre-registered secondary extension E1 (length-adjusted gate score). Uses TEST rows, so it has the same guards as the final evaluation.
  python scripts/10_extension_length_adjusted.py --frozen     [TEST REQUIRED]
Runs only if the length confound is present by the frozen rule; otherwise it records 'not run' and why."""
import argparse
import json

import pandas as pd

from _common import ROOT, git_tag_exists, load_cfg

from uncertainty.extensions import run_length_adjusted_extension

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--frozen", action="store_true")
    ap.add_argument("--skip-tag-check", action="store_true")
    ap.add_argument("--n-boot", type=int, default=None)
    a = ap.parse_args()
    if not a.frozen:
        raise SystemExit("E1 reads the TEST rows: pass --frozen only after the protocol is frozen and tagged.")
    if not git_tag_exists("protocol-v1") and not a.skip_tag_check:
        raise SystemExit("git tag protocol-v1 not found.")
    res = run_length_adjusted_extension(pd.read_csv(ROOT / "results" / "per_question.csv"), load_cfg(), ROOT / "results" / "final", a.n_boot)
    print(json.dumps(res, indent=2, default=float))


Writing scripts/10_extension_length_adjusted.py


In [ ]:
%%writefile scripts/11_freeze_protocol.py
#!/usr/bin/env python
"""Freeze the protocol and make the freeze survive Colab session resets.

  python scripts/11_freeze_protocol.py --freeze     records a SHA-256 of config + code, commits them to a local git repo, tags protocol-v1
  python scripts/11_freeze_protocol.py --restore    new session: recreates the tag ONLY IF the code is byte-identical to what was frozen
The record is stored in data/protocol_freeze.json (on Drive in the notebook). Any code change after freezing is refused on purpose:
it would be an undisclosed protocol change. Make a new version (protocol-v1.1) and log it in protocol_changes.md instead."""
import argparse
import json
import subprocess
import time

from _common import ROOT, code_digest, code_files, git_tag_exists

REC = ROOT / "data" / "protocol_freeze.json"


files = code_files
digest = code_digest


def git(*args):
    return subprocess.run(["git", "-C", str(ROOT), "-c", "user.name=protocol-freeze", "-c", "user.email=freeze@localhost", *args],
                          capture_output=True, text=True)


def make_tag():
    if not (ROOT / ".git").exists():
        r = git("init")
        if r.returncode:
            raise SystemExit("git init failed: " + r.stderr)
    (ROOT / ".gitignore").write_text("__pycache__/\n*.pyc\n")
    git("add", ".gitignore", "configs", "src", "scripts")
    r = git("commit", "-m", "protocol-v1 freeze", "--allow-empty")
    if r.returncode:
        raise SystemExit("git commit failed: " + r.stderr + r.stdout)
    r = git("tag", "protocol-v1")
    if r.returncode and not git_tag_exists():
        raise SystemExit("git tag failed: " + r.stderr)
    return git("rev-parse", "HEAD").stdout.strip()


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--freeze", action="store_true")
    g.add_argument("--restore", action="store_true")
    a = ap.parse_args()
    d = digest()
    if a.freeze:
        if REC.exists():
            old = json.load(open(REC))
            if old["sha256"] != d:
                raise SystemExit("Already frozen with DIFFERENT code. Refusing to overwrite. Create protocol-v1.1 and log the change.")
            print("Already frozen with identical code:", old["sha256"][:16]); make_tag(); return
        REC.parent.mkdir(exist_ok=True)
        commit = make_tag()
        json.dump({"sha256": d, "commit": commit, "time": time.strftime("%Y-%m-%d %H:%M:%S"),
                   "files": [str(p.relative_to(ROOT)) for p in files()]}, open(REC, "w"), indent=2)
        print(f"FROZEN. tag protocol-v1 -> {commit[:10]} | code sha256 {d[:16]}... | record {REC}")
    else:
        if not REC.exists():
            raise SystemExit("No freeze record found: the protocol has not been frozen.")
        old = json.load(open(REC))
        if old["sha256"] != d:
            raise SystemExit("Code differs from the frozen version (undisclosed change). Do not run the test part. "
                             "Restore the original files or create protocol-v1.1 with a log entry.")
        make_tag()
        print("Freeze verified: code identical to the frozen version; tag protocol-v1 present:", git_tag_exists())


if __name__ == "__main__":
    main()


Writing scripts/11_freeze_protocol.py


In [ ]:
%%writefile scripts/12_audit_outputs.py
#!/usr/bin/env python
"""Pre-evaluation audit (alignment, missing values, partitions, metric direction, leakage, comparator consistency). No metric is computed.
  python scripts/12_audit_outputs.py --mode pilot
  python scripts/12_audit_outputs.py --mode final          (needs the freeze; --skip-tag-check downgrades freeze errors to warnings)
Exit code 1 if any ERROR. Writes results/<mode>/audit_report.json. scripts/06_evaluate.py runs this automatically and refuses to continue on ERROR."""
import argparse
import json

from _common import ROOT, load_cfg, audit_from_disk

from uncertainty.audit import format_report

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["pilot", "final"], required=True)
    ap.add_argument("--skip-tag-check", action="store_true")
    a = ap.parse_args()
    rep = audit_from_disk(a.mode, load_cfg(), strict_freeze=not a.skip_tag_check)
    out = ROOT / "results" / a.mode
    out.mkdir(parents=True, exist_ok=True)
    json.dump(rep, open(out / "audit_report.json", "w"), indent=2, default=str)
    print(format_report(rep))
    raise SystemExit(0 if rep["passed"] else 1)


Writing scripts/12_audit_outputs.py


In [ ]:
%%writefile scripts/13_results_outline.py
#!/usr/bin/env python
"""Prompt 14: write results/final/results_discussion_outline.md from the REAL final outputs (no new analysis).
Run only after scripts/06_evaluate.py --mode final --frozen has produced results/final/. Mechanisms and alternative explanations are
listed as things to check; [AUTHOR] marks the parts only you can write after looking at the evidence."""
import json

import pandas as pd

from _common import ROOT

from uncertainty.report import build_outline

if __name__ == "__main__":
    fin = ROOT / "results" / "final"
    if not (fin / "summary.json").exists():
        raise SystemExit("results/final/summary.json not found: run scripts/06_evaluate.py --mode final --frozen first. "
                         "This tool never interprets pilot or synthetic numbers.")
    S = json.load(open(fin / "summary.json"))
    if S.get("mode") != "final" or S.get("rq3") is None:
        raise SystemExit("summary.json is not from a final evaluation.")

    def opt_csv(name):
        p = fin / name
        return pd.read_csv(p) if p.exists() and p.stat().st_size > 5 else None

    def opt_json(p):
        return json.load(open(p)) if p.exists() else None

    md = build_outline(S, pd.read_csv(fin / "table2.csv"), pd.read_csv(fin / "table3.csv"), pd.read_csv(fin / "table4.csv"),
                       opt_csv("table6.csv"), opt_json(fin / "extension_E1.json"), opt_csv("table5.csv"),
                       opt_json(ROOT / "results" / "pilot" / "handcheck_agreement.json"))
    out = fin / "results_discussion_outline.md"
    out.write_text(md)
    print(md)
    print(f"\nwrote {out}")


Writing scripts/13_results_outline.py


In [ ]:
%%writefile scripts/collect_diagnostics.py
#!/usr/bin/env python
"""Collect the evidence needed to debug a failure, safely, into logs/diagnostics.txt (Prompt 11).

Usage (after a failure):
  python scripts/collect_diagnostics.py --traceback-file my_traceback.txt --command "python scripts/02_generate.py --split pilot"
  (save the failing console output to a text file first; you can also add --log logs/env_test_output.txt more than once)
It never prints environment variables and redacts token-like strings. Read the file once before sharing it."""
import argparse
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib import metadata
from pathlib import Path

from _common import ROOT, git_commit, load_cfg

from uncertainty.diagnostics import redact, tail

PKGS = ["torch", "transformers", "datasets", "accelerate", "numpy", "scipy", "scikit-learn", "pandas", "PyYAML", "matplotlib"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--traceback-file", default=None)
    ap.add_argument("--command", default="(not given)")
    ap.add_argument("--expected", default="(not given)")
    ap.add_argument("--log", action="append", default=[])
    a = ap.parse_args()
    L = [f"DIAGNOSTICS | {time.strftime('%Y-%m-%d %H:%M:%S')}", f"failing command : {a.command}", f"expected        : {a.expected}",
         f"python          : {sys.version.split()[0]} | {platform.platform()}", f"git commit      : {git_commit()}"]
    try:
        cfg = load_cfg()
        L.append(f"protocol        : {cfg['protocol_version']} ({cfg['status']})")
    except Exception as e:  # noqa: BLE001
        L.append(f"protocol        : could not load config ({type(e).__name__}: {e})")
    L.append("\npackages:")
    for p in PKGS:
        try:
            L.append(f"  {p:<14}{metadata.version(p)}")
        except metadata.PackageNotFoundError:
            L.append(f"  {p:<14}MISSING")
    smi = shutil.which("nvidia-smi")
    if smi:
        out = subprocess.run([smi], capture_output=True, text=True, timeout=30).stdout.splitlines()[:12]
        L += ["\nnvidia-smi (first lines):"] + ["  " + x for x in out]
    else:
        L.append("\nnvidia-smi: not found (no NVIDIA GPU visible)")
    try:
        import torch
        L.append(f"torch.cuda.is_available(): {torch.cuda.is_available()} | torch CUDA build: {torch.version.cuda}")
    except Exception as e:  # noqa: BLE001
        L.append(f"torch import failed: {type(e).__name__}: {e}")
    L.append("\nfiles (sizes only, no contents):")
    for rel in ["data/subset_question_ids.csv", "data/subset_manifest.json", "results/raw/generations.jsonl", "results/raw/semantic.jsonl",
                "results/per_question.csv", "results/pilot/summary.json", "results/final/summary.json"]:
        p = ROOT / rel
        if p.exists():
            n = sum(1 for _ in open(p, errors="replace")) if p.suffix in (".jsonl", ".csv") else "-"
            L.append(f"  {rel:<36}{p.stat().st_size:>10} bytes | lines {n}")
        else:
            L.append(f"  {rel:<36}missing")
    if a.traceback_file:
        L += ["\n=== traceback / observed output (from file) ===", tail(a.traceback_file, 200)]
    for lg in a.log:
        L += [f"\n=== last 60 lines of {lg} ===", tail(lg, 60)]
    text = redact("\n".join(L))
    (ROOT / "logs").mkdir(exist_ok=True)
    (ROOT / "logs" / "diagnostics.txt").write_text(text)
    print(text)
    print("\nSaved to logs/diagnostics.txt. Read it once, then paste it to Claude together with the code of the failing function.")


if __name__ == "__main__":
    main()


Writing scripts/collect_diagnostics.py


In [ ]:
%%writefile tests/run_all.py
"""Run every test file. Usage: python tests/run_all.py   (no GPU or internet needed)"""
import subprocess
import sys
from pathlib import Path

here = Path(__file__).resolve().parent
files = ["run_tests.py", "test_data.py", "test_core.py", "test_evaluation.py", "test_handcheck.py", "test_pipeline_scripts.py", "test_diagnostics.py", "test_extensions.py", "test_freeze.py", "test_audit.py", "test_report.py"]
bad = 0
for f in files:
    r = subprocess.run([sys.executable, str(here / f)], capture_output=True, text=True)
    last = [l for l in r.stdout.strip().splitlines() if l.strip()][-1] if r.stdout.strip() else "(no output)"
    print(f"{f:<22}{'OK  ' if r.returncode == 0 else 'FAIL'}  {last}")
    if r.returncode:
        bad += 1
        print(r.stdout[-1500:], r.stderr[-1500:])
print("\nALL TESTS PASSED (synthetic data only)" if not bad else f"\n{bad} test file(s) FAILED")
sys.exit(1 if bad else 0)


Writing tests/run_all.py


In [ ]:
%%writefile tests/run_tests.py
"""Small unit tests for the helper functions that do not need a GPU.
Run:  python tests/run_tests.py     (also pytest-compatible: pytest tests/)"""
import math
import random
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty.seed import seed_for_question, set_seed  # noqa: E402
from uncertainty.scoring import aggregate_scores, worked_example_from_review  # noqa: E402
from uncertainty.textnorm import (extract_short_answer, is_correct,  # noqa: E402
                                  is_correct_containment, normalize_answer)


def test_normalize():
    assert normalize_answer("The Beatles") == "beatles"
    # TriviaQA style: punctuation becomes a space, THEN articles are removed, so the letter "a" in "u s a" is dropped.
    # This is a faithful property of the official metric; it is harmless because answer AND aliases use the same function.
    assert normalize_answer("  U.S.A.  ") == "u s"
    assert normalize_answer("An Apple, a day!") == "apple day"


def test_normalization_matches_triviaqa_style():
    from uncertainty.textnorm import normalize_squad_style
    assert normalize_answer("Jean-Paul Sartre") == "jean paul sartre"
    assert normalize_answer("O'Brien") == "o brien"
    assert normalize_answer("New_York") == "new york"
    assert normalize_answer("\u2018The Beatles\u2019") == "beatles"
    assert normalize_answer("Paris,France") == "paris france"
    assert normalize_squad_style("U.S.A.") == "usa" != normalize_answer("U.S.A.")   # the two styles really differ


def test_regression_dataset_style_aliases_are_matched():
    """BUG found in the Prompt-10 audit: with SQuAD-style (delete-punctuation) normalization, 'U.S.A.' became 'usa' and
    could never match an alias written in the TriviaQA style ('u s a' / 'u s')."""
    assert is_correct("U.S.A.", ["u s a", "united states"]) is True
    assert is_correct("Jean-Paul Sartre", ["jean paul sartre", "sartre"]) is True
    assert is_correct("the Beatles", ["beatles"]) is True
    assert is_correct("Beatles!", ["beatles"]) is True
    assert is_correct("Rolling Stones", ["beatles"]) is False


def test_extract_short_answer():
    assert extract_short_answer("Jane Austen.\nShe wrote it.") == "Jane Austen"
    assert extract_short_answer("   ") == ""


def test_labels():
    aliases = ["jane austen", "austen"]
    assert is_correct("Jane Austen.", aliases) is True
    assert is_correct("Charlotte Bronte", aliases) is False
    # exact-match boundary: a verbose correct answer is labelled WRONG by the exact rule ...
    assert is_correct("The answer is Austen", aliases) is False
    # ... and CORRECT by the containment sensitivity rule
    assert is_correct_containment("The answer is Austen", aliases) is True
    assert is_correct("", aliases) is False


def test_seed_reproducible():
    set_seed(42)
    a = (random.random(), float(np.random.rand()))
    set_seed(42)
    b = (random.random(), float(np.random.rand()))
    assert a == b
    assert seed_for_question(1000, 7) == 1007


def test_aggregate_known_values():
    lps = [math.log(0.8), math.log(0.6), math.log(0.9)]
    ents = [0.5, 1.5, 1.0]
    agg = aggregate_scores(lps, ents)
    assert abs(agg["sum_nll"] - (-sum(lps))) < 1e-12
    assert abs(agg["mean_nll"] - (-sum(lps) / 3)) < 1e-12
    assert abs(agg["max_nll"] - (-math.log(0.6))) < 1e-12   # least probable token
    assert abs(agg["mean_entropy"] - 1.0) < 1e-12 and agg["max_entropy"] == 1.5
    assert math.isnan(aggregate_scores([], [])["mean_nll"])


def test_review_worked_example():
    """Recomputes the review's Sec. 2.3 illustration (probabilities 0.80, 0.60, 0.90)."""
    ex = worked_example_from_review()
    assert abs(ex["joint_probability"] - 0.432) < 1e-9          # review says 0.432: matches
    assert abs(ex["mean_nll"] - 0.2798) < 5e-4                  # recomputed value; review states about 0.310


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t()
            print(f"PASS  {t.__name__}")
        except AssertionError as e:
            failed += 1
            print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed")
    sys.exit(1 if failed else 0)


Writing tests/run_tests.py


In [ ]:
%%writefile tests/test_audit.py
"""Tests of the pre-evaluation audit on SYNTHETIC data: a clean case passes, and each injected defect is caught.
Run: python tests/test_audit.py"""
import copy
import sys
from pathlib import Path

import numpy as np
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
sys.path.insert(0, str(Path(__file__).resolve().parent))
from test_pipeline_scripts import make_raw  # noqa: E402
from uncertainty.audit import format_report, run_audit  # noqa: E402
from uncertainty.data import ids_hash  # noqa: E402
from uncertainty.table import build_table  # noqa: E402

CFG = yaml.safe_load(open(Path(__file__).resolve().parents[1] / "configs" / "protocol_v1.yaml"))


def world():
    gens, sems = make_raw()
    df = build_table(gens, sems)
    ids = df[["pool_index", "question_id", "split", "is_pilot"]].copy()
    manifest = {"ids_sha256": ids_hash(ids)}
    metas = [{"stage": "generate", "time": "2026-11-01 10:00:00", "split": "pilot", "torch": "2.x", "transformers": "4.x", "gpu": "T4", "protocol_status": "draft"},
             {"stage": "generate", "time": "2026-11-05 09:00:00", "split": "test", "torch": "2.x", "transformers": "4.x", "gpu": "T4", "protocol_status": "draft"}]
    freeze = {"tag_exists": True, "record": {"time": "2026-11-03 12:00:00", "sha256": "x"}, "code_matches_record": True}
    return dict(df=df, ids=ids, manifest=manifest, gens=gens, sems=sems, metas=metas, freeze=freeze)


def audit(w, mode="final", **over):
    a = {**w, **over}
    return run_audit(a.pop("df"), CFG, mode, **a)


def has(rep, check, sev, contains=""):
    return any(f["check"] == check and f["severity"] == sev and contains in f["message"] for f in rep["findings"])


def test_clean_world_passes():
    rep = audit(world())
    assert rep["passed"], format_report(rep)
    rep_p = audit(world(), mode="pilot")
    assert rep_p["passed"], format_report(rep_p)


def test_ids_partitions_labels():
    w = world()
    d = w["df"].copy(); d.loc[5, "question_id"] = d.loc[4, "question_id"]
    assert has(audit(w, df=d), "ids", "ERROR")
    d = w["df"].copy(); d.loc[d.split == "test", "question_id"] = d.loc[d.split == "test", "question_id"].values  # no-op baseline
    d.loc[d[d.split == "test"].index[0], "question_id"] = d.loc[0, "question_id"]
    rep = audit(w, df=d)
    assert has(rep, "leakage", "ERROR") and not rep["passed"]
    d = w["df"].copy(); d.loc[3, "is_pilot"] = True; d.loc[d[d.split == "test"].index[0], "is_pilot"] = True
    assert has(audit(w, df=d), "partitions", "ERROR", "pilot")
    d = w["df"].copy(); d.loc[7, "y_wrong_exact"] = np.nan
    assert has(audit(w, df=d), "labels", "ERROR")
    d = w["df"].copy(); i = d.index[d.y_wrong_exact == 0][0]; d.loc[i, "y_wrong_contain"] = 1
    assert has(audit(w, df=d), "labels", "ERROR", "impossible")


def test_ranges_and_identities():
    w = world()
    d = w["df"].copy(); d.loc[10, "all_sumNLL"] += 1.0
    assert has(audit(w, df=d), "ranges", "ERROR", "n_tokens")
    d = w["df"].copy(); d.loc[10, "SE10"] = 5.0
    assert has(audit(w, df=d), "ranges", "ERROR", "SE10 outside")
    d = w["df"].copy(); j = d.index[d.n_clusters > 1][0]; d.loc[j, "n_clusters"] = 1
    assert has(audit(w, df=d), "ranges", "ERROR", "n_clusters == 1")


def test_raw_alignment_and_labels():
    w = world()
    d = w["df"].copy(); d.loc[[20, 21, 22], "y_wrong_exact"] = 1 - d.loc[[20, 21, 22], "y_wrong_exact"]
    d.loc[[20, 21, 22], "y_wrong_contain"] = d.loc[[20, 21, 22], "y_wrong_exact"]
    assert has(audit(w, df=d), "raw", "ERROR", "labels recomputed")
    s2 = copy.deepcopy(w["sems"]); s2.append({"question_id": "ghost", "cluster_ids": [0] * 10, "nli_calls": 0, "seconds_nli": 0})
    assert has(audit(w, sems=s2), "raw", "ERROR", "no generation record")
    g2 = copy.deepcopy(w["gens"]); g2[3]["samples"] = g2[3]["samples"][:9]
    assert has(audit(w, gens=g2), "comparators", "ERROR", "exactly 10")
    g3 = copy.deepcopy(w["gens"]); a = next(g for g in g3 if g["split"] == "calibration"); b = next(g for g in g3 if g["split"] == "test")
    b["question"] = a["question"]
    assert has(audit(w, gens=g3), "leakage", "ERROR", "more than one split")


def test_subset_alignment():
    w = world()
    ids = w["ids"].copy(); ids.loc[0, "split"] = "test"
    assert has(audit(w, ids=ids), "alignment", "ERROR")
    ids2 = w["ids"].copy(); ids2.loc[1, "question_id"] = "other"
    assert has(audit(w, ids=ids2, manifest=w["manifest"]), "alignment", "ERROR")
    short = w["ids"].iloc[:900]
    rep = audit(w, mode="final", ids=w["ids"])
    assert has(rep, "alignment", "INFO", "agree") and rep["passed"]


def test_missing_values_balance_direction():
    w = world()
    d = w["df"].copy(); te = d.index[d.split == "test"]
    d.loc[te[:2], "SE10"] = np.nan
    rep2 = audit(w, df=d)
    assert has(rep2, "missing", "WARN") and rep2["passed"], format_report(rep2)       # 2 missing rows: warning only
    d2 = w["df"].copy(); d2.loc[te[:100], ["SE10", "SE5", "n_clusters"]] = np.nan
    assert has(audit(w, df=d2), "missing", "ERROR")
    d3 = w["df"].copy(); d3.loc[te, "y_wrong_exact"] = 0; d3.loc[te[:5], "y_wrong_exact"] = 1; d3["y_wrong_contain"] = d3["y_wrong_exact"]
    assert has(audit(w, df=d3, gens=None, sems=None), "balance", "ERROR")
    d4 = w["df"].copy(); ca = d4.index[d4.split == "calibration"]
    d4.loc[ca, "y_wrong_exact"] = 1 - d4.loc[ca, "y_wrong_exact"]; d4.loc[ca, "y_wrong_contain"] = d4.loc[ca, "y_wrong_exact"]
    assert has(audit(w, df=d4, gens=None, sems=None), "direction", "WARN", "orientation may be inverted")


def test_protocol_order_and_comparators():
    w = world()
    early = copy.deepcopy(w["metas"]); early[1]["time"] = "2026-11-02 09:00:00"          # test generated BEFORE the freeze
    rep = audit(w, metas=early)
    assert has(rep, "protocol", "ERROR", "BEFORE the freeze") and not rep["passed"]
    rep = audit(w, metas=early, strict_freeze=False)
    assert has(rep, "protocol", "WARN", "BEFORE the freeze") and rep["passed"]           # --skip-tag-check downgrades, still visible
    assert has(audit(w, freeze={**w["freeze"], "tag_exists": False}), "protocol", "ERROR", "tag")
    assert has(audit(w, freeze={**w["freeze"], "code_matches_record": False}), "protocol", "ERROR", "undisclosed")
    assert has(audit(w, freeze=None), "protocol", "ERROR")
    het = copy.deepcopy(w["metas"]); het[0]["gpu"] = "A100"
    assert has(audit(w, metas=het), "comparators", "WARN", "different environments")


def test_missing_inputs_are_skipped_not_passed():
    w = world()
    rep = audit(w, mode="pilot", ids=None, manifest=None, gens=None, sems=None, metas=None)
    skipped = [f["check"] for f in rep["findings"] if f["severity"] == "SKIPPED"]
    assert {"alignment", "raw", "comparators"} <= set(skipped) and rep["n_skipped"] >= 3
    assert "SKIPPED" in format_report(rep)


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            import traceback; traceback.print_exc()
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_audit.py


In [ ]:
%%writefile tests/test_core.py
"""Unit tests for pure-Python/numpy logic: jsonl resume, span masks, semantic entropy, metrics, cascade.
No GPU, no network. Run: python tests/test_core.py"""
import json
import math
import sys
import tempfile
from pathlib import Path

import numpy as np
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty import cascade as C  # noqa: E402
from uncertainty import metrics as M  # noqa: E402
from uncertainty.generation import answer_span_mask, token_char_spans  # noqa: E402
from uncertainty.io_utils import append_jsonl, done_ids, read_jsonl, repair_jsonl  # noqa: E402
from uncertainty.semantic_entropy import cluster_samples, semantic_entropy_from_ids  # noqa: E402
from uncertainty.textnorm import normalize_answer  # noqa: E402


def test_jsonl_resume_and_repair():
    with tempfile.TemporaryDirectory() as t:
        p = Path(t) / "x.jsonl"
        append_jsonl(p, {"question_id": "a", "v": 1})
        append_jsonl(p, {"question_id": "b", "v": 2})
        with open(p, "a") as f:
            f.write('{"question_id": "c", "v"')          # interrupted write
        assert done_ids(p) == {"a", "b"}
        assert repair_jsonl(p) == 2
        append_jsonl(p, {"question_id": "c", "v": 3})
        assert [r["question_id"] for r in read_jsonl(p)] == ["a", "b", "c"]


def test_token_spans_and_answer_mask():
    # tokens: "The", " answer", " is", " Paris", ".", EOS('')  -> decoded "The answer is Paris."
    pieces = ["The", " answer", " is", " Paris", ".", ""]
    decode = lambda ids: "".join(pieces[i] for i in ids)  # noqa: E731  (ids index into pieces)
    spans = token_char_spans(decode, list(range(6)))
    assert spans[0] == (0, 3) and spans[3] == (13, 19) and spans[5] == (20, 20)
    text = "The answer is Paris."
    mask = answer_span_mask(spans, text, "The answer is Paris")       # trailing '.' is stripped from the short answer, so excluded
    assert mask == [True, True, True, True, False, False]
    mask2 = answer_span_mask(spans, text, "Paris")
    assert mask2 == [False, False, False, True, False, False]
    assert answer_span_mask(spans, text, "Rome") == [False] * 6 and answer_span_mask(spans, text, "") == [False] * 6


def _fake_entails(q, a, b):
    return normalize_answer(a).split()[0] == normalize_answer(b).split()[0]


def test_clustering_and_call_counts():
    samples = ["Paris", "paris.", "Paris, France", "Lyon", "Lyon", "Marseille"]
    ids, calls = cluster_samples("Capital?", samples, _fake_entails)
    assert ids == [0, 0, 0, 1, 1, 2], ids
    assert calls == 6, calls          # 2 (Paris,France) + 1 (Lyon vs cl.0) + 1 (Lyon dup vs cl.0) + 2 (Marseille)
    expected = -(0.5 * math.log(0.5) + (2 / 6) * math.log(2 / 6) + (1 / 6) * math.log(1 / 6))
    assert abs(semantic_entropy_from_ids(ids) - expected) < 1e-12


def test_empty_answers_skip_nli():
    ids, calls = cluster_samples("q", ["", "", "x"], lambda *a: (_ for _ in ()).throw(AssertionError("NLI called")))
    assert ids == [0, 0, 1] and calls == 0


def test_se_values_and_prefix_property():
    ids = [0] * 6 + [1] * 3 + [2]
    assert abs(semantic_entropy_from_ids(ids) - 0.8979457) < 1e-6    # the review's worked example (0.60/0.30/0.10)
    assert semantic_entropy_from_ids([0] * 10) == 0.0
    # prefix property: SE@5 from stored ids equals clustering the first 5 samples alone
    samples = ["a x", "a y", "b", "a z", "b q", "c", "a", "b", "c", "d"]
    ids_full, _ = cluster_samples("q", samples, _fake_entails)
    ids_5, _ = cluster_samples("q", samples[:5], _fake_entails)
    assert ids_full[:5] == ids_5
    assert semantic_entropy_from_ids(ids_full, k=5) == semantic_entropy_from_ids(ids_5)


def test_auroc_matches_sklearn_with_ties():
    rs = np.random.RandomState(0)
    y = rs.randint(0, 2, 300)
    s = np.round(rs.randn(300) + y * 0.7, 1)        # many ties
    assert abs(M.auroc(y, s) - roc_auc_score(y, s)) < 1e-12
    assert math.isnan(M.auroc(np.zeros(5), np.arange(5)))
    assert M.auroc([0, 1, 0, 1], [0.1, 0.9, 0.2, 0.8]) == 1.0 and M.auroc([0, 1], [0.9, 0.1]) == 0.0


def test_bootstrap_paired_and_ci():
    rs = np.random.RandomState(1)
    y = rs.randint(0, 2, 200)
    s = rs.randn(200) + y
    idx = M.bootstrap_indices(200, 300, 2026)
    idx2 = M.bootstrap_indices(200, 300, 2026)
    assert (idx == idx2).all() and idx.shape == (300, 200)
    boots = M.boot_stat(M.auroc, y, s, idx)
    lo, hi = M.percentile_ci(boots)
    assert lo < M.auroc(y, s) < hi


def test_platt_ece_brier():
    rs = np.random.RandomState(2)
    x = rs.randn(2000)
    p_true = 1 / (1 + np.exp(-(1.5 * x)))
    y = (rs.rand(2000) < p_true).astype(int)
    pl = M.PlattScaler().fit(x, y)
    p = pl.predict(x)
    assert pl.slope > 0 and M.ece(y, p) < 0.05                     # well calibrated by construction
    assert M.ece([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]) == 0.0
    assert abs(M.brier([1, 0], [1.0, 0.0])) < 1e-12 and abs(M.brier([1, 0], [0.0, 1.0]) - 1.0) < 1e-12
    try:
        M.PlattScaler().fit([1, 2, 3], [1, 1, 1]); raise AssertionError("should raise")
    except ValueError:
        pass


def test_cascade_endpoints_and_rank_rule():
    rs = np.random.RandomState(3)
    y_cal = rs.randint(0, 2, 400); s_cal = rs.randn(400) + y_cal; se_cal = rs.randn(400) + 1.5 * y_cal
    y_te = rs.randint(0, 2, 600); s_te = rs.randn(600) + y_te; se_te = rs.randn(600) + 1.5 * y_te
    pt, ps = M.PlattScaler().fit(s_cal, y_cal), M.PlattScaler().fit(se_cal, y_cal)
    r0, e0 = C.cascade_risk(0.0, s_cal, s_te, se_te, pt, ps)
    r1, e1 = C.cascade_risk(1.0, s_cal, s_te, se_te, pt, ps)
    assert e0.all() and not e1.any()                                 # q=0: SE for all; q=1: token score only
    assert abs(M.auroc(y_te, r0) - M.auroc(y_te, se_te)) < 1e-12    # positive slope -> same ranking as raw SE
    assert abs(M.auroc(y_te, r1) - M.auroc(y_te, s_te)) < 1e-12
    r5, e5 = C.cascade_risk(0.5, s_cal, s_te, se_te, pt, ps)
    assert 0.3 < e5.mean() < 0.7
    rr = C.rank_rule_risk(e5, s_te, se_te)
    assert rr[e5].min() > rr[~e5].max()                              # escalated always ranked above accepted
    cost = C.cost_summary(e5, np.full(600, 4), np.full(600, 0.1), np.full(600, 1.0), np.full(600, 0.5), 10)
    assert abs(cost["extra_generations_per_question"] - 10 * e5.mean()) < 1e-12
    assert abs(cost["seconds_per_question"] - (0.1 + 1.5 * e5.mean())) < 1e-9


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_core.py


In [ ]:
%%writefile tests/test_data.py
"""Sanity tests for Cycle 1 on SYNTHETIC data (no network, no GPU). Run: python tests/test_data.py
These tests prove the logic of the code only; they say nothing about the real TriviaQA data."""
import random
import sys
import tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty import data as D  # noqa: E402

CFG = {"pool_size": 1000, "pool_seed": 42, "split_seed": 42, "calibration_fraction": 0.4, "pilot_size": 50}


def make_records(n_unique=3000, duplicate_sources=True, extra_text_dupes=3, no_alias=0):
    recs = []
    for i in range(n_unique):
        base = {"question": f"Question number {i}?", "question_id": f"tc_{i}",
                "answer": {"value": f"Ans{i}", "aliases": [f"Ans{i}", f"A{i}"], "normalized_aliases": [f"ans{i}", f"a{i}"]}}
        recs.append({**base, "question_source": "http://web"})
        if duplicate_sources:
            recs.append({**base, "question_source": "http://wikipedia"})
    for j in range(extra_text_dupes):   # same text, different id
        recs.append({"question": "Question number 0?", "question_id": f"dupe_{j}", "question_source": "x",
                     "answer": {"value": "Z", "aliases": ["Z"], "normalized_aliases": ["z"]}})
    for j in range(no_alias):
        recs.append({"question": f"No alias q {j}?", "question_id": f"noalias_{j}", "question_source": "x",
                     "answer": {"value": "", "aliases": [], "normalized_aliases": []}})
    return recs


def test_rows_and_dedupe():
    df = D.rows_from_records(make_records())
    assert list(df.columns) == D.COLUMNS and len(df) == 6003
    clean, st = D.dedupe(df)
    assert st["rows_before"] == 6003
    assert st["dropped_repeated_question_id"] == 3000
    assert st["dropped_repeated_question_text"] == 3          # the 3 extra rows with repeated text
    assert st["rows_after"] == 3000 and clean["question_id"].is_unique
    assert st["question_ids_with_conflicting_aliases"] == 0


def test_dedupe_order_independent():
    df = D.rows_from_records(make_records())
    a, _ = D.dedupe(df)
    b, _ = D.dedupe(df.sample(frac=1.0, random_state=7).reset_index(drop=True))
    assert a[["question_id", "question_source"]].equals(b[["question_id", "question_source"]])


def test_conflicting_aliases_counted():
    recs = make_records(10, extra_text_dupes=0)
    recs[1]["answer"] = {"value": "Other", "aliases": ["Other"], "normalized_aliases": ["other"]}  # wikipedia copy of tc_0
    _, st = D.dedupe(D.rows_from_records(recs))
    assert st["question_ids_with_conflicting_aliases"] == 1


def test_validate_and_drop_unlabellable():
    df = D.rows_from_records(make_records(10, extra_text_dupes=0, no_alias=2))
    assert D.validate(df)["no_normalized_aliases"] == 2
    kept, n = D.drop_unlabellable(df)
    assert n == 2 and D.validate(kept)["no_normalized_aliases"] == 0


def test_pool_deterministic_and_order_independent():
    df = D.rows_from_records(make_records())
    clean, _ = D.dedupe(df)
    p1 = D.sample_pool(clean, 1000, 42)
    p2 = D.sample_pool(clean.sample(frac=1.0, random_state=1).reset_index(drop=True), 1000, 42)
    p3 = D.sample_pool(clean, 1000, 43)
    assert len(p1) == 1000 and p1["question_id"].is_unique
    assert list(p1["pool_index"]) == list(range(1000))
    assert list(p1["question_id"]) == list(p2["question_id"])
    assert list(p1["question_id"]) != list(p3["question_id"])


def test_split_properties():
    clean, _ = D.dedupe(D.rows_from_records(make_records()))
    sub = D.split_pool(D.sample_pool(clean, 1000, 42), 0.4, 42, 50)
    cal, test = sub[sub.split == "calibration"], sub[sub.split == "test"]
    assert len(cal) == 400 and len(test) == 600
    assert set(cal.question_id).isdisjoint(set(test.question_id))
    assert sub.is_pilot.sum() == 50 and set(sub[sub.is_pilot].split) == {"calibration"}


def test_hash_and_ids_only_roundtrip():
    clean, _ = D.dedupe(D.rows_from_records(make_records()))
    sub, rep = D.build_subset(D.rows_from_records(make_records()), CFG)
    sub_b, rep_b = D.build_subset(D.rows_from_records(make_records()).sample(frac=1.0, random_state=5), CFG)
    assert rep["ids_sha256"] == rep_b["ids_sha256"]
    sub_c, rep_c = D.build_subset(D.rows_from_records(make_records()), {**CFG, "pool_seed": 7})
    assert rep_c["ids_sha256"] != rep["ids_sha256"]
    with tempfile.TemporaryDirectory() as t:
        D.save_subset(sub, rep, Path(t) / "ids.csv", Path(t) / "m.json")
        ids = pd.read_csv(Path(t) / "ids.csv")
        assert list(ids.columns) == D.IDS_COLUMNS            # no question text is saved
        rebuilt = D.attach_text(ids, clean)
        assert len(rebuilt) == 1000 and rebuilt["question"].notna().all()
        bad = ids.copy(); bad.loc[0, "question_id"] = "does_not_exist"
        try:
            D.attach_text(bad, clean); raise AssertionError("should have raised")
        except ValueError:
            pass


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_data.py


In [ ]:
%%writefile tests/test_diagnostics.py
"""Tests for diagnostics redaction. Run: python tests/test_diagnostics.py"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty.diagnostics import redact, tail  # noqa: E402


def test_redaction():
    s = "login hf_abcdefghijklmnopqrstuvwxyz123456 ok; HF_TOKEN=abc123; api_key: XYZ; Authorization: Bearer abc.def"
    r = redact(s)
    assert "abcdefghijklmnop" not in r and "abc123" not in r and "XYZ" not in r and "abc.def" not in r
    assert "login" in r and "ok" in r                      # normal text survives
    assert redact("CUDA out of memory. Tried to allocate 2.00 GiB") == "CUDA out of memory. Tried to allocate 2.00 GiB"


def test_tail_missing_file():
    assert "not found" in tail("/nonexistent/file.txt")


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed")
    sys.exit(1 if failed else 0)


Writing tests/test_diagnostics.py


In [ ]:
%%writefile tests/test_evaluation.py
"""End-to-end tests of table building + evaluation + LaTeX tables on SYNTHETIC data with a planted signal.
They prove the pipeline logic, not any real result. Run: python tests/test_evaluation.py"""
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty import metrics as M  # noqa: E402
from uncertainty.evaluation import PRIMARY, run_analysis  # noqa: E402
from uncertainty.latex_tables import make_tables  # noqa: E402
from uncertainty.table import TOKEN_VARIANTS, build_table, row_from_records  # noqa: E402

CFG = yaml.safe_load(open(Path(__file__).resolve().parents[1] / "configs" / "protocol_v1.yaml"))


def synth_df(n_cal=400, n_test=600, seed=0, token_effect=0.5, se_effect=1.2):
    rs = np.random.RandomState(seed)
    n = n_cal + n_test
    y = rs.randint(0, 2, n)
    d = {"question_id": [f"q{i}" for i in range(n)], "pool_index": np.arange(n),
         "split": ["calibration"] * n_cal + ["test"] * n_test,
         "is_pilot": [i < 50 for i in range(n_cal)] + [False] * n_test,
         "y_wrong_exact": y, "y_wrong_contain": np.where(rs.rand(n) < 0.05, 1 - y, y),
         "n_tokens": rs.randint(1, 8, n).astype(float)}
    base = rs.randn(n)
    for v in TOKEN_VARIANTS:
        d[v] = base + (token_effect * y if "meanNLL" in v else 0.2 * y) + 0.1 * rs.randn(n)
    d["SE10"] = rs.randn(n) + se_effect * y
    d["SE5"] = d["SE10"] + 0.3 * rs.randn(n)
    d["nli_calls"] = rs.randint(0, 20, n).astype(float)
    d["seconds_greedy"] = np.full(n, 0.1); d["seconds_samples"] = np.full(n, 1.0); d["seconds_nli"] = np.full(n, 0.3)
    return pd.DataFrame(d)


def test_row_from_records_and_build_table():
    gen = {"question_id": "q1", "pool_index": 3, "split": "test", "is_pilot": False, "short_answer": "Paris",
           "greedy_text": "Paris.", "token_logprob": [-0.2, -0.1, -0.05], "token_entropy": [1.0, 0.5, 0.1],
           "is_eos": [False, False, True], "span_mask": [True, False, False], "truncated": False,
           "normalized_aliases": ["paris"], "seconds_greedy": 0.2, "seconds_samples": 1.5}
    sem = {"question_id": "q1", "cluster_ids": [0] * 6 + [1] * 3 + [2], "nli_calls": 7, "seconds_nli": 0.4}
    r = row_from_records(gen, sem)
    assert r["n_tokens"] == 2 and abs(r["all_meanNLL"] - 0.15) < 1e-12 and abs(r["all_maxNLL"] - 0.2) < 1e-12
    assert abs(r["span_meanNLL"] - 0.2) < 1e-12 and r["span_fallback"] is False
    assert r["y_wrong_exact"] == 0 and abs(r["SE10"] - 0.8979457) < 1e-6 and r["n_clusters"] == 3
    gen2 = {**gen, "span_mask": [False, False, False], "question_id": "q2", "pool_index": 1, "greedy_text": "Rome"}
    df = build_table([gen, gen2], [sem])
    assert list(df.pool_index) == [1, 3] and df.loc[0, "span_fallback"] and df.loc[0, "y_wrong_exact"] == 1
    assert np.isnan(df.loc[0, "SE10"])                      # no semantic record for q2
    assert "question" not in df.columns and "gold_value" not in df.columns and "short_answer" not in df.columns


def test_final_analysis_with_planted_signal():
    df = synth_df()
    with tempfile.TemporaryDirectory() as t:
        s = run_analysis(df, CFG, "final", t, n_boot=200, make_figs=True, provenance={"input_sha256": "abc", "git_commit": None})
        res = pd.read_csv(Path(t) / "results.csv")
        assert (res["input_sha256"] == "abc").all() and "git_commit" in res.columns and s["provenance"]["input_sha256"] == "abc"
        for f in ("table2.csv", "table3.csv", "table4.csv", "table6.csv", "results.csv", "summary.json",
                  "figs/fig2_roc.png", "figs/fig2_roc.pdf", "figs/fig3_aggregators.png", "figs/fig4_cascade.png", "figs/fig5_reliability.png"):
            assert (Path(t) / f).exists(), f
        assert s["rq1"]["token_is_predictive"] and s["rq1"]["H1"] == "supported"      # planted: SE stronger than token
        t2 = pd.read_csv(Path(t) / "table2.csv").set_index("key")
        assert 0.4 < t2.loc["random", "auroc"] < 0.6 and abs(t2.loc["length", "auroc"] - 0.5) < 0.1
        assert t2.loc["se", "auroc"] > t2.loc["token", "auroc"] > 0.55
        t4 = pd.read_csv(Path(t) / "table4.csv").set_index("role")
        assert abs(t4.loc["SE-for-all", "escalated_fraction"] - 1.0) < 1e-12 and t4.loc["token-only", "escalated_fraction"] == 0.0
        assert abs(t4.loc["SE-for-all", "auroc"] - t2.loc["se", "auroc"]) < 1e-9
        assert abs(t4.loc["token-only", "auroc"] - t2.loc["token", "auroc"]) < 1e-9
        assert s["rq3"]["H3"] in ("supported", "inconclusive", "not supported")
        assert abs(s["rq3"]["D"] - (s["rq3"]["auroc_se_raw"] - s["rq3"]["auroc_cascade"])) < 1e-12
        assert set(s["rq2"]["contrasts"]) == {"C1_span_vs_all_meanNLL", "C2_maxNLL_vs_meanNLL", "C3_meanH_vs_meanNLL"}
        out = make_tables(t, Path(t) / "tex")
        assert {"table_main.tex", "table_cascade.tex", "table_ablation.tex", "table_sensitivity.tex"} <= set(out)
        assert "\\label{tab:main}" in (Path(t) / "tex" / "table_main.tex").read_text()


def test_null_signal_is_reported_not_hidden():
    df = synth_df(token_effect=0.0, se_effect=0.0, seed=5)
    for v in TOKEN_VARIANTS:
        df[v] = np.random.RandomState(9).randn(len(df))
    with tempfile.TemporaryDirectory() as t:
        s = run_analysis(df, CFG, "final", t, n_boot=200, make_figs=False)
        assert not s["rq1"]["token_is_predictive"] or s["rq1"]["token_auroc_ci"][0] < 0.56
        assert s["rq1"]["H1"].startswith("not supported")
        assert any("not predictive" in f for f in s["flags"]) or s["rq1"]["token_is_predictive"]


def test_pilot_mode_never_reads_test_rows():
    df = synth_df()
    test_rows = df.split == "test"
    df.loc[test_rows, ["SE10", PRIMARY]] = np.nan       # poison: any use of test rows would break or show up
    with tempfile.TemporaryDirectory() as t:
        s = run_analysis(df, CFG, "pilot", t, n_boot=100, make_figs=False)
        assert s["n_eval"] == 50 and s["rq3"] is None and not (Path(t) / "table4.csv").exists()


def test_final_requires_class_balance():
    df = synth_df()
    df.loc[df.split == "test", "y_wrong_exact"] = 0
    df.loc[df[df.split == "test"].index[:5], "y_wrong_exact"] = 1
    with tempfile.TemporaryDirectory() as t:
        try:
            run_analysis(df, CFG, "final", t, n_boot=50, make_figs=False); raise AssertionError("should raise")
        except RuntimeError as e:
            assert "class balance" in str(e)
        s = run_analysis(df, CFG, "final", t, n_boot=50, make_figs=False, ignore_balance=True)
        assert s["class_balance_ok"] is False and any("class balance" in f for f in s["flags"])


def test_missing_scores_dropped_listwise():
    df = synth_df()
    df.loc[df[df.split == "test"].index[:7], "SE10"] = np.nan
    with tempfile.TemporaryDirectory() as t:
        s = run_analysis(df, CFG, "final", t, n_boot=50, make_figs=False)
        assert s["n_eval"] == 593 and any("dropped" in f for f in s["flags"])


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            import traceback; traceback.print_exc()
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_evaluation.py


In [ ]:
%%writefile tests/test_extensions.py
"""Tests for the isolated extension E1 on SYNTHETIC data. Run: python tests/test_extensions.py"""
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
sys.path.insert(0, str(Path(__file__).resolve().parent))
from test_evaluation import synth_df  # noqa: E402
from uncertainty.evaluation import PRIMARY  # noqa: E402
from uncertainty.extensions import LengthAdjuster, run_length_adjusted_extension  # noqa: E402

CFG = yaml.safe_load(open(Path(__file__).resolve().parents[1] / "configs" / "protocol_v1.yaml"))


def confounded_df(seed=0):
    df = synth_df(seed=seed)
    rs = np.random.RandomState(seed + 1)
    n = len(df)
    df["n_tokens"] = rs.randint(1, 9, n).astype(float)
    p = 1 / (1 + np.exp(-0.6 * (df["n_tokens"] - 4.5)))          # longer answers are more often wrong
    df["y_wrong_exact"] = (rs.rand(n) < p).astype(int)
    df[PRIMARY] = 0.5 * df["y_wrong_exact"] + 0.4 * df["n_tokens"] + 0.3 * rs.randn(n)
    return df


def test_adjuster_fit_on_calibration_only_and_removes_linear_length():
    df = confounded_df()
    cal, te = df[df.split == "calibration"], df[df.split == "test"]
    adj = LengthAdjuster().fit(cal[PRIMARY], cal["n_tokens"])
    resid = adj.transform(cal[PRIMARY], cal["n_tokens"])
    assert abs(np.corrcoef(resid, cal["n_tokens"])[0, 1]) < 1e-9 and abs(resid.mean()) < 1e-9
    te2 = te.copy(); te2[PRIMARY] = te2[PRIMARY] + 100                   # changing TEST data must not change the fit
    a2 = LengthAdjuster().fit(cal[PRIMARY], cal["n_tokens"])
    assert (adj.a, adj.b) == (a2.a, a2.b)
    try:
        LengthAdjuster().fit([1, 2, 3], [5, 5, 5]); raise AssertionError("should raise")
    except ValueError:
        pass


def test_extension_runs_when_triggered_and_preserves_baseline():
    df = confounded_df()
    before = df.copy(deep=True)
    with tempfile.TemporaryDirectory() as t:
        res = run_length_adjusted_extension(df, CFG, t, n_boot=200)
        assert res["trigger_length_confound_present"] and res["ran"]
        assert res["verdict"] in ("improves", "worse", "no evidence of improvement")      # no direction is assumed
        assert abs(res["difference"] - (res["auroc_adjusted"] - res["auroc_primary"])) < 1e-12
        assert (Path(t) / "extension_E1.json").exists()
        assert json.load(open(Path(t) / "extension_E1.json"))["status"].startswith("pre_registered")
    pd.testing.assert_frame_equal(df, before)                                  # baseline data untouched


def test_extension_not_run_without_length_confound():
    df = synth_df(seed=3)                                                      # n_tokens independent of the label
    with tempfile.TemporaryDirectory() as t:
        res = run_length_adjusted_extension(df, CFG, t, n_boot=200)
        assert res["ran"] is False and "trigger not met" in res["reason"]


def test_extension_handles_constant_length():
    df = synth_df(seed=4)
    df["n_tokens"] = 3.0
    with tempfile.TemporaryDirectory() as t:
        res = run_length_adjusted_extension(df, CFG, t, n_boot=100)
        assert res["ran"] is False


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            import traceback; traceback.print_exc()
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_extensions.py


In [ ]:
%%writefile tests/test_freeze.py
"""Tests for scripts/11_freeze_protocol.py in a temporary copy of the repo (needs git; skipped if absent). Run: python tests/test_freeze.py"""
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]


def copy_repo(dst):
    for d in ("scripts", "src", "configs"):
        shutil.copytree(REPO / d, dst / d, ignore=shutil.ignore_patterns("__pycache__"))
    (dst / "data").mkdir(exist_ok=True)


def run(cwd, *args):
    return subprocess.run([sys.executable, "scripts/11_freeze_protocol.py", *args], cwd=cwd, capture_output=True, text=True)


def tag_exists(cwd):
    return subprocess.run(["git", "-C", str(cwd), "rev-parse", "-q", "--verify", "refs/tags/protocol-v1"], capture_output=True).returncode == 0


def test_freeze_restore_and_refusal():
    if shutil.which("git") is None:
        print("SKIP (git not installed)"); return
    with tempfile.TemporaryDirectory() as t1, tempfile.TemporaryDirectory() as t2:
        a, b = Path(t1), Path(t2)
        copy_repo(a)
        assert not tag_exists(a)
        r = run(a, "--freeze"); assert r.returncode == 0 and tag_exists(a), r.stdout + r.stderr
        assert run(a, "--freeze").returncode == 0                                   # idempotent with identical code
        # new "session": same code, only the Drive record survives
        copy_repo(b); shutil.copy(a / "data" / "protocol_freeze.json", b / "data" / "protocol_freeze.json")
        assert not tag_exists(b)
        r = run(b, "--restore"); assert r.returncode == 0 and tag_exists(b), r.stdout + r.stderr
        # silent code change after freezing must be refused
        (a / "src" / "uncertainty" / "metrics.py").write_text((a / "src" / "uncertainty" / "metrics.py").read_text() + "\n# tweak\n")
        r = run(a, "--freeze"); assert r.returncode != 0 and "DIFFERENT code" in (r.stdout + r.stderr)
        r = run(a, "--restore"); assert r.returncode != 0 and "undisclosed" in (r.stdout + r.stderr)
    with tempfile.TemporaryDirectory() as t3:
        c = Path(t3); copy_repo(c)
        r = run(c, "--restore"); assert r.returncode != 0 and "not been frozen" in (r.stdout + r.stderr)


if __name__ == "__main__":
    try:
        test_freeze_restore_and_refusal(); print("PASS  test_freeze_restore_and_refusal\n\n1/1 passed")
    except Exception as e:  # noqa: BLE001
        print("FAIL", repr(e)); sys.exit(1)


Writing tests/test_freeze.py


In [ ]:
%%writefile tests/test_handcheck.py
"""Tests for hand-check helpers (synthetic). Run: python tests/test_handcheck.py"""
import math
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from uncertainty.handcheck import (agreement, parse_human_label, select_error_analysis,  # noqa: E402
                                   select_for_labelcheck)


def test_parse():
    assert parse_human_label("Yes") is True and parse_human_label(" N ") is False
    assert parse_human_label("") is None and parse_human_label(None) is None and parse_human_label("maybe") is None


def test_selection_deterministic_and_blind():
    recs = [{"pool_index": i, "question_id": f"q{i}", "SE10": 9.9} for i in range(100)]
    a = select_for_labelcheck(recs, 30, 42)
    b = select_for_labelcheck(list(reversed(recs)), 30, 42)
    assert [r["pool_index"] for r in a] == [r["pool_index"] for r in b] and len(a) == 30
    assert len(select_for_labelcheck(recs[:10], 30, 42)) == 10


def test_agreement():
    out = agreement([True, False, True, False, True], [True, False, False, None, True])
    assert out["n_compared"] == 4 and out["n_unclear_excluded"] == 1 and abs(out["agreement"] - 0.75) < 1e-12
    assert out["auto_correct_human_wrong"] == 1 and out["auto_wrong_human_correct"] == 0
    assert math.isnan(agreement([True], [None])["agreement"])


def test_error_analysis_selection():
    rows = [{"question_id": f"q{i}", "SE10": i / 10, "y_wrong_exact": i % 2} for i in range(100)]
    wrong, right = select_error_analysis(rows, 5)
    assert all(r["y_wrong_exact"] == 1 for r in wrong) and [r["question_id"] for r in wrong] == ["q1", "q3", "q5", "q7", "q9"]
    assert all(r["y_wrong_exact"] == 0 for r in right) and right[0]["question_id"] == "q98"


def test_compare_generation_runs():
    from uncertainty.repro import compare_generation_runs
    a = [{"question_id": "q1", "greedy_text": "x", "samples": ["a", "b"], "token_logprob": [-0.1, -0.2]},
         {"question_id": "q2", "greedy_text": "y", "samples": ["c", "d"], "token_logprob": [-0.3]}]
    b = [{"question_id": "q1", "greedy_text": "x", "samples": ["a", "b"], "token_logprob": [-0.1, -0.21]},
         {"question_id": "q2", "greedy_text": "z", "samples": ["c", "e"], "token_logprob": [-0.3]},
         {"question_id": "q3", "greedy_text": "w", "samples": [], "token_logprob": []}]
    r = compare_generation_runs(a, b)
    assert r["n_common"] == 2 and r["greedy_identical"] == 1 and r["samples_identical"] == 1
    assert abs(r["max_abs_token_logprob_diff"] - 0.01) < 1e-12 and r["logprob_ok"] and not r["greedy_ok"] and not r["samples_ok"]


def test_interactive_labelcheck_scripted_input():
    from uncertainty.handcheck import interactive_labelcheck
    rows = [{"question": f"q{i}", "model_answer": "a", "gold_value": "g", "gold_aliases": "g", "human_correct": ""} for i in range(4)]
    rows[1]["human_correct"] = "yes"                                   # already filled: must be skipped
    answers = iter(["y", "maybe", "?", "q"])                          # row0=y ; row2: invalid then '?' ; row3: q (stop)
    shown = []
    interactive_labelcheck(rows, input_fn=lambda p: next(answers), show=shown.append)
    assert [r["human_correct"] for r in rows] == ["yes", "yes", "unclear", ""]
    assert any("please answer one of" in str(x) for x in shown)
    assert not any("SE10" in str(x) or "entropy" in str(x).lower() for x in shown)      # blind: no uncertainty information is shown
    from uncertainty.handcheck import parse_human_label
    assert parse_human_label("unclear") is None


def test_interactive_error_categories():
    from uncertainty.handcheck import ERROR_CATEGORIES, interactive_error_categories
    rows = [{"group": "confident_error", "question": "q", "model_answer": "a", "gold_value": "g", "n_clusters": 1, "category": ""},
            {"group": "uncertain_correct", "question": "q", "model_answer": "a", "gold_value": "g", "n_clusters": 4, "category": ""}]
    answers = iter(["9", "2", "5"])
    interactive_error_categories(rows, input_fn=lambda p: next(answers), show=lambda *_: None)
    assert [r["category"] for r in rows] == [ERROR_CATEGORIES[1], ERROR_CATEGORIES[4]]


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_handcheck.py


In [ ]:
%%writefile tests/test_pipeline_scripts.py
"""Integration test of the script glue (04 -> 05 -> 06 -> 07) on a SYNTHETIC copy of the repository.
Raw generation/semantic files are fabricated with the same structure that 02/03 write; nothing here is a real result.
Run: python tests/test_pipeline_scripts.py"""
import csv
import json
import random
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))


def make_raw(n_cal=400, n_test=600, seed=0):
    rs = random.Random(seed)
    gens, sems = [], []
    n = n_cal + n_test
    for i in range(n):
        wrong = rs.random() < 0.5
        split = "calibration" if i < n_cal else "test"
        gold = f"Answer {i}"
        text = f"Wrong {i}" if wrong else gold
        lp = [-(0.3 + (0.6 if wrong else 0.0) + 0.2 * rs.random()) for _ in range(3)]
        samples = [f"alt {rs.randint(0, 5)}" if wrong else gold for _ in range(10)]
        gens.append({"greedy_text": text, "short_answer": text, "token_ids": [1, 2, 3], "tokens": ["a", "b", "c"],
                     "token_logprob": lp + [-0.01], "token_entropy": [1.0 + rs.random() for _ in range(3)] + [0.1],
                     "is_eos": [False, False, False, True], "span_mask": [True, True, True, False], "truncated": False,
                     "argmax_match_rate": 1.0, "logit_vector_size": 151936, "samples": samples,
                     "seconds_greedy": 0.2, "seconds_samples": 1.0, "question_id": f"q{i}", "pool_index": i, "split": split,
                     "is_pilot": i < 50, "question": f"Question {i}?", "gold_value": gold,
                     "normalized_aliases": [f"answer {i}"]})
        ids = [rs.randint(0, 4) for _ in range(10)] if wrong else [0] * 9 + [rs.randint(0, 1)]
        sems.append({"question_id": f"q{i}", "cluster_ids": ids, "nli_calls": rs.randint(0, 20), "seconds_nli": 0.3})
    return gens, sems


def run(cwd, *args):
    r = subprocess.run([sys.executable, *args], cwd=cwd, capture_output=True, text=True)
    assert r.returncode == 0, f"{args} failed:\n{r.stdout[-1500:]}\n{r.stderr[-2500:]}"
    return r.stdout


def test_scripts_04_to_07_on_synthetic_repo():
    with tempfile.TemporaryDirectory() as t:
        root = Path(t)
        for d in ("scripts", "src", "configs"):
            shutil.copytree(REPO / d, root / d, ignore=shutil.ignore_patterns("__pycache__"))
        raw = root / "results" / "raw"
        raw.mkdir(parents=True)
        gens, sems = make_raw()
        with open(raw / "generations.jsonl", "w") as f:
            f.writelines(json.dumps(g) + "\n" for g in gens)
        with open(raw / "semantic.jsonl", "w") as f:
            f.writelines(json.dumps(s) + "\n" for s in sems)

        out = run(root, "scripts/04_build_table.py")
        assert "rows 1000" in out and "with semantic entropy 1000" in out
        tab = (root / "results" / "per_question.csv").read_text()
        assert "Question 1?" not in tab and "Answer 1" not in tab           # no question text / gold answers in the public table

        run(root, "scripts/05_handcheck.py", "--make-labelcheck")
        sheet = root / "results" / "raw" / "handcheck_sheet.csv"
        rows = list(csv.DictReader(open(sheet)))
        assert len(rows) == 30 and "SE10" not in rows[0] and "uncertainty" not in " ".join(rows[0].keys())   # blind sheet
        for k, r in enumerate(rows):                                          # simulate a careful human who disagrees once
            gold_ok = r["model_answer"] == r["gold_value"]
            r["human_correct"] = ("no" if gold_ok else "yes") if k == 0 else ("yes" if gold_ok else "no")
        with open(sheet, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
        out = run(root, "scripts/05_handcheck.py", "--score-labelcheck")
        assert '"meets_target": true' in out and "DISAGREEMENT" in out

        out = run(root, "scripts/06_evaluate.py", "--mode", "pilot", "--n-boot", "100", "--no-figs")
        assert "PILOT = exploratory" in out and (root / "results" / "pilot" / "summary.json").exists()

        # ---- audit gate + provenance
        import pandas as pd
        assert (root / "results" / "pilot" / "audit_report.json").exists()
        res = pd.read_csv(root / "results" / "pilot" / "results.csv")
        assert {"input_sha256", "git_commit", "audit_passed", "code_sha256"} <= set(res.columns) and res["input_sha256"].nunique() == 1
        csvp = root / "results" / "per_question.csv"
        good = csvp.read_text()
        bad = pd.read_csv(csvp); bad.loc[1, "question_id"] = bad.loc[0, "question_id"]; bad.to_csv(csvp, index=False)
        (root / "results" / "pilot" / "summary.json").unlink()
        r = subprocess.run([sys.executable, "scripts/06_evaluate.py", "--mode", "pilot", "--n-boot", "100", "--no-figs"], cwd=root, capture_output=True, text=True)
        assert r.returncode != 0 and "no metric was computed" in (r.stdout + r.stderr) and "duplicated question_id" in r.stdout
        assert not (root / "results" / "pilot" / "summary.json").exists()                    # nothing was computed
        r = subprocess.run([sys.executable, "scripts/06_evaluate.py", "--mode", "pilot", "--skip-audit", "--n-boot", "100", "--no-figs"], cwd=root, capture_output=True, text=True)
        assert r.returncode == 0 and "--skip-audit" in r.stdout                               # override works but is recorded
        assert json.load(open(root / "results" / "pilot" / "summary.json"))["provenance"]["audit_overridden"] is True
        csvp.write_text(good)
        out = run(root, "scripts/12_audit_outputs.py", "--mode", "pilot")
        assert "AUDIT PASSED" in out and "SKIPPED" in out                                      # missing inputs are shown as SKIPPED

        r = subprocess.run([sys.executable, "scripts/06_evaluate.py", "--mode", "final", "--n-boot", "100", "--no-figs"],
                           cwd=root, capture_output=True, text=True)
        assert r.returncode != 0 and "--frozen" in (r.stdout + r.stderr)      # guard: no test access without --frozen
        r = subprocess.run([sys.executable, "scripts/06_evaluate.py", "--mode", "final", "--frozen", "--n-boot", "100", "--no-figs"],
                           cwd=root, capture_output=True, text=True)
        assert r.returncode != 0 and "protocol-v1" in (r.stdout + r.stderr)   # guard: tag required (this temp dir is not a git repo)
        out = run(root, "scripts/06_evaluate.py", "--mode", "final", "--frozen", "--skip-tag-check", "--n-boot", "100")
        assert "WITHOUT the protocol-v1 git tag" in out and (root / "results" / "final" / "figs" / "fig4_cascade.png").exists()

        out = run(root, "scripts/07_make_latex_tables.py")
        assert (root / "paper" / "tables" / "table_main.tex").exists()
        summ = json.load(open(root / "results" / "final" / "summary.json"))
        assert summ["rq1"]["se_auroc"] > 0.9 and summ["n_eval"] == 600       # planted strong synthetic signal


if __name__ == "__main__":
    try:
        test_scripts_04_to_07_on_synthetic_repo()
        print("PASS  test_scripts_04_to_07_on_synthetic_repo\n\n1/1 passed (synthetic data only)")
    except Exception as e:  # noqa: BLE001
        print(f"FAIL  {e}"); sys.exit(1)


Writing tests/test_pipeline_scripts.py


In [ ]:
%%writefile tests/test_report.py
"""Tests for the rule-based outline on SYNTHETIC outputs. They check wording rules and guard words, not any real result.
Run: python tests/test_report.py"""
import re
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
sys.path.insert(0, str(Path(__file__).resolve().parent))
from test_evaluation import synth_df  # noqa: E402
from uncertainty.evaluation import run_analysis  # noqa: E402
from uncertainty.report import build_outline, f3  # noqa: E402

CFG = yaml.safe_load(open(Path(__file__).resolve().parents[1] / "configs" / "protocol_v1.yaml"))
FORBIDDEN = re.compile(r"significan|prove|outperform|state-of-the-art|clearly show|breakthrough", re.I)


def outline(df, e1=None, prov=None):
    with tempfile.TemporaryDirectory() as t:
        S = run_analysis(df, CFG, "final", t, n_boot=300, make_figs=False, provenance=prov or {"input_sha256": "x" * 64, "audit_passed": True, "audit_overridden": False})
        t2, t3, t4, t6 = (pd.read_csv(Path(t) / f) for f in ("table2.csv", "table3.csv", "table4.csv", "table6.csv"))
        return build_outline(S, t2, t3, t4, t6, e1=e1), S, t2


def test_semantic_entropy_stronger_case():
    md, S, t2 = outline(synth_df())
    assert "H1 is supported **for this model, dataset, prompt and label rule**" in md
    assert f3(float(t2.loc[t2.key == "se", "auroc"].iloc[0])) in md                       # numbers copied from the table
    assert "'Semantic entropy is better than token scores' in general" in md             # overgeneralization is listed as unsupported
    assert not FORBIDDEN.search(md), FORBIDDEN.search(md).group(0)


def test_token_wins_contradicts_h1():
    md, S, _ = outline(synth_df(token_effect=2.5, se_effect=0.8, seed=2))
    assert "contradicts H1" in md and "do not reinterpret it as support" in md


def test_null_signal_is_reported_as_null_not_failure_of_the_idea():
    df = synth_df(token_effect=0.0, se_effect=0.0, seed=5)
    md, S, _ = outline(df)
    assert "not shown to be predictive here" in md and "Check the pipeline" in md
    assert not FORBIDDEN.search(md)


def test_token_not_predictive_but_se_is():
    md, S, _ = outline(synth_df(token_effect=0.0, se_effect=1.2, seed=7))
    assert "no evidence that the mean-NLL score predicts" in md
    assert "'Token-level scores do not work': only that this score on this setup was not shown" in md
    assert "a cascade that depends on it was expected to fail" in md
    assert "not shown to be above chance" in md and "not evidence that semantic entropy beats a token score that works" in md


def test_no_difference_is_not_equivalence():
    md, S, _ = outline(synth_df(token_effect=1.2, se_effect=1.2, seed=11))
    if S["rq1"]["diff_ci"][0] <= 0 <= S["rq1"]["diff_ci"][1]:
        assert "no evidence of a difference" in md and "not** evidence of equivalence" in md
    else:                                                                                   # extremely unlikely draw: still must not claim equivalence
        assert "equivalent" not in md.split("Wording the evidence allows")[1].split("Not supported")[0]


def test_audit_warning_and_extension_branches_and_limits():
    md, _, _ = outline(synth_df(), prov={"input_sha256": "y" * 64, "audit_passed": False, "audit_overridden": True})
    assert "did not pass or was overridden" in md
    md, _, _ = outline(synth_df(), e1={"ran": False, "reason": "trigger not met: length-only AUROC CI does not exclude 0.5"})
    assert "Not run: trigger not met" in md and "not a failure" in md
    md, _, _ = outline(synth_df(), e1={"ran": True, "auroc_adjusted": 0.6, "auroc_primary": 0.59, "difference": 0.01, "difference_ci": [-0.02, 0.04], "verdict": "no evidence of improvement"})
    assert "Do not describe it as an improvement unless the verdict says so" in md
    for needle in ("Limitations", "one stored generation run", "alias matching", "consistently wrong answers get low entropy", "Follow-up experiments (at most three"):
        assert needle in md
    assert md.count("| 1 (cheapest)") == 1 and md.count("| 2 |") == 1 and md.count("| 3 (most expensive)") == 1      # never more than three


def test_followup_choice_follows_results():
    md_null, _, _ = outline(synth_df(token_effect=0.0, se_effect=0.0, seed=5))
    assert "Suggested first, given these results: **A" in md_null


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t(); print(f"PASS  {t.__name__}")
        except Exception as e:  # noqa: BLE001
            import traceback; traceback.print_exc()
            failed += 1; print(f"FAIL  {t.__name__}: {e!r}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed (synthetic data only)")
    sys.exit(1 if failed else 0)


Writing tests/test_report.py


In [ ]:
# Check that every file was written and is valid Python
sh("python -m compileall -q src scripts tests && echo ALL FILES COMPILE", log=None)
sh("ls src/uncertainty scripts tests configs", log=None)

$ python -m compileall -q src scripts tests && echo ALL FILES COMPILE
ALL FILES COMPILE
$ ls src/uncertainty scripts tests configs
configs:
protocol_v1.yaml

scripts:
00_env_test.py
01_build_subset.py
02_generate.py
03_semantic_entropy.py
04_build_table.py
05_handcheck.py
06_evaluate.py
07_make_latex_tables.py
08_sampling_check.py
09_repro_check.py
10_extension_length_adjusted.py
11_freeze_protocol.py
12_audit_outputs.py
13_results_outline.py
collect_diagnostics.py
_common.py
__pycache__

src/uncertainty:
audit.py
cascade.py
data.py
diagnostics.py
evaluation.py
extensions.py
generation.py
handcheck.py
__init__.py
io_utils.py
latex_tables.py
metrics.py
prompting.py
__pycache__
report.py
repro.py
scoring.py
seed.py
semantic_entropy.py
table.py
textnorm.py

tests:
__pycache__
run_all.py
run_tests.py
test_audit.py
test_core.py
test_data.py
test_diagnostics.py
test_evaluation.py
test_extensions.py
test_freeze.py
test_handcheck.py
test_pipeline_scripts.py
test_report.py


0

## Step 3. Logic tests (CPU, synthetic data)
Expected: `ALL TESTS PASSED (synthetic data only)`. These prove the logic, **not** that the real model/data behave. If anything fails, stop and send the output.

In [ ]:
if RUN_TESTS:
    sh("python tests/run_all.py", log="01_tests.txt")

$ python tests/run_all.py
run_tests.py          OK    8/8 passed
test_data.py          OK    7/7 passed (synthetic data only)
test_core.py          OK    9/9 passed (synthetic data only)
test_evaluation.py    OK    6/6 passed (synthetic data only)
test_handcheck.py     OK    7/7 passed (synthetic data only)
test_pipeline_scripts.pyOK    1/1 passed (synthetic data only)
test_diagnostics.py   OK    2/2 passed
test_extensions.py    OK    4/4 passed (synthetic data only)
test_freeze.py        OK    1/1 passed
test_audit.py         OK    8/8 passed (synthetic data only)
test_report.py        OK    7/7 passed (synthetic data only)

ALL TESTS PASSED (synthetic data only)


## Step 4. Environment test (needs the GPU; downloads about 3 GB the first time)
Stages: system, libraries, GPU, seeding, config, dataset, model, greedy answers with token scores and a consistency check, sampling, NLI model.
Expected last line: `ENV TEST SUMMARY: x PASS, y WARN, z FAIL`. **WARN is acceptable if you understand it; FAIL is not.**
Look especially at: duplicates found in the dataset, `generation_config`, the rendered prompt, `max |diff|` (must be 0.05 or less), argmax rate (about 1.0).

In [ ]:
if RUN_ENV_TEST:
    sh("python scripts/00_env_test.py --check-test", log="02_env_test.txt", check=False)

$ python scripts/00_env_test.py --check-test
ENVIRONMENT TEST  | started 2026-11-13 04:32:04 | repo: /content/repo

=== [1 system] ===
Python  : 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
CPUs    : 2
RAM (GB): 12.7
Disk free (GB): 120.1
Fri Nov 13 04:32:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |       

## Step 5. Build the question subset (ids only; no data is stored)
Expected: counts of duplicates removed, pool 1000 (400 calibration / 600 test), 50 pilot questions, `determinism self-check ... True`, and a 64-character hash.

In [ ]:
if RUN_SUBSET:
    sh("python scripts/01_build_subset.py" + (f" --pool-size {POOL_SIZE}" if POOL_SIZE else ""), log="03_subset.txt")

$ python scripts/01_build_subset.py
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
CYCLE 1 | build subset | 2026-11-13 04:34:45
protocol: protocol-v1 | status: draft_ready_to_freeze
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
loaded mandarjoshi/trivia_qa / rc.nocontext / validation: 17944 rows
columns: ['que

## Step 6. Pilot: 50 calibration questions (never test questions)
Generates the greedy answer, token scores and 10 sampled answers per question, clusters the samples (semantic entropy), and builds the results table.
At the end of generation the script prints **seconds per question and a projection for the full pool**. Rule: if the projection (plus NLI time) exceeds **6 GPU-hours**, set `POOL_SIZE = 400` and rerun Step 5 **before** touching the test part, and log the change.
Tip: first set `DEBUG_LIMIT = 3` to check quickly that everything works.

In [ ]:
if RUN_PILOT:
    lim = f" --limit {DEBUG_LIMIT}" if DEBUG_LIMIT else ""
    sh("python scripts/02_generate.py --split pilot" + lim, log="04_generate_pilot.txt")
    sh("python scripts/03_semantic_entropy.py --split pilot" + lim, log="05_semantic_pilot.txt")
    sh("python scripts/04_build_table.py", log="06_table.txt")

$ python scripts/02_generate.py --split pilot
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
split=pilot | selected 50 | already done 50 | to run now 0 | protocol protocol-v1 (draft_ready_to_freeze)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



## Step 7. Validation checks on the pilot (baseline must be validated before anything else)
**7a. Hand-check 30 answers.** You see the question, the model's answer and the gold answer. Answer `y` (model answer is correct), `n`, `?` or `q` (stop and save).
Do **not** think about uncertainty scores: none are shown. Target: at least 90% agreement with the automatic label.

In [ ]:
if RUN_PILOT and DO_HAND_LABELLING:
    sh("python scripts/05_handcheck.py --make-labelcheck", log="07_handcheck.txt")
    import csv
    sys.path.insert(0, f"{REPO}/src")
    from uncertainty.handcheck import interactive_labelcheck
    sheet = f"{REPO}/results/raw/handcheck_sheet.csv"
    rows = list(csv.DictReader(open(sheet)))
    rows = interactive_labelcheck(rows)
    with open(sheet, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("\nlabelled:", sum(1 for r in rows if r["human_correct"]), "of", len(rows))

$ python scripts/05_handcheck.py --make-labelcheck
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
wrote /content/repo/results/raw/handcheck_sheet.csv (30 rows). Fill human_correct with yes/no. Do NOT look at uncertainty scores while doing it.

[1/30] QUESTION: Which company produced the Hastings and Herald aircraft?
    model answer : Boeing
    gold answer  : Handley-Page   (aliases: handley page aircraft company | o 100 and o 400 handley page | handley page slat | handley page reading | handley page o 100 and o 400 | handley page limited | handley page ltd | handley page)
  please answer one of: ['?', 'n', 'q', 'y']

[2/30] QUESTION: The village of Tightwad is in which US state?
    model answer : New Hampshire
    gold answer  : Missouri   (aliases: economy of missouri | misuri | missouri | misouri | politics of missouri |

In [ ]:
if RUN_PILOT and DO_HAND_LABELLING:
    sh("python scripts/05_handcheck.py --score-labelcheck", log="07_handcheck.txt", check=False)
    print("\nIf agreement is below 0.90: read each DISAGREEMENT; the label rule may be wrong. Send the output for review before continuing.")

NameError: name 'RUN_PILOT' is not defined

**7b. Look at the clusters** of 20 pilot questions. For each, note wrong MERGES (different meanings in one cluster) and wrong SPLITS (same meaning in two clusters).

In [ ]:
if RUN_PILOT and RUN_VALIDATION_TOOLS:
    sh("python scripts/05_handcheck.py --show-clusters 20", log="08_clusters.txt")

**7c. Is sampling really untruncated?** Expected: `largest |z| < 4 -> PASS`. A FAIL means the model's default top-k/top-p is still active: do not continue.

In [ ]:
if RUN_VALIDATION_TOOLS:
    sh("python scripts/08_sampling_check.py", log="09_sampling_check.txt", check=False)

**7d. Reproducibility:** regenerate 5 pilot questions into a second file and compare. Greedy answers must be identical; identical samples are expected, but report it if GPU non-determinism breaks that.

In [ ]:
if RUN_PILOT and RUN_VALIDATION_TOOLS:
    sh("rm -f results/raw/generations_rerun.jsonl", log=None)
    sh("python scripts/02_generate.py --split pilot --limit 5 --out results/raw/generations_rerun.jsonl", log="10_repro.txt")
    sh("python scripts/09_repro_check.py results/raw/generations.jsonl results/raw/generations_rerun.jsonl", log="10_repro.txt", check=False)

**7e. Compare my answer normalization with the official TriviaQA one (check V8).** Best effort: the cell tries to download the official evaluation file and compares
`normalize_answer` on tricky strings. If the download or extraction fails, it says so; then open the official file by hand and compare.

In [ ]:
if RUN_VALIDATION_TOOLS:
    import ast, re, string, urllib.request
    sys.path.insert(0, f"{REPO}/src")
    from uncertainty.textnorm import normalize_answer as mine
    tests = ["U.S.A.", "Jean-Paul Sartre", "O'Brien", "New_York", "The Beatles", "Paris,France", "A.C. Milan", "\u2018Hamlet\u2019", "St. Louis", "Mr. T"]
    url = "https://raw.githubusercontent.com/mandarjoshi90/triviaqa/master/evaluation/triviaqa_evaluation.py"
    try:
        src = urllib.request.urlopen(url, timeout=20).read().decode()
        tree = ast.parse(src)
        fn = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "normalize_answer"]
        assert fn, "normalize_answer not found at top level of the official file"
        ns = {"re": re, "string": string}
        exec(ast.get_source_segment(src, fn[0]), ns)
        off = ns["normalize_answer"]
        bad = 0
        for t in tests:
            a, b = mine(t), off(t)
            print(f"{t!r:<24} mine={a!r:<22} official={b!r:<22} {'OK' if a == b else 'DIFFERENT'}")
            bad += (a != b)
        print("\nRESULT:", "all equal: normalization matches the official one on these strings" if not bad else f"{bad} DIFFERENCES: send this output; the label rule must be changed and logged")
    except Exception as e:
        print("Could not run the automatic comparison:", type(e).__name__, e)
        print("Open the official triviaqa_evaluation.py in the TriviaQA GitHub repo by hand and compare normalize_answer on:", tests)
        print("Mine:", {t: mine(t) for t in tests})

## Step 8. Audit, then pilot evaluation (exploratory sanity check, 50 questions)
**First the audit** (alignment of ids and labels with the raw answers, missing values, partitions, metric direction, leakage, comparator consistency). It computes **no metric**; if it reports an ERROR the evaluation refuses to run. Read every WARN and SKIPPED line.
Then the evaluation reads **only pilot rows**. Expected: a table with AUROC and 95% intervals for the primary token score, semantic entropy, the length-only control and the random control.
**Red flags** (stop and send the output): random control far from 0.5; semantic entropy entirely below 0.5; hand-check agreement below 0.90; consistency difference above 0.05.
Wide intervals with 50 questions are normal and are **not** a failure. **Pilot numbers are not results: never put them in the paper.**

In [ ]:
if RUN_PILOT:
    sh("python scripts/12_audit_outputs.py --mode pilot", log="11_eval_pilot.txt", check=False)
    sh("python scripts/06_evaluate.py --mode pilot --no-figs", log="11_eval_pilot.txt", check=False)

## Step 9. Calibration part (400 questions) [turn on `RUN_CALIBRATION = True` after the pilot passes]
Needed to fit the calibration models and escalation thresholds. This is the long step. It resumes if Colab disconnects: just run the notebook from the top again.

In [ ]:
if RUN_CALIBRATION:
    sh("python scripts/02_generate.py --split calibration", log="12_generate_calibration.txt")
    sh("python scripts/03_semantic_entropy.py --split calibration", log="13_semantic_calibration.txt")
    sh("python scripts/04_build_table.py", log="14_table.txt")

## Step 10. Freeze the protocol (once)
Complete **all** items of the Verification and Freezing checklist in `experiment_protocol.md` Section 14, and record your decisions. Then set
`I_COMPLETED_THE_CHECKLIST = True` and `FREEZE_PROTOCOL = True`. This hashes the config and the code, tags `protocol-v1`, and stores the record on Drive.
Later sessions restore the tag **only if the code is byte-identical**; any change after freezing is refused on purpose (it would be an undisclosed protocol change).

In [ ]:
if FREEZE_PROTOCOL:
    assert I_COMPLETED_THE_CHECKLIST, "Set I_COMPLETED_THE_CHECKLIST = True only after you finished the checklist."
    sh("python scripts/11_freeze_protocol.py --freeze", log="15_freeze.txt")

## Step 11. The TEST part and the final analysis (600 questions)
Only runs with `RUN_TEST_SPLIT = True` **and** a valid freeze. It: restores the freeze tag, generates answers and clusters for the test part, builds the table,
runs the final evaluation (RQ1 to RQ3, ablation, sensitivity, figures), the pre-registered extension E1 (only if its trigger is met), and the LaTeX tables.
Do **not** look at test results before this step. Report whatever comes out, including negative or inconclusive outcomes.

In [ ]:
if RUN_TEST_SPLIT:
    sh("python scripts/11_freeze_protocol.py --restore", log="16_final.txt")        # refuses if the code changed since the freeze
    sh("python scripts/02_generate.py --split test --allow-test", log="16_final.txt")
    sh("python scripts/03_semantic_entropy.py --split test --allow-test", log="16_final.txt")
    sh("python scripts/04_build_table.py", log="16_final.txt")
    sh("python scripts/12_audit_outputs.py --mode final", log="16_final.txt", check=False)   # shown explicitly; 06 repeats it and stops on any ERROR
    sh("python scripts/06_evaluate.py --mode final --frozen", log="16_final.txt")
    sh("python scripts/10_extension_length_adjusted.py --frozen", log="16_final.txt", check=False)
    sh("python scripts/07_make_latex_tables.py", log="16_final.txt")

In [ ]:
# Show the final decisions and figures
if RUN_TEST_SPLIT:
    from IPython.display import Image, display
    s = json.load(open(f"{REPO}/results/final/summary.json"))
    print("n_eval:", s["n_eval"], "| wrong:", s["n_wrong"], "| correct:", s["n_correct"])
    print("RQ1:", json.dumps(s["rq1"], indent=1))
    print("RQ3:", json.dumps(s["rq3"], indent=1))
    print("FLAGS:", s["flags"])
    for fig in ["fig2_roc", "fig3_aggregators", "fig4_cascade", "fig5_reliability"]:
        p = f"{REPO}/results/final/figs/{fig}.png"
        if os.path.exists(p): display(Image(filename=p, width=420))

**Step 11a. Results and Discussion outline from the real outputs.** Fills `results/final/results_discussion_outline.md` with the numbers copied from your files and wording chosen by pre-registered rules
(what you may and may not claim for each outcome). It computes no new statistics and writes no mechanisms as facts: those are marked `[AUTHOR]` for you.

In [ ]:
if RUN_TEST_SPLIT:
    sh("python scripts/13_results_outline.py", log="16_final.txt", check=False)

**Step 11b. Error analysis (by hand, 60 test answers).** Choose one category per answer. Then the counts are written to `results/final/table5.csv`.

In [ ]:
if RUN_TEST_SPLIT and RUN_ERROR_ANALYSIS:
    sh("python scripts/05_handcheck.py --make-erroranalysis --frozen", log="17_error_analysis.txt")
    import csv
    sys.path.insert(0, f"{REPO}/src")
    from uncertainty.handcheck import interactive_error_categories
    path = f"{REPO}/results/raw/error_analysis_sheet.csv"
    rows = list(csv.DictReader(open(path)))
    rows = interactive_error_categories(rows)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    if all(r["category"] for r in rows):
        sh("python scripts/05_handcheck.py --summarize-erroranalysis --frozen", log="17_error_analysis.txt")
    else:
        print("Not all rows categorized yet: run this cell again to continue.")

## Step 12. Save and download
Everything is already on Drive in `MyDrive/AITR_Lab02_run/`. This cell also makes one zip of the **public** results (tables, figures, logs, no question text) to download.
`results/raw/` contains TriviaQA question text and gold answers: **do not publish it**.

In [ ]:
# build a zip that excludes results/raw (contains question text and gold answers)
import zipfile
zp = f"{BASE}/AITR_results_public.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ["results", "data", "logs", "paper"]:
        for root, _, fs in os.walk(f"{BASE}/{d}"):
            if os.sep + "raw" in root.replace(BASE, ""): continue
            for fn in fs:
                full = os.path.join(root, fn); z.write(full, os.path.relpath(full, BASE))
print("wrote", zp, f"({os.path.getsize(zp)/1e6:.2f} MB)")
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print("download manually from the file browser:", zp)

## If something fails: collect safe diagnostics
Save the red error text into a file, or just run the cell below right after the failure. It never prints passwords or tokens (token-like strings are masked).
Send me the printed block **and** the code of the failing function.

In [ ]:
cmd = "(describe the command that failed)"
expected = "(describe what you expected)"
sh(f'python scripts/collect_diagnostics.py --command "{cmd}" --expected "{expected}" --log logs/02_env_test.txt', log=None, check=False)